# The projection from degenerate FWM to the coherent coupler

In this notebook we use our analytic solution to the coherent coupler in the Weierstrass notation to explore coordinate transformations. We show that it is possible to transform the coherent coupler to a canonical coordinate system in which the solutions are single valued functions, namely Kronecker theta functions. We further show that there is a projection from the degenerate FWM system to the coherent coupler system.

In [1]:
from sympy import *
from time import time

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi) = symbols(
    '''delta, nu, Aeff, chi'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

## Elliptic function identities

In [2]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)

quasi_period_sigma_b = Eq(
    sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3),
    (-1)**(m*n + m + n)*sigma(z, g2, g3)*
    exp(2*(m*omega[3] + n*omega[1] + z)*zeta(m*omega[3] + n*omega[1], g2, g3))
)

# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_sigma_dbl_ratio
quasi_period_sigma_b

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(\wp'(x, g2*c**4, g3*c**6), \wp'(x*c, g2, g3)*c**3)

Eq(omega[1], f(1, g2, g3))

Eq(omega[1]*c, f(1, g2/c**4, g3/c**6))

Eq(omega[3], f(3, g2, g3))

Eq(omega[3]*c, f(3, g2/c**4, g3/c**6))

## The solution in original coordinates

We start by reminding ourselves of the solutions derived in *The general 2 mode case of the coherent coupler* notebook for modes in the original coordinates $u,v$:

In [3]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]

u_sqrt_wp_z0_z1 = Eq(
    u(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
v_sqrt_wp_z0_z1 = Eq(
    v(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)

uv_wp_mu1_mu2_const = Eq(
    u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]),
    (wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*wpp(z1, g2, g3)/(
        (-wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))*(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))*sqrt(d[4])
    )
)

sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1
uv_wp_mu1_mu2_const
sum_r1j_1
r1j_log_part

for _ in Lams:
    _

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]), (wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*\wp'(z1, g2, g3)/((-wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))*(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))*sqrt(d[4])))

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

Eq(Lamda[0, m], -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))

### The original parameters

In [4]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _

g2_dj
g3_dj

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2])

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

In [5]:
B_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(gamma[j] - lamda[1])*sqrt(d[4])
)
C_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)/(gamma[j] - lamda[1])
)
D_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)*wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2,
    (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4])
)


wp_z1_lam_uv_j = Eq(
    wpp(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])),
    -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2
)
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)
wppz1_djs = Eq( 4*wpp(z1, g2, g3)/sqrt(d[4]), -d[5])

wp_z0_mu_j_d = Eq(
    wp(-z0 + mu[j], g2, g3),
    d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - 
    (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1]))
)
wpp_z0_mu_j_d = Eq(
    wpp(-z0 + mu[j], g2, g3),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)
    *(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2)
)


B_wpz1_z0_muj_to_d_lam1_gamj
C_wpz1_z0_muj_to_d_lam1_gamj
D_wpz1_z0_muj_to_d_lam1_gamj
wp_z1_lam_uv_j
wppz1_djs
wp_z0_mu_j_d
wpp_z0_mu_j_d
d4_lam_0
d5_dj

Eq(\wp'(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-gamma[j] + lamda[1])*sqrt(d[4]))

Eq(\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)/(gamma[j] - lamda[1]))

Eq(\wp'(z1, g2, g3)*\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2, (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4]))

Eq(\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])), -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2)

Eq(4*\wp'(z1, g2, g3)/sqrt(d[4]), -d[5])

Eq(wp(-z0 + mu[j], g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*gamma[j] - 4*lamda[1]))

Eq(\wp'(-z0 + mu[j], g2, g3), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)*(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2))

Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)

Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

We recall the equations from the original derivation of the solution in terms of $\rho, b_j, \gamma_j, \lambda_1$:

In [6]:
drhop_b = Eq(
    Derivative(rho(z), z)**2, 
    (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2))
)
drhop_d = Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

drho_2z = Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j-1)*d[j]/2, (j, 0, 4)))
drho_2z_b = Eq(
    Derivative(rho(z), (z, 2)),
    (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2])
)

gamma_lamda_bj = Eq(
    drhop_b.rhs.subs(rho(z), lamda[1]).doit()
    .subs(gamma[2], - gamma[1])
    .expand().collect(4*gamma[1]**2 - 4*lamda[1]**2, factor),
    0
)
gamma_lamda_bj_gam1 = Eq(gamma[1]**2, solve(gamma_lamda_bj, gamma[1]**2)[0])

drhop_b
drhop_d
drho_2z
drho_2z_b
gamma_lamda_bj_gam1

Eq(Derivative(rho(z), z)**2, (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2)))

Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j - 1)*d[j]/2, (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) - 4*rho(z) + 2*gamma[1] + 2*gamma[2])

Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)

In [7]:
bs_as_ds = [_.subs([_.args for _ in c_j_coeffs]) for _ in d_j_coeffs]

b0_d1 = Eq(b[0], solve(bs_as_ds[1], b[0])[0])
b2_d3 = Eq(b[2], solve(bs_as_ds[3], b[2])[0])
b1_sqrd_as_d = Eq(
    b[1]**2,
    solve(
        bs_as_ds[2]
        .subs([
            b0_d1.args, 
            (2*bs_as_ds[4].rhs/bs_as_ds[3].rhs, 2*bs_as_ds[4].lhs/bs_as_ds[3].lhs)
        ]),
        b[1]**2
    )[0]
)
b_to_d_subs = [
    b0_d1.args,
    b2_d3.args,
    b1_sqrd_as_d.args
]

for _ in bs_as_ds:
    _
print()
for _ in b_to_d_subs:
    Eq(*_)
    
def generate_lambda_reductions(max_power=12):
    """
    Generate substitutions reducing lamda**n (n >= 4)
    to expressions at most cubic in lamda.
    """
    subs = {}

    # base reduction: λ⁴
    subs[lamda[1]**4] = -(d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3) / d[4]

    # build higher powers
    for _n in range(5, max_power + 1):
        subs[lamda[1]**_n] = expand(lamda * subs[lamda[1]**(_n - 1)]).subs(subs)

    return subs

lamda_d_reductions = generate_lambda_reductions()

Eq(d[0], b[0]**2 - 4*Product(gamma[j], (j, 1, 2)))

Eq(d[1], 2*b[0]*b[1])

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4)

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(b[0], d[1]/(2*b[1]))

Eq(b[2], d[3]/(2*b[1]))

Eq(b[1]**2, -2*d[1]*d[4]/d[3] + d[2] + 4)

### Reorganising the original equations of motion

In [8]:
ajk_syms = [(a[2, 1], a[1, 2])]

uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z), 
    Product(v(z, mu[k]), (k, 1, 2)) - Product(u(z, mu[k]), (k, 1, 2))
)

uprodj = Eq(
    Product(u(z, mu[j]), (j, 1, 2)),
    -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + 
    Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + 
    Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2
)
vprodj = Eq(uprodj.lhs + duvj.rhs.replace(k,j), uprodj.rhs + duvj.lhs.replace(k,j))
a_0_eq = Eq(uprodj.rhs + vprodj.rhs, uprodj.lhs + vprodj.lhs)

du_phase_mod_part = (a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*u(z,mu[j])
dv_phase_mod_part = -(a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*v(z,mu[j])

du_mixing_part = Product(v(z,mu[k]), (k,1,2))/v(z, mu[j])
dv_mixing_part = -Product(u(z,mu[k]), (k,1,2))/u(z, mu[j])

duj = Eq(diff(u(z,mu[j]),z), -du_phase_mod_part + du_mixing_part)
dvj = Eq(diff(v(z,mu[j]),z), (-dv_phase_mod_part).factor() + dv_mixing_part)
duj_subs = [duj.subs(j, _j + 1).args for _j in range(2)]
dvj_subs = [dvj.subs(j, _j + 1).args for _j in range(2)]
duj_dvj_subs = duj_subs + dvj_subs


assert (
    diff(solve(a_0_eq.doit(),a[0])[0],z).subs(duj_dvj_subs).doit().subs(ajk_syms).simplify() == 0
), 'a0 not conserved'

uv_j_rho

uprodj
vprodj
a_0_eq

duj
dvj
duvj

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(Product(v(z, mu[j]), (j, 1, 2)), Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)))

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), -Product(u(z, mu[k]), (k, 1, 2)) + Product(v(z, mu[k]), (k, 1, 2)))

## Gauge transform to meromorphic functions

In this section we implement a gauge transform to remove the log of $\sigma$ terms from the exponentials in the solutions, i.e., we wish to remove the essential singularities to leave meromorphic functions. This worked well in the FWM case and we achieve a similarly positive result here.

In [9]:
uU_sub = Eq(
    u(z,mu[j]), 
    uhat(z, mu[j])
    *exp(-varphi(z,j))
)
vV_sub = Eq(
    v(z,mu[j]), 
    vhat(z, mu[j])
    *exp(varphi(z,j))
)

Uu_sub = Eq(uU_sub.rhs*exp(varphi(z, j)), uU_sub.lhs*exp(varphi(z, j)))
Vv_sub = Eq(vV_sub.rhs*exp(-varphi(z, j)), vV_sub.lhs*exp(-varphi(z, j)))

uv_gamma_cons =Eq(
    uv_j_rho.lhs - uv_j_rho.lhs.subs(j,k),
    uv_j_rho.rhs - uv_j_rho.rhs.subs(j,k)
)

uv_gamma_cons_gauge = Eq(
    uv_gamma_cons.lhs.subs([
        uU_sub.args, 
        vV_sub.args,
        uU_sub.subs(j,k).args, 
        vV_sub.subs(j,k).args
    ]).simplify(),
    uv_gamma_cons.rhs
)
uv_gam_gauge_k_to_j = Eq(
    uhat(z, mu[j])*vhat(z, mu[j]) - uv_gamma_cons_gauge.lhs,
    uhat(z, mu[j])*vhat(z, mu[j]) - uv_gamma_cons_gauge.rhs
)

uv_hat_j_rho = Eq(uv_j_rho.lhs.subs([uU_sub.args, vV_sub.args]).simplify(), uv_j_rho.rhs)


uU_sub
vV_sub

Uu_sub
Vv_sub

uv_j_rho
uv_hat_j_rho
uv_gamma_cons
uv_gamma_cons_gauge

Eq(u(z, mu[j]), uhat(z, mu[j])*exp(-varphi(z, j)))

Eq(v(z, mu[j]), vhat(z, mu[j])*exp(varphi(z, j)))

Eq(uhat(z, mu[j]), u(z, mu[j])*exp(varphi(z, j)))

Eq(vhat(z, mu[j]), v(z, mu[j])*exp(-varphi(z, j)))

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(uhat(z, mu[j])*vhat(z, mu[j]), -rho(z) + gamma[j])

Eq(u(z, mu[j])*v(z, mu[j]) - u(z, mu[k])*v(z, mu[k]), gamma[j] - gamma[k])

Eq(uhat(z, mu[j])*vhat(z, mu[j]) - uhat(z, mu[k])*vhat(z, mu[k]), gamma[j] - gamma[k])

### Exploring the required phase shift

Before we define the $\varphi$ shift in terms of integrals of modes $u,v$, we want to see what the required shift is in terms of an integral over $\rho$. In general we require two things from the phase shifts applied to the modes:

- they cancel the essential singularities from our analytic solutions
- they sum to zero so that our differential equations are still polynomial like and we do not introduce new complicated integral terms in exponents

It is not too difficult to guess a form to meet these requirement. Something like the below appears to work, with $\beta_0$ some yet to be fixed constant:

In [10]:
phi_opt_1 = beta[0]*z + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*(-r[1, 1]+r[1,2])/2

phi_j_eqs = [
    Eq(varphi(z,1), phi_opt_1),
    Eq(varphi(z,2), -phi_opt_1)
]
phi_j_subs = [_.args for _ in phi_j_eqs]

gauge_phi_sum = Eq(
    Sum(varphi(z,j),(j,1,2)), 
    Sum(varphi(z,j),(j,1,2)).doit().subs(phi_j_subs)
)

sum_r1j_1
for _ in phi_j_eqs:
    _

gauge_phi_sum

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(varphi(z, 1), z*beta[0] + (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(varphi(z, 2), -z*beta[0] - (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(Sum(varphi(z, j), (j, 1, 2)), 0)

In [11]:
uhat_gauge_1 = Eq(
    Uu_sub.lhs.subs(j,1),
    Uu_sub.rhs.subs([
        u_sqrt_wp_z0_z1.args,
        (j,1),
        phi_j_subs[0]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

vhat_gauge_1 = Eq(
    Vv_sub.lhs.subs(j,1),
    Vv_sub.rhs.subs([
        v_sqrt_wp_z0_z1.args,
        (j,1),
        phi_j_subs[0]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

uhat_gauge_2 = Eq(
    Uu_sub.lhs.subs(j,2),
    Uu_sub.rhs.subs([
        u_sqrt_wp_z0_z1.args,
        (j,2),
        phi_j_subs[1]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

vhat_gauge_2 = Eq(
    Vv_sub.lhs.subs(j,2),
    Vv_sub.rhs.subs([
        v_sqrt_wp_z0_z1.args,
        (j,2),
        phi_j_subs[1]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

# upsilon captures the sign ambiguity

sig_sqrt_1 = Eq(
    sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/
    sqrt(
        sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/
        (sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)
    ),
    upsilon*sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/sigma(z - z0 - z1, g2, g3)
)

sig_sqrt_2 = Eq(
    1/(
        sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*
        sqrt(
            sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/
            (sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)
        )
    ),
    sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/sigma(z - z0 + z1, g2, g3)/upsilon
)
assert (sig_sqrt_1.lhs*sig_sqrt_2.lhs/(sig_sqrt_1.rhs*sig_sqrt_2.rhs)) == 1, 'sqrt ratios not 1'
sig_sqrt_subs = [
    sig_sqrt_1.args,
    sig_sqrt_2.args
]


uhat_gauge_1_b = uhat_gauge_1.subs(sig_sqrt_subs)
vhat_gauge_1_b = vhat_gauge_1.subs(sig_sqrt_subs)
uhat_gauge_2_b = uhat_gauge_2.subs(sig_sqrt_subs)
vhat_gauge_2_b = vhat_gauge_2.subs(sig_sqrt_subs)

uv_hat_gauge_bs = [
    uhat_gauge_1_b,
    vhat_gauge_1_b,
    uhat_gauge_2_b,
    vhat_gauge_2_b
]


uhat_gauge_1
vhat_gauge_1
uhat_gauge_2
vhat_gauge_2

sig_sqrt_1
sig_sqrt_2

for _ in uv_hat_gauge_bs:
    _

Eq(uhat(z, mu[1]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1] + epsilon[1])/(sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1] - epsilon[1])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(uhat(z, mu[2]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2] + epsilon[2])/(sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2] - epsilon[2])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)), sigma(z1, g2, g3)*sigma(z - z0, g2, g3)*upsilon/sigma(z - z0 - z1, g2, g3))

Eq(1/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))), sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/(sigma(z - z0 + z1, g2, g3)*upsilon))

Eq(uhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1] + epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1] - epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

Eq(uhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2] + epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2] - epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

In terms of $a_{j,k}$ such a shift would be:

In [12]:
r1j_a_d4 = r1j_log_part.subs(*Lams[1].replace(j,l).replace(m,j).args)

r11_min_r22 = Eq(
    r1j_a_d4.lhs.subs(j,1).doit() - r1j_a_d4.lhs.subs(j,2).doit(),
    (r1j_a_d4.rhs.subs(j,1).doit() - r1j_a_d4.rhs.subs(j,2).doit())
    .simplify().subs(ajk_syms)
)

r1j_a_d4
r11_min_r22

phi_j_eqs[0]
phi_j_eqs[0].subs([r11_min_r22.args])

Eq(r[1, j], (Sum(a[j, k], (k, 1, 2)) - Sum(a[l, k]/4, (l, 1, 2), (k, 1, 2)))/sqrt(d[4]))

Eq(r[1, 1] - r[1, 2], (a[1, 1] - a[2, 2])/sqrt(d[4]))

Eq(varphi(z, 1), z*beta[0] + (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(varphi(z, 1), z*beta[0] - (a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])))

and we can get such log terms from modal power integrals such as:

In [13]:
rho_integral = Eq(
    Integral(-rho(z) + rho(mu[j]),z),
    z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) -
    log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]) + C
)

rho_integral

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]))

### In terms of $u, v$

In this section we now consider a $\varphi$ shift in terms of $u,v$. Inspired by the FWM case, we try the shift in which the modal power weight is the distance from the mean of that matrix vector and similarly in the linear $z$ term. 

In [14]:
gauge_phi = Eq(
    varphi(z, j), 
    (a[j] - Sum(a[l], (l, 1, 2))/2 - gamma[j]*Sum(a[l, k]/2, (l, 1, 2), (k, 1, 2)))*z +
    Sum((a[j, k] - Sum(a[l, k]/2, (l, 1, 2)))*Integral(uhat(z, mu[k])*vhat(z, mu[k]),z), (k, 1, 2))
)
gauge_phi

Eq(varphi(z, j), z*(a[j] - gamma[j]*Sum(a[l, k]/2, (l, 1, 2), (k, 1, 2)) - Sum(a[l], (l, 1, 2))/2) + Sum((a[j, k] - Sum(a[l, k]/2, (l, 1, 2)))*Integral(uhat(z, mu[k])*vhat(z, mu[k]), z), (k, 1, 2)))

In [15]:
(
    gauge_phi.replace(uhat(z,mu[k])*vhat(z,mu[k]), gamma[k] - rho(z)).doit()
    .expand().subs(j,1).doit().subs(ajk_syms)
    .subs(gamma[2], - gamma[1])
)
(
    gauge_phi.replace(uhat(z,mu[k])*vhat(z,mu[k]), gamma[k] - rho(z)).doit()
    .expand().subs(j,2).doit().subs(ajk_syms)
    .subs(gamma[2], - gamma[1])
)

Eq(varphi(z, 1), -2*z*a[1, 2]*gamma[1] + z*a[1]/2 - z*a[2]/2 - a[1, 1]*Integral(rho(z), z)/2 + a[2, 2]*Integral(rho(z), z)/2)

Eq(varphi(z, 2), 2*z*a[1, 2]*gamma[1] - z*a[1]/2 + z*a[2]/2 + a[1, 1]*Integral(rho(z), z)/2 - a[2, 2]*Integral(rho(z), z)/2)

We confirm that this choice for $\phi$ can deliver the required log term to cancel essential singularities:

In [16]:
_int_subs = [
    (
        Integral(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]),z), 
        Integral(rho(mu[_j+1]) - rho(z),z).subs([rho_integral.subs(j,_j+1).args])
    )
    for _j in range(2)
]

gauge_phi_logs = [
    Eq(
        gauge_phi.lhs.subs(j, _j + 1),
        gauge_phi.rhs.subs(j, _j + 1)
        .doit().subs(_int_subs)
        .subs(ajk_syms)
        .expand()
        .collect([log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), sqrt(d[4])], factor)
    )
    for _j in range(2)
]

assert (
    gauge_phi_logs[0].rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))) -
    phi_j_eqs[0].subs([r11_min_r22.args]).rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)))
) == 0, 'log coeffs do not match for j=1'
assert (
    gauge_phi_logs[1].rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))) -
    phi_j_eqs[1].subs([r11_min_r22.args]).rhs.coeff(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)))
) == 0, 'log coeffs do not match for j=2'

for _ in gauge_phi_logs:
    _

Eq(varphi(z, 1), z*(zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 1] - zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 2] + zeta(-z0 + z1 + mu[2], g2, g3)*a[1, 2] - zeta(-z0 + z1 + mu[2], g2, g3)*a[2, 2] + zeta(z0 + z1 - mu[1], g2, g3)*a[1, 1] - zeta(z0 + z1 - mu[1], g2, g3)*a[1, 2] + zeta(z0 + z1 - mu[2], g2, g3)*a[1, 2] - zeta(z0 + z1 - mu[2], g2, g3)*a[2, 2])/(2*sqrt(d[4])) - (a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])) - (-C*a[1, 1] + C*a[2, 2] + z*a[1, 1]*gamma[1] + 2*z*a[1, 2]*gamma[1] - z*a[1] + z*a[2, 2]*gamma[1] + z*a[2])/2)

Eq(varphi(z, 2), -z*(zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 1] - zeta(-z0 + z1 + mu[1], g2, g3)*a[1, 2] + zeta(-z0 + z1 + mu[2], g2, g3)*a[1, 2] - zeta(-z0 + z1 + mu[2], g2, g3)*a[2, 2] + zeta(z0 + z1 - mu[1], g2, g3)*a[1, 1] - zeta(z0 + z1 - mu[1], g2, g3)*a[1, 2] + zeta(z0 + z1 - mu[2], g2, g3)*a[1, 2] - zeta(z0 + z1 - mu[2], g2, g3)*a[2, 2])/(2*sqrt(d[4])) + (a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])) - (C*a[1, 1] - C*a[2, 2] + z*a[1, 1]*gamma[2] + 2*z*a[1, 2]*gamma[2] + z*a[1] + z*a[2, 2]*gamma[2] - z*a[2])/2)

### Effect on the differential equations

In this section we explore the new differential equations which, by design, we know must have meromorphic solutions:

In [17]:
gauge_subs_phi1 = [
    uU_sub.subs(j,1).args,
    vV_sub.subs(j,1).args,
    uU_sub.subs(j,2).args,
    vV_sub.subs(j,2).args,
]
gauge_subs_phi1_vals = [
    gauge_phi.subs(j,1).doit().args,
    gauge_phi.subs(j,2).doit().args
]


duhat_1_phi = Eq(
    diff(uhat(z, mu[j]), z).subs(j, 1), 
    (
        solve(duj.subs(j, 1).doit().subs(gauge_subs_phi1).doit(), diff(uhat(z, mu[j]), z).subs(j, 1))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify()
        .subs([
            (
                (b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])).expand(), 
                b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])
            ),
            (gamma[2] + gamma[1], 0)
        ])
        - vhat(z, mu[2])
    ).factor()
    + vhat(z, mu[2])
    
)
duhat_2_phi = Eq(
    diff(uhat(z, mu[j]), z).subs(j, 2), 
    (
        solve(duj.subs(j, 2).doit().subs(gauge_subs_phi1).doit(), diff(uhat(z, mu[j]), z).subs(j, 2))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify()
        .subs([
            (
                (b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])).expand(), 
                b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])
            ),
            (gamma[2] + gamma[1], 0)
        ])
        - vhat(z, mu[1])
    ).factor()
    + vhat(z, mu[1])
    
)
dvhat_1_phi = Eq(
    diff(vhat(z, mu[j]), z).subs(j, 1), 
    (
        solve(dvj.subs(j, 1).doit().subs(gauge_subs_phi1).doit(), diff(vhat(z, mu[j]), z).subs(j, 1))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify().simplify()
        .subs([
            (
                (b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])).expand(), 
                b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])
            ),
            (gamma[2] + gamma[1], 0)
        ])
        + uhat(z, mu[2])
    ).factor()
    - uhat(z, mu[2])
    
)
dvhat_2_phi = Eq(
    diff(vhat(z, mu[j]), z).subs(j, 2), 
    (
        solve(dvj.subs(j, 2).doit().subs(gauge_subs_phi1).doit(), diff(vhat(z, mu[j]), z).subs(j, 2))[0]
        .subs(gauge_subs_phi1_vals).doit().simplify().simplify()
        .subs([
            (
                (b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])).expand(), 
                b_j_coeffs[2].doit().rhs*2*(gamma[2] + gamma[1])
            ),
            (gamma[2] + gamma[1], 0)
        ])
        + uhat(z, mu[1])
    ).factor()
    - uhat(z, mu[1])
    
)
du_dv_hat_eqs = [
    duhat_1_phi,
    duhat_2_phi,
    dvhat_1_phi,
    dvhat_2_phi
]
uhat1_to_uhat2 = Eq(uhat(z, mu[1])*vhat(z, mu[1]), gamma[1] - gamma[2] + uhat(z, mu[2])*vhat(z, mu[2]))
uhat2_to_uhat1 = Eq(uhat(z, mu[2])*vhat(z, mu[2]), gamma[2] - gamma[1] + uhat(z, mu[1])*vhat(z, mu[1]))
du_dv_hat_eqs = [
    du_dv_hat_eqs[0].subs([uhat2_to_uhat1.args]),
    du_dv_hat_eqs[1].subs([uhat1_to_uhat2.args]),
    du_dv_hat_eqs[2].subs([uhat2_to_uhat1.args]),
    du_dv_hat_eqs[3].subs([uhat1_to_uhat2.args])
]
_ajk_to_bj_subs = [
    (gamma[2], -gamma[1]),
    (a[2], solve(b_j_coeffs[1].doit().subs(gamma[2], -gamma[1]), a[2])[0]),
    (a[1,2], solve(b_j_coeffs[2].doit().subs(ajk_syms).subs(gamma[2], -gamma[1]), a[1,2])[0])
]
du_dv_hat_eqs = [
    _.expand().subs(ajk_syms).subs(_ajk_to_bj_subs).simplify()
    for _ in du_dv_hat_eqs
]
du_dv_hat_subs_b = [_.args for _ in du_dv_hat_eqs]

for _ in du_dv_hat_eqs:
    _


Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])**2*vhat(z, mu[1])*b[2] + uhat(z, mu[1])*b[1]/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[2])**2*vhat(z, mu[2])*b[2] + uhat(z, mu[2])*b[1]/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), uhat(z, mu[1])*vhat(z, mu[1])**2*b[2] - uhat(z, mu[2]) - vhat(z, mu[1])*b[1]/2)

Eq(Derivative(vhat(z, mu[2]), z), -uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*b[2] - vhat(z, mu[2])*b[1]/2)

In [18]:
Hhat_uv = Eq(
    Hhat, 
    Sum(-b[2]/2*(uhat(z, mu[j])*vhat(z, mu[j]))**2 + b[1]/2*uhat(z, mu[j])*vhat(z, mu[j]), (j,1,2)) +
    uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2])
)
assert diff(Hhat_uv.rhs.doit(),z).subs(du_dv_hat_subs_b).simplify() == 0, 'Hamiltonian not conserved'

Hhat_uv

Eq(Hhat, uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2)))

In [19]:
du_dv_hat_from_H = [
    Eq(diff(uhat(z, mu[1]),z), diff(Hhat_uv.doit().rhs, vhat(z, mu[1]))),
    Eq(diff(uhat(z, mu[2]),z), diff(Hhat_uv.doit().rhs, vhat(z, mu[2]))),
    Eq(diff(vhat(z, mu[1]),z), -diff(Hhat_uv.doit().rhs, uhat(z, mu[1]))),
    Eq(diff(vhat(z, mu[1]),z), -diff(Hhat_uv.doit().rhs, uhat(z, mu[2])))
]

for _j in range(4):
    assert (
        (du_dv_hat_from_H[_j].rhs - du_dv_hat_eqs[_j].rhs).simplify() == 0
    ), f'diff eqn from Hamiltonian badly defined for eq: {_j + 1}'

In [20]:
a_0_eq_hat = a_0_eq.doit().subs(gauge_subs_phi1).subs(gauge_subs_phi1_vals).subs(gamma[2], -gamma[1]).simplify()
Hhat_b0 = Eq(
    Hhat_uv.lhs,
    Hhat_uv.rhs.doit().subs([
        a_0_eq_hat.args,
        (uhat(z, mu[1])*vhat(z, mu[1]), gamma[1] - rho(z)),
        (uhat(z, mu[2])*vhat(z, mu[2]), gamma[2] - rho(z)),
        b_j_coeffs[1].doit().args,
        b_j_coeffs[2].doit().args,
        (gamma[2], -gamma[1])
    ])
    .expand().collect(gamma[1], factor)
    .subs(
        b_j_coeffs[0].doit().rhs.subs(gamma[2], -gamma[1]).expand().collect(gamma[1], factor),
        b_j_coeffs[0].lhs
    )
    .subs(ajk_syms)
)

Hhat_b0 = Eq(
    Hhat_b0.lhs,
    b_j_coeffs[0].lhs 
    - b_j_coeffs[2].lhs*(gamma[1]-gamma[2])**2/4
    +
    (
        Hhat_b0.rhs 
        - b_j_coeffs[0].rhs.doit().expand().subs(ajk_syms).subs(gamma[2], -gamma[1]).collect(gamma[1], factor)
        + (b_j_coeffs[2].rhs.doit().subs(ajk_syms)*(gamma[1]-gamma[2])**2/4).subs(gamma[2], -gamma[1])
    ).expand().factor()
)

Hhat_uv_b = Hhat_uv.subs([Hhat_b0.args])


Hhat_b0
Hhat_uv_b

Eq(Hhat, -(gamma[1] - gamma[2])**2*b[2]/4 + b[0])

Eq(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0], uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2)))

### The connection between $\varphi$ and $\phi$

In the unified theory we in general write a gauge term using $\phi$ as follows:

In [21]:
duz_unified = Eq(
    Derivative(u(z, mu[j]), z)/u(z, mu[j]),
    (rhop(z) - rhop(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z)
)
dvz_unified = Eq(
    Derivative(v(z, mu[j]), z)/v(z, mu[j]),
    (rhop(z) + rhop(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z)
)


duz_unified
dvz_unified

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (\rho'(z) - \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z))

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]), (\rho'(z) + \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z))

Which for the coherent coupler is:

In [22]:
dphi_m_Q = Eq(
    Derivative(phi(z, mu[m]), z),
    (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) 
    - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
    + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
)

dphi_m_Q_Lam = Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])
dphi_m_Q_Lam_sum = Eq(
    Sum(Derivative(phi(z, mu[j]), z), (j,1,2)), 
    rho(z)*b[2]
)

Lam0_sum_b = Eq(
    Sum(Lamda[0, m], (m, 1, 2)),
    (Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))
    .subs(j,1).doit().simplify().subs(ajk_syms).subs(gamma[2], - gamma[1])
)
Lam1_sum_b = Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])


dphi_m_Q
dphi_m_Q_Lam
Lam0_sum_b
Lam1_sum_b
dphi_m_Q_Lam_sum

Eq(Derivative(phi(z, mu[m]), z), (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])

Eq(Sum(Lamda[0, m], (m, 1, 2)), 0)

Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

Eq(Sum(Derivative(phi(z, mu[j]), z), (j, 1, 2)), rho(z)*b[2])

The new equations of motion could thus be written:

In [23]:
# du_dv_hat_eqs_d_phi = [
#     duhat_1_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)]),
#     duhat_2_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)]),
#     dvhat_1_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)]),
#     dvhat_2_phi.subs([uv_phase_term_rho.args, (dphi_m_Q_Lam_sum.rhs, dphi_m_Q_Lam_sum.lhs)])
# ]
# du_dv_hat_eqs_d_phi = [
#     Eq(_.lhs, _.rhs.collect(_.rhs.args[1], factor)) for _ in du_dv_hat_eqs_d_phi
# ]

# for _ in du_dv_hat_eqs_d_phi:
#     _


## Projecting from the 3-mode degenerate FWM model to the 2-mode coherent coupler

In this section we look to find a projection from the 3-mode FWM model to the 2-mode coherent coupler model. This transform is motivated by observing the form of the solutions in the coherent coupler and recognising that they can be written as a ratio of Kronecker theta functions which naturally raises the question what is the system for these Kronecker theta functions, to which the answer is the 3-mode degenerate FWM system. 

Definign such a map from a general 3-mode degenerate FWM system to a 2-mode system is not too difficult and we do show such a generic map in a later section. However, we set ourselves the challenge of finding the map to our particualr coherent coupler system and so we wanted to express the degenerate FWM system in terms of the coefficients from the target 2-mode system and this was much harder to do. We did ultimately achieve the aim and proving it works in reverse once it is known is much simpler.

To begin, we remind ourselves of the form of the 2-mode coherent coupler solutions after gauge transforming away the essential singularity:

In [24]:
for _ in uv_hat_gauge_bs:
    _

Eq(uhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1] + epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1] - epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

Eq(uhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2] + epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2] - epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

We then introduce the following coordinate change by introducing three bar modes that are related to the two hat modes via the following, where the form and constant is directly inspired by the form of our analytic solutions:

In [25]:
u_v_hat_to_bar_eqs = [
    Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])/vbar(z, mu[3])),
    Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])/vbar(z, mu[3])),
    Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])/ubar(z, mu[3])),
    Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])/ubar(z, mu[3]))
]
u_v_hat_to_bar_subs = [_.args for _ in u_v_hat_to_bar_eqs]

for _ in u_v_hat_to_bar_eqs:
    _

Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])/vbar(z, mu[3]))

Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])/vbar(z, mu[3]))

Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])/ubar(z, mu[3]))

Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])/ubar(z, mu[3]))

For example, if our solutions for the three bar modes were the following Kronecker theta functions, then the above relations would map correctly to our solutions in the hat coordinates:

In [26]:
_ubar1 = Eq(
    ubar(z, mu[1]), 
    exp(epsilon[1])*upsilon/
    (sqrt(2)*sigma(-z0 + mu[1], g2, g3))
    *sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0,1])/sigma(z - z0, g2, g3)
)
_ubar2 = Eq(
    ubar(z, mu[2]),
    exp(epsilon[2])*upsilon/
    (sqrt(2)*sigma(-z0 + mu[2], g2, g3))
    *sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0,2])/sigma(z - z0, g2, g3)
)
_ubar3 = Eq(
    ubar(z, mu[3]), 
    sigma(z - z0 + z1, g2, g3)/sigma(z - z0, g2, g3)/sigma(z1, g2, g3)
)
_vbar1 = Eq(
    vbar(z, mu[1]), 
    exp(-epsilon[1])/
    (sqrt(2)*sigma(-z0 + mu[1], g2, g3)*upsilon)
    *sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0,1])/sigma(z - z0, g2, g3)
)
_vbar2 = Eq(
    vbar(z, mu[2]), 
    exp(-epsilon[2])/
    (sqrt(2)*sigma(-z0 + mu[2], g2, g3)*upsilon)
    *sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0,2])/sigma(z - z0, g2, g3)
)

_vbar3 = Eq(
    vbar(z, mu[3]), 
    sigma(z - z0 - z1, g2, g3)/sigma(z - z0, g2, g3)/sigma(z1, g2, g3)
)

bar_sig_uv = [
    _ubar1,
    _ubar2,
    _ubar3,
    _vbar1,
    _vbar2,
    _vbar3
]
bar_sig_uv_subs = [_.args for _ in bar_sig_uv]


B_wpz1_z0_muj_to_d_lam1_gamj_sqrt = Eq(
    sqrt(wpp(z1, g2, g3))/sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)),
    sqrt(gamma[j] - lamda[1])*d[4]**Rational(1,4),
)

_c1 = Eq(
    uv_hat_gauge_bs[0].lhs/u_v_hat_to_bar_eqs[0].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[0].rhs/u_v_hat_to_bar_eqs[0].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
    .subs([B_wpz1_z0_muj_to_d_lam1_gamj_sqrt.subs(j,1).args])
)
_c3 = Eq(
    uv_hat_gauge_bs[1].lhs/u_v_hat_to_bar_eqs[2].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[1].rhs/u_v_hat_to_bar_eqs[2].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
    .subs([B_wpz1_z0_muj_to_d_lam1_gamj_sqrt.subs(j,1).args])
)

_c2 = Eq(
    uv_hat_gauge_bs[2].lhs/u_v_hat_to_bar_eqs[1].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[2].rhs/u_v_hat_to_bar_eqs[1].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
    .subs([B_wpz1_z0_muj_to_d_lam1_gamj_sqrt.subs(j,2).args])
)
_c4 = Eq(
    uv_hat_gauge_bs[3].lhs/u_v_hat_to_bar_eqs[3].lhs.subs([_.args for _ in bar_sig_uv]), 
    (uv_hat_gauge_bs[3].rhs/u_v_hat_to_bar_eqs[3].rhs.subs([_.args for _ in bar_sig_uv]))
    .simplify()
    .subs([B_wpz1_z0_muj_to_d_lam1_gamj_sqrt.subs(j,2).args])
)

assert _c1
assert _c2
assert _c3
assert _c4

for _ in bar_sig_uv:
    _

print()
for _ in u_v_hat_to_bar_eqs:
    _.subs(bar_sig_uv_subs)

Eq(ubar(z, mu[1]), sqrt(2)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1])*exp(epsilon[1])*upsilon/(2*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)))

Eq(ubar(z, mu[2]), sqrt(2)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2])*exp(epsilon[2])*upsilon/(2*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)))

Eq(ubar(z, mu[3]), sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)*sigma(z - z0, g2, g3)))

Eq(vbar(z, mu[1]), sqrt(2)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1])*exp(-epsilon[1])/(2*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*upsilon))

Eq(vbar(z, mu[2]), sqrt(2)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2])*exp(-epsilon[2])/(2*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*upsilon))

Eq(vbar(z, mu[3]), sigma(z - z0 - z1, g2, g3)/(sigma(z1, g2, g3)*sigma(z - z0, g2, g3)))

Eq(uhat(z, mu[1]), sqrt(gamma[1] - lamda[1])*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*beta[0] + z*r[0, 1])*exp(epsilon[1])*upsilon/(sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)))

Eq(uhat(z, mu[2]), sqrt(gamma[2] - lamda[1])*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(-z*beta[0] + z*r[0, 2])*exp(epsilon[2])*upsilon/(sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)))

Eq(vhat(z, mu[1]), sqrt(gamma[1] - lamda[1])*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*beta[0] - z*r[0, 1])*exp(-epsilon[1])/(sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*upsilon))

Eq(vhat(z, mu[2]), sqrt(gamma[2] - lamda[1])*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(z*beta[0] - z*r[0, 2])*exp(-epsilon[2])/(sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*upsilon))

So, we have defined how we want to go from three modes to two and what the solutions should look like in each system, but we do not yet know how to get the system of the 3 individual bar modes from what we know about the 2 hat modes. We do know that there is an intermodal power conservation law for the hat modes and this translates into a relation between the three modal powers in the bar coordinates:

In [27]:
uv_hat_const = Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])
uv_hat_bar_const = Eq(uv_hat_const.lhs.subs(u_v_hat_to_bar_subs), uv_hat_const.rhs)
uv_bar_3_12 = Eq(
    ubar(z, mu[3])*vbar(z, mu[3]), 
    solve(uv_hat_bar_const, ubar(z, mu[3])*vbar(z, mu[3]))[0]
    .expand()
    .collect([ubar(z, mu[1])*vbar(z, mu[1]), ubar(z, mu[2])*vbar(z, mu[2])], factor)
)

uv_hat_const
uv_hat_bar_const
uv_bar_3_12

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

Eq(2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3])) - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3])), gamma[1] - gamma[2])

Eq(ubar(z, mu[3])*vbar(z, mu[3]), 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])/(gamma[1] - gamma[2]) - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])/(gamma[1] - gamma[2]))

We also know from the form of our Kronceker theta function solutions for the bar modes that they would yield the below expression in terms of $\wp$ and thus they would yield corresponding intermodal power conservation laws among bar modes. Thus, we will assume the 3-mode system we seek has this property and that it will have 2 independent intermodal pwoer conservation laws. Remark: establishing intermodal power conservation laws in both coordinate systems is sufficient to define a Mobius transform between any modal power in one coordinate system and any other in the other coordinate system.

In [28]:
# Define modal powers in bar mdoes in terms of wp
_sigma_p_id_flipped = Eq(sigma_p_identity.rhs, sigma_p_identity.lhs)

uv_bar_wps = [
    Eq(
        ubar(z, mu[_j+1])*vbar(z, mu[_j+1]),
        (ubar(z, mu[_j+1])*vbar(z, mu[_j+1]))
        .subs(bar_sig_uv_subs)
        .simplify().expand()
        .subs([
            _sigma_p_id_flipped.subs([(x, z - z0), (y, mu[1] - z0)]).args,
            _sigma_p_id_flipped.subs([(x, z - z0), (y, mu[2] - z0)]).args,
            _sigma_p_id_flipped.subs([(x, z - z0), (y, z1)]).args
        ])
    )
    for _j in range(3)
]
uv_bar_wp_subs = [_.args for _ in uv_bar_wps]

# Derive corresponding intermodal power conservation laws
wp_muj_d_lam_gam = Eq(
    sqrt(d[4])/4*wppz1_djs.lhs/B_wpz1_z0_muj_to_d_lam1_gamj.lhs, 
    sqrt(d[4])/4*wppz1_djs.rhs/B_wpz1_z0_muj_to_d_lam1_gamj.rhs
)
_sigma_p_id_flipped = Eq(sigma_p_identity.rhs, sigma_p_identity.lhs)
uv_bar_sig_wp_ids = [
    Eq(
        ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]),
        (ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]))
        .subs(uv_bar_wp_subs)
    ),
    Eq(
        2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]),
        (2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]))
        .subs(uv_bar_wp_subs)
    ),
    Eq(
        2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]),
        (2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]))
        .subs(uv_bar_wp_subs)
    )
]

_wp_muj_subs_d = [
    wp_muj_d_lam_gam.subs(j,1).args,
    wp_muj_d_lam_gam.subs(j,2).args,
    (
        (wp_muj_d_lam_gam.subs(j,1).lhs - wp_muj_d_lam_gam.subs(j,2).lhs)/2, 
        (wp_muj_d_lam_gam.subs(j,1).rhs - wp_muj_d_lam_gam.subs(j,2).rhs)/2
    )
]
uv_bar_sig_dj_ids = [ _.subs(_wp_muj_subs_d) for _ in uv_bar_sig_wp_ids ]


for _ in uv_bar_wps:
    _
print()
for _ in uv_bar_sig_wp_ids:
    _
print() 
wp_muj_d_lam_gam
print()
for _ in uv_bar_sig_dj_ids:
    _

Eq(ubar(z, mu[1])*vbar(z, mu[1]), -wp(z - z0, g2, g3)/2 + wp(-z0 + mu[1], g2, g3)/2)

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -wp(z - z0, g2, g3)/2 + wp(-z0 + mu[2], g2, g3)/2)

Eq(ubar(z, mu[3])*vbar(z, mu[3]), wp(z1, g2, g3) - wp(z - z0, g2, g3))

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), wp(-z0 + mu[1], g2, g3)/2 - wp(-z0 + mu[2], g2, g3)/2)

Eq(2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]), -wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))

Eq(2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]), -wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))

Eq(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3), -d[5]/(4*(-gamma[j] + lamda[1])))

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), d[5]/(8*(-gamma[2] + lamda[1])) - d[5]/(8*(-gamma[1] + lamda[1])))

Eq(2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]), -d[5]/(4*(-gamma[1] + lamda[1])))

Eq(2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]), -d[5]/(4*(-gamma[2] + lamda[1])))

In [29]:
rhobar_uvbar_mean = Eq(rhobar(z), -Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j,1,3))/3)
rhobar_uvbar_mean_wp = Eq(
    rhobar_uvbar_mean.lhs, 
    rhobar_uvbar_mean.rhs.doit()
    .subs(uv_bar_wp_subs)
)
wp_rhobar = Eq(wp(z-z0,g2,g3), solve(rhobar_uvbar_mean_wp, wp(z-z0,g2,g3))[0])
uv_bar_rho_wp_consts = [_.subs([wp_rhobar.args]) for _ in uv_bar_wps]
gamma_bar_js_wps = [Eq(gammabar[_j+1], uv_bar_rho_wp_consts[_j].rhs.subs(rhobar(z),0)) for _j in range(3)]
_wp_muj_subs_gam = solve(
    [wp_muj_d_lam_gam.subs(j, _j+1) for _j in range(2)], 
    [wp(-z0 +mu[1],g2,g3), wp(-z0 +mu[2],g2,g3)]
)
gamma_bar_js_gamma = [
    Eq(_.lhs, apart(_.rhs.subs(_wp_muj_subs_gam).factor().subs(lamda[1],x), x).subs(x, lamda[1])) 
    for _ in gamma_bar_js_wps
]
gamma_bar_js_gamma_b = [_.subs([(gamma[2], -gamma[1])]).expand().simplify() for _ in gamma_bar_js_gamma]
gamma_bar_js_gamma_subs = [_.args for _ in gamma_bar_js_gamma]
gamma_bar_sum = Eq(
    Sum(gammabar[j], (j,1,3)), 
    Sum(gammabar[j], (j,1,3))
    .doit().subs(gamma_bar_js_gamma_subs)
    .simplify()
)

wp_to_gambar_subs = [(_.rhs, _.lhs) for _ in gamma_bar_js_wps]
uv_bar_rho_gambar = [_.subs(wp_to_gambar_subs) for _ in uv_bar_rho_wp_consts]
uv_bar_rho_gambar_subs = [_.args for _ in uv_bar_rho_gambar]

# gamma as gammabar

d6_lam_gam = Eq(d[6], d[5]/(gamma[1]**2 - lamda[1]**2)/8)
d6_lam = Eq(
    d6_lam_gam.lhs, 
    d6_lam_gam.rhs
    .subs([d5_dj.args, gamma_lamda_bj_gam1.args])
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
)
d5_as_d6 = Eq(d[5], solve(d6_lam_gam, d[5])[0])

gamma1_lam_as_gammabar_subs = solve([
    gamma_bar_js_gamma_b[0].subs([d5_as_d6.args]), 
    gamma_bar_js_gamma_b[1].subs([d5_as_d6.args])
], [gamma[1], lamda[1]])

    
    


rhobar_uvbar_mean
rhobar_uvbar_mean_wp
wp_rhobar
for _ in uv_bar_rho_wp_consts:
    _
print()
for _ in gamma_bar_js_wps:
    _
print()
for _ in gamma_bar_js_gamma:
    _
print()
for _ in gamma_bar_js_gamma_b:
    _
print()
for _ in uv_bar_rho_gambar:
    _
print()
gamma_bar_sum
print()
d6_lam_gam
d6_lam
print()
for _k in gamma1_lam_as_gammabar_subs:
    Eq(_k, gamma1_lam_as_gammabar_subs[_k].simplify())

Eq(rhobar(z), -Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/3)

Eq(rhobar(z), -wp(z1, g2, g3)/3 + 2*wp(z - z0, g2, g3)/3 - wp(-z0 + mu[1], g2, g3)/6 - wp(-z0 + mu[2], g2, g3)/6)

Eq(wp(z - z0, g2, g3), 3*rhobar(z)/2 + wp(z1, g2, g3)/2 + wp(-z0 + mu[1], g2, g3)/4 + wp(-z0 + mu[2], g2, g3)/4)

Eq(ubar(z, mu[1])*vbar(z, mu[1]), -3*rhobar(z)/4 - wp(z1, g2, g3)/4 + 3*wp(-z0 + mu[1], g2, g3)/8 - wp(-z0 + mu[2], g2, g3)/8)

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -3*rhobar(z)/4 - wp(z1, g2, g3)/4 - wp(-z0 + mu[1], g2, g3)/8 + 3*wp(-z0 + mu[2], g2, g3)/8)

...

Eq(Sum(gammabar[j], (j, 1, 3)), 0)

Eq(d[6], d[5]/(8*(gamma[1]**2 - lamda[1]**2)))

Eq(d[6], -(2*(2*b[0]*b[2] + b[1]**2 - 4)*lamda[1] + 2*b[0]*b[1] + 6*b[1]*b[2]*lamda[1]**2 + 4*b[2]**2*lamda[1]**3)/(2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2))

Eq(gamma[1], (gammabar[1] - gammabar[2])/(2*d[6]))

Eq(lamda[1], (gammabar[1] + gammabar[2])/d[6])

We can obtain four differential equations in bar coordinates by substituting the transforms into the original four hat coordinate differential equations. We arrange these for modes 1 and 2 and then need to find a way to elliminate the derivative of mode 3.

In [30]:
bar_d_uv_eqs = [_.subs(u_v_hat_to_bar_subs).doit() for _ in du_dv_hat_eqs]
bar_d_uv_eqs = [
    Eq(diff(ubar(z, mu[1]),z), solve(bar_d_uv_eqs[0], diff(ubar(z, mu[1]),z))[0]),
    Eq(diff(ubar(z, mu[2]),z), solve(bar_d_uv_eqs[1], diff(ubar(z, mu[2]),z))[0]),
    Eq(diff(vbar(z, mu[1]),z), solve(bar_d_uv_eqs[2], diff(vbar(z, mu[1]),z))[0]),
    Eq(diff(vbar(z, mu[2]),z), solve(bar_d_uv_eqs[3], diff(vbar(z, mu[2]),z))[0])
]
bar_d_uv_subs = [_.args for _ in bar_d_uv_eqs]


assert (
    diff(Hhat_uv.rhs.doit().subs(u_v_hat_to_bar_subs),z)
    .subs(bar_d_uv_subs).simplify() == 0
), 'hamiltonian in bar coords not conserved with bar diff eqns'

for _ in bar_d_uv_eqs:
    _

Eq(Derivative(ubar(z, mu[1]), z), -2*ubar(z, mu[1])**2*vbar(z, mu[1])*b[2]*gamma[1]/(ubar(z, mu[3])*vbar(z, mu[3])) + 2*ubar(z, mu[1])**2*vbar(z, mu[1])*b[2]*lamda[1]/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[1])*b[1]/2 + ubar(z, mu[1])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3]) + sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])*vbar(z, mu[3])/(sqrt(gamma[1] - lamda[1])*ubar(z, mu[3])))

Eq(Derivative(ubar(z, mu[2]), z), sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])*vbar(z, mu[3])/(sqrt(gamma[2] - lamda[1])*ubar(z, mu[3])) - 2*ubar(z, mu[2])**2*vbar(z, mu[2])*b[2]*gamma[2]/(ubar(z, mu[3])*vbar(z, mu[3])) + 2*ubar(z, mu[2])**2*vbar(z, mu[2])*b[2]*lamda[1]/(ubar(z, mu[3])*vbar(z, mu[3])) + ubar(z, mu[2])*b[1]/2 + ubar(z, mu[2])*Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3]))

Eq(Derivative(vbar(z, mu[1]), z), 2*ubar(z, mu[1])*vbar(z, mu[1])**2*b[2]*gamma[1]/(ubar(z, mu[3])*vbar(z, mu[3])) - 2*ubar(z, mu[1])*vbar(z, mu[1])**2*b[2]*lamda[1]/(ubar(z, mu[3])*vbar(z, mu[3])) - vbar(z, mu[1])*b[1]/2 + vbar(z, mu[1])*Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]) - sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])/(sqrt(gamma[1] - lamda[1])*vbar(z, mu[3])))

Eq(Derivative(vbar(z, mu[2]), z), -sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])/(sqrt(gamma[2] - lamda[1])*vbar(z, mu[3])) + 2*ubar(z, mu[2])*vbar(z, mu[2])**2*b[2]*gamma[2]/(ubar(z, mu[3])*vbar(z, mu[3])) - 2*ubar(z, mu[2])*vbar(z, mu[2])**2*b[2]*lamda[1]/(ubar(z, mu[3])*vbar(z, mu[3])) - vbar(z, mu[2])*b[1]/2 + vbar(z, mu[2])*Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]))

From these differential equations and the intermodal power conservation law in bar coordinates we can derive a differential equation for the mode power in mode 3 and, by modal power conservation, modes 1 and 2:

In [31]:
_uv_12_pow_diff = Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z) - Derivative(ubar(z, mu[2])*vbar(z, mu[2]),z)

_uv13_duv3 = Eq(
    (
        ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3]))
        *Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z)
    )
    .doit().expand(),
    ubar(z, mu[1])*vbar(z, mu[1])/(ubar(z, mu[3])*vbar(z, mu[3]))
    *Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z)
    
)
_uv23_duv3 = Eq(
    (
        ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3]))
        *Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z)
    )
    .doit().expand(),
    ubar(z, mu[2])*vbar(z, mu[2])/(ubar(z, mu[3])*vbar(z, mu[3]))
    *Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z)
    
)

duv_12_diff = Eq(
    _uv_12_pow_diff,
    _uv_12_pow_diff
    .doit()
    .subs(bar_d_uv_subs)
    .simplify()
    .subs([
        _uv13_duv3.args,
        _uv23_duv3.args
    ])
    .collect(Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z), factor)
)
uvbar2_as_uvbar1 = Eq(
    ubar(z, mu[2])*vbar(z, mu[2]), 
    solve(uv_bar_sig_dj_ids[0], ubar(z, mu[2])*vbar(z, mu[2]))[0]
    .expand().collect(ubar(z, mu[1])*vbar(z, mu[1]), factor)
)
uvbar3_as_uvbar1 = Eq(
    ubar(z, mu[3])*vbar(z, mu[3]), 
    solve(uv_bar_sig_dj_ids[1], ubar(z, mu[3])*vbar(z, mu[3]))[0]
    .expand().collect(ubar(z, mu[1])*vbar(z, mu[1]), factor)
)
duv_3bar_diff = Eq(
    duv_12_diff.lhs
    .subs([uvbar2_as_uvbar1.args])
    .doit(),
    duv_12_diff.rhs
    .subs([uvbar2_as_uvbar1.args])
    .subs([(Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z), x)])
)
duv_3bar_diff = Eq(
    Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z), 
    solve(duv_3bar_diff, x)[0].factor()
)
# By intermodal power conservation
_uvbar_3_ = ubar(z, mu[3])*vbar(z, mu[3])
duv_1bar_diff = Eq(
    duv_3bar_diff.lhs
    .subs(_uvbar_3_, solve(uv_bar_sig_dj_ids[1], _uvbar_3_)[0])
    .doit().simplify()
    + 2*(Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z) - diff(ubar(z, mu[1])*vbar(z, mu[1]),z).doit()), 
    duv_3bar_diff.rhs
)
duv_1bar_diff = Eq(duv_1bar_diff.lhs/2, duv_1bar_diff.rhs/2)
duv_2bar_diff = Eq(
    duv_3bar_diff.lhs
    .subs(_uvbar_3_, solve(uv_bar_sig_dj_ids[2], _uvbar_3_)[0])
    .doit().simplify()
    + 2*(Derivative(ubar(z, mu[2])*vbar(z, mu[2]),z) - diff(ubar(z, mu[2])*vbar(z, mu[2]),z).doit()),
    duv_3bar_diff.rhs
)
duv_2bar_diff = Eq(duv_2bar_diff.lhs/2, duv_2bar_diff.rhs/2)

duv_bar_diffs = [
    duv_1bar_diff,
    duv_2bar_diff,
    duv_3bar_diff
]

# Get conjugate derivatives in terms of unconjugated (not yet used)
dvbar_sols = solve([_.doit() for _ in duv_bar_diffs], [diff(vbar(z,mu[_j+1]),z) for _j in range(3)])

duv_12_diff
print()
for _ in duv_bar_diffs:
    _

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z) - Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z), (ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]))*Derivative(ubar(z, mu[3])*vbar(z, mu[3]), z)/(ubar(z, mu[3])*vbar(z, mu[3])) + (ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*(gamma[1] - gamma[2])/(sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*ubar(z, mu[3])*vbar(z, mu[3])))

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), 4*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

Eq(Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z), 4*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

Eq(Derivative(ubar(z, mu[3])*vbar(z, mu[3]), z), 8*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

We now transform the hat Hamiltonian to bar coordinates. This may not end up being the canonical Hamiltonian we seek in bar coorindates but we would expect it to be a conserved quantity of that system and closely related to the Hamiltonian we seek. We multiply by $2/d_5$ so that we recover the expected differential equations for modal powers. We confirm it is conserved using the transformed differential equations.

In [32]:
_norm = 2/d[5]
Hbar_uv = Eq(
    0, 
    (
        _norm*(Hhat_uv.lhs - Hhat_uv.rhs.doit())
        .subs(u_v_hat_to_bar_subs)
        *(ubar(z, mu[3])*vbar(z, mu[3]))**2 
    )
    .expand()
    .collect([
        sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1]),
        (ubar(z, mu[1])*vbar(z, mu[1]))**2,
        (ubar(z, mu[2])*vbar(z, mu[2]))**2,
        (ubar(z, mu[3])*vbar(z, mu[3]))**2,
        ubar(z, mu[1])*vbar(z, mu[1])*ubar(z, mu[2])*vbar(z, mu[2]),
        ubar(z, mu[1])*vbar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[3]),
        ubar(z, mu[3])*vbar(z, mu[3])*ubar(z, mu[2])*vbar(z, mu[2])
    ], factor)
    .subs([Hhat_b0.args])
)

Hbar_uv_b = Eq(
    (Hbar_uv.lhs - Hbar_uv.rhs.subs([(Hhat,0), (b[0], 0), (b[1], 0), (b[2], 0)])).factor(),
    (Hbar_uv.rhs - Hbar_uv.rhs.subs([(Hhat,0), (b[0], 0), (b[1], 0), (b[2], 0)]))
    .subs([Hhat_b0.args])
)

assert (
    diff(Hbar_uv.rhs, z)
    .subs(bar_d_uv_subs)
    .subs(b[0], solve(Hbar_uv, b[0])[0]).simplify()
    .simplify()
) == 0, 'hamiltonian in bar coords not conserved with bar diff eqns'

Hbar_uv
print()
Hbar_uv_b

Eq(0, 2*(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])**2*vbar(z, mu[3])**2/d[5] - 4*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5] + 4*(gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])**2*b[2]/d[5] - 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*b[1]/d[5] + 4*(gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])**2*b[2]/d[5] - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])*vbar(z, mu[2])*vbar(z, mu[3])*b[1]/d[5])

Eq(4*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5], 2*(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])**2*vbar(z, mu[3])**2/d[5] + 4*(gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])**2*b[2]/d[5] - 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*b[1]/d[5] + 4*(gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])**2*b[2]/d[5] - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])*vbar(z, mu[2])*vbar(z, mu[3])*b[1]/d[5])

If we do treat this transformed conserved quantity as a canonical Hamiltonian we obtain differential equations that do reproduce the correct form for the modal power evolution that we obtained earlier by direct transformation. This tells us that we have the modal power part correct but from experience we know we may not have the phase/gauge part correct yet.

In [33]:
d_u_v_bars = [
    Eq(diff(ubar(z, mu[1]),z), diff(Hbar_uv.rhs, vbar(z, mu[1]))),
    Eq(diff(ubar(z, mu[2]),z), diff(Hbar_uv.rhs, vbar(z, mu[2]))),
    Eq(diff(ubar(z, mu[3]),z), diff(Hbar_uv.rhs, vbar(z, mu[3]))),
    Eq(diff(vbar(z, mu[1]),z), -diff(Hbar_uv.rhs, ubar(z, mu[1]))),
    Eq(diff(vbar(z, mu[2]),z), -diff(Hbar_uv.rhs, ubar(z, mu[2]))),
    Eq(diff(vbar(z, mu[3]),z), -diff(Hbar_uv.rhs, ubar(z, mu[3]))) 
]
d_u_v_bars_subs = [_.args for _ in d_u_v_bars]


duvj_bars_trnsd_ham = [
    Eq(
        Derivative(ubar(z, mu[1])*vbar(z,mu[1]),z), 
        diff(ubar(z, mu[1])*vbar(z,mu[1]),z)
        .subs(d_u_v_bars_subs)
        .simplify()
    ),
    Eq(
        Derivative(ubar(z, mu[2])*vbar(z,mu[2]),z), 
        diff(ubar(z, mu[2])*vbar(z,mu[2]),z)
        .subs(d_u_v_bars_subs)
        .simplify()
    ),
    Eq(
        Derivative(ubar(z, mu[3])*vbar(z,mu[3]),z), 
        diff(ubar(z, mu[3])*vbar(z,mu[3]),z)
        .subs(d_u_v_bars_subs)
        .simplify()
    )
]

for _j in range(3):
    assert duv_bar_diffs[_j].subs([duvj_bars_trnsd_ham[_j].args]), 'mode power diff not well defined'

for _ in d_u_v_bars:
    _
print()  
for _ in duvj_bars_trnsd_ham:
    _

Eq(Derivative(ubar(z, mu[1]), z), -4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])*vbar(z, mu[3])**2/d[5] + 8*(gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])*b[2]/d[5] - 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[3])*b[1]/d[5])

Eq(Derivative(ubar(z, mu[2]), z), -4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*vbar(z, mu[1])*vbar(z, mu[3])**2/d[5] + 8*(gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])*b[2]/d[5] - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])*vbar(z, mu[3])*b[1]/d[5])

Eq(Derivative(ubar(z, mu[3]), z), 4*(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])**2*vbar(z, mu[3])/d[5] - 8*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/d[5] - 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*b[1]/d[5] - 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])*vbar(z, mu[2])*b[1]/d[5])

Eq(Derivative(vbar(z, mu[1]), z), 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])**2/d[5] - 8*(gamma[1] - lamda[1])**2*ubar(z, mu[1])*vbar(z, mu[1])**2*b[2]/d[5] + 2*(gamma[1] - lamda[1])*ubar(z, mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*b[1]/d[5])

Eq(Derivative(vbar(z, mu[2]), z), 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])**2/d[5] - 8*(gamma[2] - lamda[1])**2*ubar(z, mu[2])*vbar(z, mu[2])**2*b[2]/d[5] + 2*(gamma[2] - lamda[1])*ubar(z, mu[3])*vbar(z, mu[2])*vbar(z, mu[3])*b[1]/d[5])

Eq(Derivative(vbar(z, mu[3]), z), -4*(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])*vbar(z, mu[3])**2/d[5] + 8*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])/d[5] + 2*(gamma[1] - lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*vbar(z, mu[3])*b[1]/d[5] + 2*(gamma[2] - lamda[1])*ubar(z, mu[2])*vbar(z, mu[2])*vbar(z, mu[3])*b[1]/d[5])

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), 4*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

Eq(Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z), 4*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

Eq(Derivative(ubar(z, mu[3])*vbar(z, mu[3]), z), 8*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

We are also able to confirm that the differential equation for the modal power evolution in mode 1 contains no quartic term as we would expect given the anticipated form of the solutions:

In [34]:
uvbar123_to_1_subs = [
    (ubar(z, mu[2])*vbar(z, mu[2]), solve(uv_bar_sig_dj_ids[0], ubar(z, mu[2])*vbar(z, mu[2]))[0]),
    (ubar(z, mu[3])*vbar(z, mu[3]), solve(uv_bar_sig_dj_ids[1], ubar(z, mu[3])*vbar(z, mu[3]))[0])
]

duv1bar_sqrd = Eq(
    duvj_bars_trnsd_ham[0].lhs**2, 
    ((duvj_bars_trnsd_ham[0].rhs**2 - Hbar_uv_b.lhs**2).factor() + Hbar_uv_b.rhs**2)
    .subs([(gamma[2], - gamma[1])]) 
    .subs(uvbar123_to_1_subs)
    .subs([(gamma[2], - gamma[1])])
    .subs(uvbar123_to_1_subs)
    .expand().collect(ubar(z, mu[1])*vbar(z, mu[1]), factor)
)

_coeff4_uv1 = duv1bar_sqrd.rhs.coeff(ubar(z, mu[1])*vbar(z, mu[1]), 4)
_coeff3_uv1 = duv1bar_sqrd.rhs.coeff(ubar(z, mu[1])*vbar(z, mu[1]), 3)
coeff4_uv1_eq = Eq(_coeff4_uv1, _coeff4_uv1.subs([gamma_lamda_bj_gam1.args]).simplify())
assert coeff4_uv1_eq.rhs == 0, 'error: coeff of quartic term non-zero'

coeff3_uv1_eq = Eq(
    _coeff3_uv1,
    _coeff3_uv1
    .subs([gamma_lamda_bj_gam1.args])
    .subs([d5_dj.args])
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .simplify()
)

duv1bar_sqrd = duv1bar_sqrd.subs([coeff4_uv1_eq.args, coeff3_uv1_eq.args])

gamma_lamda_bj_gam1
print()
duv1bar_sqrd

Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z)**2, -8*ubar(z, mu[1])**3*vbar(z, mu[1])**3 + (6*b[0]**2 + 6*b[0]*b[1]*gamma[1] + 6*b[0]*b[1]*lamda[1] + 2*b[0]*b[2]*gamma[1]**2 + 8*b[0]*b[2]*gamma[1]*lamda[1] + 2*b[0]*b[2]*lamda[1]**2 + b[1]**2*gamma[1]**2 + 4*b[1]**2*gamma[1]*lamda[1] + b[1]**2*lamda[1]**2 + 6*b[1]*b[2]*gamma[1]**2*lamda[1] + 6*b[1]*b[2]*gamma[1]*lamda[1]**2 + 6*b[2]**2*gamma[1]**2*lamda[1]**2 + 20*gamma[1]**2 - 16*gamma[1]*lamda[1] - 4*lamda[1]**2)*ubar(z, mu[1])**2*vbar(z, mu[1])**2/(gamma[1] - lamda[1])**2 - (2*b[0]**2 + 3*b[0]*b[1]*gamma[1] + b[0]*b[1]*lamda[1] + 2*b[0]*b[2]*gamma[1]**2 + 2*b[0]*b[2]*gamma[1]*lamda[1] + b[1]**2*gamma[1]**2 + b[1]**2*gamma[1]*lamda[1] + b[1]*b[2]*gamma[1]**3 + 3*b[1]*b[2]*gamma[1]**2*lamda[1] + 2*b[2]**2*gamma[1]**3*lamda[1] + 4*gamma[1]**2 - 4*gamma[1]*lamda[1])*ubar(z, mu[1])*vbar(z, mu[1])*d[5]/(4*(gamma[1] - lamda[1])**3) + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)**2*d[5]**2/(64*(gamma[1] - lamda[1])**4))

We confirm that when written in Weierstrass form the modal power differential equation in the new bar coordinates uses the same elliptic invariants $g_2,g_3$ as in the original coordinates:

In [35]:
## -- Lamda reduction -- 

# seed: lam^4 already reduced
lam4_rhs = solve(gamma_lamda_bj_gam1.expand(), lamda[1]**4)[0]
lam_step_down = [(lamda[1]**4, lam4_rhs)]

# build higher powers by multiplying previous result by lam
# and immediately reducing lam^4 → rhs
max_power = 13
for _n in range(5, max_power + 1):
    prev_rhs = lam_step_down[-1][1]
    new_rhs = (prev_rhs * lamda[1]).expand().subs(lamda[1]**4, lam4_rhs)
    lam_step_down.append((lamda[1]**_n, new_rhs))
    
    
## -- gamma1 reduction --
max_gam_red = 14
gamma1_reducers = [
    (
        gamma[1]**_j, 
        gamma_lamda_bj_gam1.rhs**(_j//2)*gamma[1]**(_j % 2)
    )
    for _j in range(2, max_gam_red)
]

for _ in gamma1_reducers:
    assert Eq(*_).subs(gamma_lamda_bj_gam1.rhs, gamma_lamda_bj_gam1.lhs)
    
# Convert d[5] to bj and lamda
d_j_b_lam_subs = [
    _.subs([_.args for _ in c_j_coeffs]).doit().subs([(gamma[2], - gamma[1])])
    .subs([gamma_lamda_bj_gam1.args])
    .args
    for _ in d_j_coeffs
]

# Simplify cubic coeff to 4 and remove quadratic
w_ulbar_sub = Eq(ubar(z, mu[1])*vbar(z,mu[1]), -w(z)/2 + c[0])
dw_uv1bar = Eq(
    duv1bar_sqrd.lhs.subs([w_ulbar_sub.args]).doit()*4,
    (duv1bar_sqrd.rhs.subs([w_ulbar_sub.args])*4)
    .expand().collect(w(z), factor)
)
_c0_no_quad = Eq(c[0], solve(dw_uv1bar.rhs.coeff(w(z),2),c[0])[0])
dw_uv1bar = Eq(
    dw_uv1bar.lhs, 
    dw_uv1bar.rhs.subs([_c0_no_quad.args])
    .expand().collect(w(z), factor)
)

g2bar_as_bd = Eq(gbar2, -dw_uv1bar.rhs.coeff(w(z), 1))
g3bar_as_bd = Eq(gbar3, -dw_uv1bar.rhs.coeff(w(z), 0))

dw_uv1bar = Eq(
    dw_uv1bar.lhs, 
    dw_uv1bar.rhs.subs([
        (-(g2bar_as_bd.rhs*w(z)).factor(), -(g2bar_as_bd.lhs*w(z)).factor()),
         (-g3bar_as_bd.rhs, -g3bar_as_bd.lhs)
    ])
)

# Confirm that gbar2 = g2
_gam_b2_lam_denom_g2 = 12*(gamma[1] - lamda[1])**4*b[2]
_g2_g2bar = Eq(
    (g2_dj.lhs - g2bar_as_bd.lhs)*_gam_b2_lam_denom_g2,
    (g2_dj.rhs*_gam_b2_lam_denom_g2 - g2bar_as_bd.rhs*_gam_b2_lam_denom_g2)
    .subs([d5_dj.args])
    .subs(d_j_b_lam_subs)
    .expand().collect(gamma[1])
    .subs(gamma1_reducers)
    .expand()
)
g2bar_as_g2 = Eq(gbar2, solve(_g2_g2bar, gbar2)[0])
assert g2bar_as_g2.subs(gbar2, g2)

# Confirm that gbar3 = g3
_gam_b2_lam_denom_g3 = 432*(gamma[1] - lamda[1])**6
_g3_g3bar = Eq(
    (g3_dj.lhs - g3bar_as_bd.lhs)*_gam_b2_lam_denom_g3,
    (g3_dj.rhs*_gam_b2_lam_denom_g3 - g3bar_as_bd.rhs*_gam_b2_lam_denom_g3)
    .subs([d5_dj.args])
    .subs(d_j_b_lam_subs)
    .expand().collect(gamma[1])
    .subs(gamma1_reducers)
    .expand().collect(lamda[1])
    .subs(lam_step_down)
    .expand()
)
g3bar_as_g3 = Eq(gbar3, solve(_g3_g3bar, gbar3)[0])
assert g3bar_as_g3.subs(gbar3, g3)

dw_uv1bar = dw_uv1bar.subs([(gbar2, g2), (gbar3, g3)])

# Check the form of c0
c0_wp = Eq(c[0], wp(mu[1]-z0,g2,g3)/2)
c0_d = c0_wp.subs([wp_z0_mu_j_d.subs(j,1).args])

assert (
    (_c0_no_quad.rhs - c0_d.rhs)
    .subs([_.subs([__.args for __ in c_j_coeffs]).args for _ in d_j_coeffs])
    .factor()
    .subs(lam_step_down)
    .factor()
) == 0, 'c0 does not have the right form'


c0_wp
c0_d
w_ulbar_sub
dw_uv1bar
# g2bar
# g3bar

Eq(c[0], wp(-z0 + mu[1], g2, g3)/2)

Eq(c[0], d[2]/24 + d[3]*lamda[1]/8 + d[4]*lamda[1]**2/4 - (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(2*(4*gamma[1] - 4*lamda[1])))

Eq(ubar(z, mu[1])*vbar(z, mu[1]), -w(z)/2 + c[0])

Eq(Derivative(w(z), z)**2, -g2*w(z) - g3 + 4*w(z)**3)

We now turn our attention to gauge/phase terms. We derive the logarithmic derivatives of the three $\bar{u}$ modes and we will compare them against what we would expect for the target form of our desired Weierstrass elliptic function solutions. This should tell us if we are in the right gauge and if not how to get there:

In [36]:
_u_bars = [ubar(z,mu[_j+1]) for _j in range(3)]
_v_bars = [vbar(z,mu[_j+1]) for _j in range(3)]
_u_v_bars = _u_bars + _v_bars

# Build log derivatives
dlog_u_v_bars = [
    Eq(_eq.lhs/_f, sum(_/_f for _ in _eq.rhs.args))
    for _f, _eq in zip(_u_v_bars, d_u_v_bars)
]

# Swap wave mixing terms for log diffs of mode powers using hamiltonian
wv_mix_subs = [
    (
        (Hbar_uv_b.lhs/8 - _eq.rhs/_n/2).factor()/(ubar(z, mu[_j])*vbar(z, mu[_j])), 
        sum(_/(ubar(z, mu[_j])*vbar(z, mu[_j])) for _ in (Hbar_uv_b.rhs/8 - _eq.lhs/_n/2).args)
    )
    for _j, _n, _eq in zip([1, 2, 3], [4, 4, 8], duvj_bars_trnsd_ham)
]
wv_mix_subs_conj = [
    (
        (Hbar_uv_b.lhs/8 + _eq.rhs/_n/2).factor()/(ubar(z, mu[_j])*vbar(z, mu[_j])), 
        sum(_/(ubar(z, mu[_j])*vbar(z, mu[_j])) for _ in (Hbar_uv_b.rhs/8 + _eq.lhs/_n/2).args)
    )
    for _j, _n, _eq in zip([1, 2, 3], [4, 4, 8], duvj_bars_trnsd_ham)
]
dlog_u_v_bars_b = [_.subs(wv_mix_subs + wv_mix_subs_conj) for _ in dlog_u_v_bars]

# Convert to rho
dlog_u_v_bars_rhobar = [
    Eq(
        dlog_u_v_bars_b[0].lhs,
        sum(
            _.subs([_.args for _ in gamma_bar_js_gamma]).subs(gamma[2], -gamma[1]).factor()
            for _ in
            dlog_u_v_bars_b[0].rhs
            .subs(uv_bar_rho_gambar_subs)
            .doit()
            .subs(rhobar(z), h(z) + 4*gammabar[1]/3)
            .doit()
            .expand().collect([diff(h(z),z), h(z)], factor).args

        ).subs(h(z), rhobar(z) - 4*gammabar[1]/3).doit()
    ),
    Eq(
        dlog_u_v_bars_b[1].lhs,
        sum(
            _.subs([_.args for _ in gamma_bar_js_gamma]).subs(gamma[1], -gamma[2]).factor()
            for _ in
            dlog_u_v_bars_b[1].rhs
            .subs(uv_bar_rho_gambar_subs)
            .doit()
            .subs(rhobar(z), h(z) + 4*gammabar[2]/3)
            .doit()
            .expand().collect([diff(h(z),z), h(z)], factor).args

        ).subs(h(z), rhobar(z) - 4*gammabar[2]/3).doit()
    ),
    Eq(
        dlog_u_v_bars_b[2].lhs,
        sum(
            _.subs([_.args for _ in gamma_bar_js_gamma]).subs(gamma[1], -gamma[2]).factor()
            for _ in
            dlog_u_v_bars_b[2].rhs
            .subs(uv_bar_rho_gambar_subs)
            .doit()
            .subs(rhobar(z), h(z) + 2*gammabar[3]/3)
            .doit()
            .expand().collect([diff(h(z),z), h(z)], factor).args
        ).subs(h(z), rhobar(z) - 2*gammabar[3]/3).doit()
    ),
    Eq(
        dlog_u_v_bars_b[3].lhs,
        sum(
            _.subs([_.args for _ in gamma_bar_js_gamma]).subs(gamma[2], -gamma[1]).factor()
            for _ in
            dlog_u_v_bars_b[3].rhs
            .subs(uv_bar_rho_gambar_subs)
            .doit()
            .subs(rhobar(z), h(z) + 4*gammabar[1]/3)
            .doit()
            .expand().collect([diff(h(z),z), h(z)], factor).args

        ).subs(h(z), rhobar(z) - 4*gammabar[1]/3).doit()
    ),
    Eq(
        dlog_u_v_bars_b[4].lhs,
        sum(
            _.subs([_.args for _ in gamma_bar_js_gamma]).subs(gamma[1], -gamma[2]).factor()
            for _ in
            dlog_u_v_bars_b[4].rhs
            .subs(uv_bar_rho_gambar_subs)
            .doit()
            .subs(rhobar(z), h(z) + 4*gammabar[2]/3)
            .doit()
            .expand().collect([diff(h(z),z), h(z)], factor).args

        ).subs(h(z), rhobar(z) - 4*gammabar[2]/3).doit()
    ),
    Eq(
        dlog_u_v_bars_b[5].lhs,
        sum(
            _.subs([_.args for _ in gamma_bar_js_gamma]).subs(gamma[1], -gamma[2]).factor()
            for _ in
            dlog_u_v_bars_b[5].rhs
            .subs(uv_bar_rho_gambar_subs)
            .doit()
            .subs(rhobar(z), h(z) + 2*gammabar[3]/3)
            .doit()
            .expand().collect([diff(h(z),z), h(z)], factor).args
        ).subs(h(z), rhobar(z) - 2*gammabar[3]/3).doit()
    )
]
# Check we get the expected modal power diffs by combing a log diff and its conjugate
_expected_drhos = [
    diff(rhobar(z),z)/(rhobar(z) - 4*gammabar[1]/3),
    diff(rhobar(z),z)/(rhobar(z) - 4*gammabar[2]/3),
    diff(rhobar(z),z)/(rhobar(z) - 2*gammabar[3]/3)
]
for _j in range(3):
    assert (
        dlog_u_v_bars_rhobar[_j].rhs + dlog_u_v_bars_rhobar[_j + 3].rhs 
        - _expected_drhos[_j]
    ).simplify() == 0, 'expected rho diff not seen'


for _ in dlog_u_v_bars_b[0:3]:
    print()
    _
for _ in dlog_u_v_bars_rhobar:
    print()
    _

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), -(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])**2*vbar(z, mu[3])**2/(ubar(z, mu[1])*vbar(z, mu[1])*d[5]) + 6*(gamma[1] - lamda[1])**2*ubar(z, mu[1])*vbar(z, mu[1])*b[2]/d[5] - (gamma[1] - lamda[1])*ubar(z, mu[3])*vbar(z, mu[3])*b[1]/d[5] - 2*(gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])**2*b[2]/(ubar(z, mu[1])*vbar(z, mu[1])*d[5]) + (gamma[2] - lamda[1])*ubar(z, mu[2])*ubar(z, mu[3])*vbar(z, mu[2])*vbar(z, mu[3])*b[1]/(ubar(z, mu[1])*vbar(z, mu[1])*d[5]) + Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z)/(2*ubar(z, mu[1])*vbar(z, mu[1])))

Eq(Derivative(ubar(z, mu[2]), z)/ubar(z, mu[2]), -(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])**2*vbar(z, mu[3])**2/(ubar(z, mu[2])*vbar(z, mu[2])*d[5]) - 2*(gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])**2*b[2]/(ubar(z, mu[2])*vbar(z, mu[2])*d[5]) + (gamma[1] - lamda[1])*ubar(z, mu[1])*ubar(z, mu[3])*vbar(z, mu[1])*vbar(z, mu[3])*b[1]/(ubar(z, mu[2])*vbar(z, mu[2])*d[5]) + 6*(gamma[2] - lamda[1])**2*ubar(z, mu[2])*vbar(z, mu[2])*b[2]/d[5] - (gamma[2] - lamda[1])*ubar(z, mu[3])*vbar(z, mu[3])*b[1]/d[5] + Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z)/(2*ubar(z, mu[2])*vbar(z, mu[2])))

Eq(Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]), 2*(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0])*ubar(z, mu[3])*vbar(z, mu[3])/d[5] - 4*(gamma[1] - lamda[1])**2*ubar(z, mu[1])**2*vbar(z, mu[1])**2*b[2]/(ubar(z, mu[3])*vbar(z, mu[3])*d[5]) - 4*(gamma[2] - lamda[1])**2*ubar(z, mu[2])**2*vbar(z, mu[2])**2*b[2]/(ubar(z, mu[3])*vbar(z, mu[3])*d[5]) + Derivative(ubar(z, mu[3])*vbar(z, mu[3]), z)/(2*ubar(z, mu[3])*vbar(z, mu[3])))

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), 3*(rhobar(z) - 4*gammabar[1]/3)*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)/d[5] + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) + Derivative(rhobar(z), z)/(2*(rhobar(z) - 4*gammabar[1]/3)) + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(12*(rhobar(z) - 4*gammabar[1]/3)*(gamma[1] - lamda[1])**2))

Eq(Derivative(ubar(z, mu[2]), z)/ubar(z, mu[2]), 3*(rhobar(z) - 4*gammabar[2]/3)*(b[0] + b[1]*gamma[2] - 2*b[2]*gamma[2]**2 + 4*b[2]*gamma[2]*lamda[1] - b[2]*lamda[1]**2)/d[5] + (b[0] + b[1]*gamma[2] + b[2]*gamma[2]*lamda[1])/(gamma[2] - lamda[1]) + Derivative(rhobar(z), z)/(2*(rhobar(z) - 4*gammabar[2]/3)) + (b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)*d[5]/(12*(rhobar(z) - 4*gammabar[2]/3)*(gamma[2] - lamda[1])**2))

Eq(Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]), -3*(rhobar(z) - 2*gammabar[3]/3)*(b[0] - 2*b[2]*gamma[2]**2 - b[2]*lamda[1]**2)/d[5] + b[2]*lamda[1] + Derivative(rhobar(z), z)/(2*(rhobar(z) - 2*gammabar[3]/3)) + b[2]*d[5]/(12*(rhobar(z) - 2*gammabar[3]/3)))

Eq(Derivative(vbar(z, mu[1]), z)/vbar(z, mu[1]), -3*(rhobar(z) - 4*gammabar[1]/3)*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)/d[5] - (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) + Derivative(rhobar(z), z)/(2*(rhobar(z) - 4*gammabar[1]/3)) - (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(12*(rhobar(z) - 4*gammabar[1]/3)*(gamma[1] - lamda[1])**2))

Eq(Derivative(vbar(z, mu[2]), z)/vbar(z, mu[2]), -3*(rhobar(z) - 4*gammabar[2]/3)*(b[0] + b[1]*gamma[2] - 2*b[2]*gamma[2]**2 + 4*b[2]*gamma[2]*lamda[1] - b[2]*lamda[1]**2)/d[5] - (b[0] + b[1]*gamma[2] + b[2]*gamma[2]*lamda[1])/(gamma[2] - lamda[1]) + Derivative(rhobar(z), z)/(2*(rhobar(z) - 4*gammabar[2]/3)) - (b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)*d[5]/(12*(rhobar(z) - 4*gammabar[2]/3)*(gamma[2] - lamda[1])**2))

Eq(Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3]), 3*(rhobar(z) - 2*gammabar[3]/3)*(b[0] - 2*b[2]*gamma[2]**2 - b[2]*lamda[1]**2)/d[5] - b[2]*lamda[1] + Derivative(rhobar(z), z)/(2*(rhobar(z) - 2*gammabar[3]/3)) - b[2]*d[5]/(12*(rhobar(z) - 2*gammabar[3]/3)))

In [37]:
rhobar_w_c0 = Eq(rhobar(z), solve(w_ulbar_sub.subs(uv_bar_rho_gambar_subs), rhobar(z))[0])

dlogu1_w = Eq(
    dlog_u_v_bars_rhobar[0].lhs,
    sum(
        _
        .subs([rhobar_w_c0.args])
        .doit()
        .factor()
        for _ in dlog_u_v_bars_rhobar[0].rhs.args
    )
)
dlogu1_wp_wpp = Eq(
    dlogu1_w.lhs,
    dlogu1_w.rhs
    .subs([
        (w(z), wp(z-z0,g2,g3)),
        (diff(wp(z-z0,g2,g3),z), wpp(z-z0,g2,g3)),
        c0_wp.args,
        (-4*wpp_z0_mu_j_d.rhs.subs(d5_dj.rhs, d5_dj.lhs).factor().subs(j,1), -4*wpp_z0_mu_j_d.lhs.subs(j,1))
    ])
)

rhobar_w_c0
print()
dlogu1_w
print()
dlogu1_wp_wpp

Eq(rhobar(z), 2*w(z)/3 - 4*c[0]/3 + 4*gammabar[1]/3)

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), -2*(-w(z) + 2*c[0])*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)/d[5] + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) - Derivative(w(z), z)/(2*(-w(z) + 2*c[0])) - (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(8*(-w(z) + 2*c[0])*(gamma[1] - lamda[1])**2))

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), -2*(-wp(z - z0, g2, g3) + wp(-z0 + mu[1], g2, g3))*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)/d[5] + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) + \wp'(-z0 + mu[1], g2, g3)/(2*(-wp(z - z0, g2, g3) + wp(-z0 + mu[1], g2, g3))) - Derivative(wp(z - z0, g2, g3), z)/(-2*wp(z - z0, g2, g3) + 2*wp(-z0 + mu[1], g2, g3)))

We thus have a plan. We will write each logarithmic derivative in terms of its associated modal power and the derivative of its modal power then gauge transform away the terms linear in the modal power as they cause unwanted essential singularities which we know from the form of our taregt solutions should not be there.

In [38]:
rev_uv_bar_rho_gambar_subs = [
    (rhobar(z),solve(Eq(*_), rhobar(z))[0]) 
    for _ in uv_bar_rho_gambar_subs
]
drev_uv_bar_rho_gambar_subs = [
    (diff(rhobar(z),z), -4*Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z)/3),
    (diff(rhobar(z),z), -4*Derivative(ubar(z, mu[2])*vbar(z, mu[2]),z)/3),
    (diff(rhobar(z),z), -2*Derivative(ubar(z, mu[3])*vbar(z, mu[3]),z)/3)
]


dlog_uvj_bars = [
    _
    .subs(*drev_uv_bar_rho_gambar_subs[_j % 3])
    .subs(*rev_uv_bar_rho_gambar_subs[_j % 3])
    for _j, _ in enumerate(dlog_u_v_bars_rhobar)
]

for _ in dlog_uvj_bars:
    _

Eq(Derivative(ubar(z, mu[1]), z)/ubar(z, mu[1]), -4*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)*ubar(z, mu[1])*vbar(z, mu[1])/d[5] + Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z)/(2*ubar(z, mu[1])*vbar(z, mu[1])) + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) - (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(16*(gamma[1] - lamda[1])**2*ubar(z, mu[1])*vbar(z, mu[1])))

Eq(Derivative(ubar(z, mu[2]), z)/ubar(z, mu[2]), -4*(b[0] + b[1]*gamma[2] - 2*b[2]*gamma[2]**2 + 4*b[2]*gamma[2]*lamda[1] - b[2]*lamda[1]**2)*ubar(z, mu[2])*vbar(z, mu[2])/d[5] + Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z)/(2*ubar(z, mu[2])*vbar(z, mu[2])) + (b[0] + b[1]*gamma[2] + b[2]*gamma[2]*lamda[1])/(gamma[2] - lamda[1]) - (b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)*d[5]/(16*(gamma[2] - lamda[1])**2*ubar(z, mu[2])*vbar(z, mu[2])))

Eq(Derivative(ubar(z, mu[3]), z)/ubar(z, mu[3]), 2*(b[0] - 2*b[2]*gamma[2]**2 - b[2]*lamda[1]**2)*ubar(z, mu[3])*vbar(z, mu[3])/d[5] + b[2]*lamda[1] + Derivative(ubar(z, mu[3])*vbar(z, mu[3]), z)/(2*ubar(z, mu[3])*vbar(z, mu[3])) - b[2]*d[5]/(8*ubar(z, mu[3])*vbar(z, mu[3])))

Eq(Derivative(vbar(z, mu[1]), z)/vbar(z, mu[1]), 4*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)*ubar(z, mu[1])*vbar(z, mu[1])/d[5] + Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z)/(2*ubar(z, mu[1])*vbar(z, mu[1])) - (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(16*(gamma[1] - lamda[1])**2*ubar(z, mu[1])*vbar(z, mu[1])))

Eq(Derivative(vbar(z, mu[2]), z)/vbar(z, mu[2]), 4*(b[0] + b[1]*gamma[2] - 2*b[2]*gamma[2]**2 + 4*b[2]*gamma[2]*lamda[1] - b[2]*lamda[1]**2)*ubar(z, mu[2])*vbar(z, mu[2])/d[5] + Derivative(ubar(z, mu[2])*vbar(z, mu[2]), z)/(2*ubar(z, mu[2])*vbar(z, mu[2])) - (b[0] + b[1]*gamma[2] + b[2]*gamma[2]*lamda[1])/(gamma[2] - lamda[1]) + (b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)*d[5]/(16*(gamma[2] - lamda[1])**2*ubar(z, mu[2])*vbar(z, mu[2])))

Eq(Derivative(vbar(z, mu[3]), z)/vbar(z, mu[3]), -2*(b[0] - 2*b[2]*gamma[2]**2 - b[2]*lamda[1]**2)*ubar(z, mu[3])*vbar(z, mu[3])/d[5] - b[2]*lamda[1] + Derivative(ubar(z, mu[3])*vbar(z, mu[3]), z)/(2*ubar(z, mu[3])*vbar(z, mu[3])) + b[2]*d[5]/(8*ubar(z, mu[3])*vbar(z, mu[3])))

In [39]:
u_v_bar_to_tilde = [
    Eq(ubar(z, mu[1]), utilde(z, mu[1])*exp(phi(z,1))),
    Eq(ubar(z, mu[2]), utilde(z, mu[2])*exp(phi(z,2))),
    Eq(ubar(z, mu[3]), utilde(z, mu[3])*exp(-(phi(z,1) + phi(z,2))/2)),
    Eq(vbar(z, mu[1]), vtilde(z, mu[1])*exp(-phi(z,1))),
    Eq(vbar(z, mu[2]), vtilde(z, mu[2])*exp(-phi(z,2))),
    Eq(vbar(z, mu[3]), vtilde(z, mu[3])*exp((phi(z,1) + phi(z,2))/2))
]
u_v_bar_to_tilde_subs = [_.args for _ in u_v_bar_to_tilde]

Htilde_uv = Eq(
    Hbar_uv_b.lhs.subs(u_v_bar_to_tilde_subs).simplify(),
    sum(
        _.subs(u_v_bar_to_tilde_subs).simplify() 
        for _ in Hbar_uv_b.rhs.args
    )
)

for _ in u_v_bar_to_tilde:
    _
print()    
Htilde_uv

Eq(ubar(z, mu[1]), utilde(z, mu[1])*exp(phi(z, 1)))

Eq(ubar(z, mu[2]), utilde(z, mu[2])*exp(phi(z, 2)))

Eq(ubar(z, mu[3]), utilde(z, mu[3])*exp(-phi(z, 1)/2 - phi(z, 2)/2))

Eq(vbar(z, mu[1]), vtilde(z, mu[1])*exp(-phi(z, 1)))

Eq(vbar(z, mu[2]), vtilde(z, mu[2])*exp(-phi(z, 2)))

Eq(vbar(z, mu[3]), vtilde(z, mu[3])*exp(phi(z, 1)/2 + phi(z, 2)/2))

Eq(4*(utilde(z, mu[1])*utilde(z, mu[2])*utilde(z, mu[3])**2 + vtilde(z, mu[1])*vtilde(z, mu[2])*vtilde(z, mu[3])**2)*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5], (-(gamma[1] - gamma[2])**2*b[2] + 4*b[0])*utilde(z, mu[3])**2*vtilde(z, mu[3])**2/(2*d[5]) + 2*(-gamma[1] + lamda[1])*utilde(z, mu[1])*utilde(z, mu[3])*vtilde(z, mu[1])*vtilde(z, mu[3])*b[1]/d[5] + 4*(gamma[1] - lamda[1])**2*utilde(z, mu[1])**2*vtilde(z, mu[1])**2*b[2]/d[5] + 2*(-gamma[2] + lamda[1])*utilde(z, mu[2])*utilde(z, mu[3])*vtilde(z, mu[2])*vtilde(z, mu[3])*b[1]/d[5] + 4*(gamma[2] - lamda[1])**2*utilde(z, mu[2])**2*vtilde(z, mu[2])**2*b[2]/d[5])

In [40]:
# Substitute tilde modes in for bar modes in log diff eqs

dlog_uvj_tildes = [
    Eq(
        _.lhs.subs(u_v_bar_to_tilde_subs).doit().simplify(),
        _.rhs.subs(u_v_bar_to_tilde_subs)
        .subs([
            (
                exp(- phi(z,1)/2 - phi(z,2)/2)*exp(phi(z,1)/2 + phi(z,2)/2),
                (exp(- phi(z,1)/2 - phi(z,2)/2)*exp(phi(z,1)/2 + phi(z,2)/2)).simplify()
            )
        ])
    )
    for _ in dlog_uvj_bars
]
_no_diff_subs = (
    [(diff(utilde(z, mu[_j + 1]),z),0) for _j in range(3)] + 
    [(diff(vtilde(z, mu[_j + 1]),z),0) for _j in range(3)]
)
dlog_uvj_tildes = [
    Eq(_.lhs - _.lhs.subs(_no_diff_subs), _.rhs - _.lhs.subs(_no_diff_subs))
    for _ in dlog_uvj_tildes
]

# Define gauge transform to kill terms linear in modal powers

dphi_u_v_tildes = [
    Eq(
        diff(phi(z, 1),z)/2 + diff(phi(z, 2),z)/2,
        -dlog_uvj_tildes[2].rhs.coeff(utilde(z, mu[3])*vtilde(z, mu[3]), 1)*utilde(z, mu[3])*vtilde(z, mu[3])
        + C
    ),
    Eq(
        diff(phi(z, 1),z),
        dlog_uvj_tildes[0].rhs.coeff(utilde(z, mu[1])*vtilde(z, mu[1]), 1)*utilde(z, mu[1])*vtilde(z, mu[1])
    ),
    Eq(
        diff(phi(z, 2),z),
        dlog_uvj_tildes[1].rhs.coeff(utilde(z, mu[2])*vtilde(z, mu[2]), 1)*utilde(z, mu[2])*vtilde(z, mu[2])
    )
]

uv_tilde_rho_gambar_subs = [
    Eq(_[0].subs(u_v_bar_to_tilde_subs).simplify(), _[1]).args 
    for _ in uv_bar_rho_gambar_subs
]

residual_C = Eq(
    0, 
    (dphi_u_v_tildes[0].lhs - dphi_u_v_tildes[0].rhs)
    .subs([dphi_u_v_tildes[1].args, dphi_u_v_tildes[2].args])
    .subs(uv_tilde_rho_gambar_subs)
    .subs(gamma_bar_js_gamma_subs)
    .subs([d5_dj.args])
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs(gamma[2], -gamma[1])
    .expand().factor()
)

assert residual_C.rhs.coeff(rhobar(z), 1) == 0, 'rhobar term not cancels in mode 3 power'

residual_C = Eq(C, solve(residual_C, C)[0].collect(b[2], factor))
dphi_u_v_tildes[0] = dphi_u_v_tildes[0].subs([residual_C.args])
dphi_u_v_tildes_subs = [_.args for _ in dphi_u_v_tildes]

_b_simple = Eq(
    b[2]*lamda[1] + 
    (-(2*gamma[1]**2 - lamda[1]**2)*b[2]*lamda[1] - b[0]*lamda[1] - b[1]*gamma[1]**2).collect(b[2], factor)/
    (2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])).factor(),
    (-b[0]*lamda[1] - b[1]*gamma[1]**2 - b[2]*lamda[1]**3)/(2*(gamma[1]**2 - lamda[1]**2))
)
assert _b_simple.simplify()

dlog_uvj_tildes = [_.subs(dphi_u_v_tildes_subs).subs([_b_simple.args]) for _ in dlog_uvj_tildes]

for _ in dphi_u_v_tildes:
    _
print()
for _ in dlog_uvj_tildes:
    _

Eq(Derivative(phi(z, 1), z)/2 + Derivative(phi(z, 2), z)/2, -2*(b[0] - 2*b[2]*gamma[2]**2 - b[2]*lamda[1]**2)*utilde(z, mu[3])*vtilde(z, mu[3])/d[5] + (-(2*gamma[1]**2 - lamda[1]**2)*b[2]*lamda[1] - b[0]*lamda[1] - b[1]*gamma[1]**2)/(2*(gamma[1] - lamda[1])*(gamma[1] + lamda[1])))

Eq(Derivative(phi(z, 1), z), -4*(b[0] + b[1]*gamma[1] - 2*b[2]*gamma[1]**2 + 4*b[2]*gamma[1]*lamda[1] - b[2]*lamda[1]**2)*utilde(z, mu[1])*vtilde(z, mu[1])/d[5])

Eq(Derivative(phi(z, 2), z), -4*(b[0] + b[1]*gamma[2] - 2*b[2]*gamma[2]**2 + 4*b[2]*gamma[2]*lamda[1] - b[2]*lamda[1]**2)*utilde(z, mu[2])*vtilde(z, mu[2])/d[5])

Eq(Derivative(utilde(z, mu[1]), z)/utilde(z, mu[1]), Derivative(utilde(z, mu[1])*vtilde(z, mu[1]), z)/(2*utilde(z, mu[1])*vtilde(z, mu[1])) + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) - (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(16*(gamma[1] - lamda[1])**2*utilde(z, mu[1])*vtilde(z, mu[1])))

Eq(Derivative(utilde(z, mu[2]), z)/utilde(z, mu[2]), Derivative(utilde(z, mu[2])*vtilde(z, mu[2]), z)/(2*utilde(z, mu[2])*vtilde(z, mu[2])) + (b[0] + b[1]*gamma[2] + b[2]*gamma[2]*lamda[1])/(gamma[2] - lamda[1]) - (b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)*d[5]/(16*(gamma[2] - lamda[1])**2*utilde(z, mu[2])*vtilde(z, mu[2])))

Eq(Derivative(utilde(z, mu[3]), z)/utilde(z, mu[3]), Derivative(utilde(z, mu[3])*vtilde(z, mu[3]), z)/(2*utilde(z, mu[3])*vtilde(z, mu[3])) - b[2]*d[5]/(8*utilde(z, mu[3])*vtilde(z, mu[3])) + (-b[0]*lamda[1] - b[1]*gamma[1]**2 - b[2]*lamda[1]**3)/(2*gamma[1]**2 - 2*lamda[1]**2))

Eq(Derivative(vtilde(z, mu[1]), z)/vtilde(z, mu[1]), Derivative(utilde(z, mu[1])*vtilde(z, mu[1]), z)/(2*utilde(z, mu[1])*vtilde(z, mu[1])) + (-b[0] - b[1]*gamma[1] - b[2]*gamma[1]*lamda[1])/(gamma[1] - lamda[1]) + (b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)*d[5]/(16*(gamma[1] - lamda[1])**2*utilde(z, mu[1])*vtilde(z, mu[1])))

Eq(Derivative(vtilde(z, mu[2]), z)/vtilde(z, mu[2]), Derivative(utilde(z, mu[2])*vtilde(z, mu[2]), z)/(2*utilde(z, mu[2])*vtilde(z, mu[2])) + (-b[0] - b[1]*gamma[2] - b[2]*gamma[2]*lamda[1])/(gamma[2] - lamda[1]) + (b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)*d[5]/(16*(gamma[2] - lamda[1])**2*utilde(z, mu[2])*vtilde(z, mu[2])))

Eq(Derivative(vtilde(z, mu[3]), z)/vtilde(z, mu[3]), Derivative(utilde(z, mu[3])*vtilde(z, mu[3]), z)/(2*utilde(z, mu[3])*vtilde(z, mu[3])) + b[2]*d[5]/(8*utilde(z, mu[3])*vtilde(z, mu[3])) - (-b[0]*lamda[1] - b[1]*gamma[1]**2 - b[2]*lamda[1]**3)/(2*gamma[1]**2 - 2*lamda[1]**2))

The gauge transform has removed the unwanted cross-phase modulation terms and now we see what the corresponding equations of motion are for the tilde coordinates:

In [41]:
_uv3_to_uv1_tilde = Eq(
    utilde(z, mu[3])*vtilde(z, mu[3]), 
    solve(uv_bar_sig_dj_ids[1], ubar(z, mu[3])*vbar(z, mu[3]))[0].subs(u_v_bar_to_tilde_subs)
)
_uv3_to_uv2_tilde = Eq(
    utilde(z, mu[3])*vtilde(z, mu[3]), 
    solve(uv_bar_sig_dj_ids[2], ubar(z, mu[3])*vbar(z, mu[3]))[0].subs(u_v_bar_to_tilde_subs)
)
_uv1_to_uv3_tilde = Eq(
    utilde(z, mu[1])*vtilde(z, mu[1]), 
    solve(uv_bar_sig_dj_ids[1], ubar(z, mu[1])*vbar(z, mu[1]))[0].subs(u_v_bar_to_tilde_subs)
)
_uv2_to_uv3_tilde = Eq(
    utilde(z, mu[2])*vtilde(z, mu[2]), 
    solve(uv_bar_sig_dj_ids[2], ubar(z, mu[2])*vbar(z, mu[2]))[0].subs(u_v_bar_to_tilde_subs)
)
_u_v_tilde_diffs = [
    diff(utilde(z, mu[1]),z), diff(utilde(z, mu[2]),z), diff(utilde(z, mu[3]),z),
    diff(vtilde(z, mu[1]),z), diff(vtilde(z, mu[2]),z), diff(vtilde(z, mu[3]),z)
]
_d_u_v_tildes = [_.subs(u_v_bar_to_tilde_subs).doit().simplify() for _ in d_u_v_bars]
d_u_v_tildes = [Eq(_f, solve(_eq, _f)[0]) for _f, _eq in zip(_u_v_tilde_diffs, _d_u_v_tildes)]

d_u_v_tildes = [
    Eq(
        d_u_v_tildes[0].lhs, 
        d_u_v_tildes[0].rhs
        .subs([dphi_u_v_tildes[1].args])
        .subs([_uv3_to_uv1_tilde.args])
        .expand().collect([utilde(z, mu[1])*vtilde(z, mu[1]), utilde(z, mu[1])], factor)
    ),
    Eq(
        d_u_v_tildes[1].lhs, 
        d_u_v_tildes[1].rhs
        .subs([dphi_u_v_tildes[2].args])
        .subs([_uv3_to_uv2_tilde.args])
        .expand().collect([utilde(z, mu[2])*vtilde(z, mu[2]), utilde(z, mu[2])], factor)
    ),
    Eq(
        d_u_v_tildes[2].lhs, 
        d_u_v_tildes[2].rhs
        .subs([(dphi_u_v_tildes[0].lhs*2, dphi_u_v_tildes[0].rhs*2)])
        .subs([
            _uv1_to_uv3_tilde.args,
            _uv2_to_uv3_tilde.args
        ])
        .subs(gamma[2], -gamma[1])
        .expand().collect([utilde(z, mu[3])*vtilde(z, mu[3]), utilde(z, mu[3])], factor)
    ),
    Eq(
        d_u_v_tildes[3].lhs, 
        d_u_v_tildes[3].rhs
        .subs([dphi_u_v_tildes[1].args])
        .subs([_uv3_to_uv1_tilde.args])
        .expand().collect([utilde(z, mu[1])*vtilde(z, mu[1]), vtilde(z, mu[1])], factor)
    ),
    Eq(
        d_u_v_tildes[4].lhs, 
        d_u_v_tildes[4].rhs
        .subs([dphi_u_v_tildes[2].args])
        .subs([_uv3_to_uv2_tilde.args])
        .expand().collect([utilde(z, mu[2])*vtilde(z, mu[2]), vtilde(z, mu[2])], factor)
    ),
    Eq(
        d_u_v_tildes[5].lhs, 
        d_u_v_tildes[5].rhs
        .subs([(dphi_u_v_tildes[0].lhs*2, dphi_u_v_tildes[0].rhs*2)])
        .subs([
            _uv1_to_uv3_tilde.args,
            _uv2_to_uv3_tilde.args
        ])
        .subs(gamma[2], -gamma[1])
        .expand().collect([utilde(z, mu[3])*vtilde(z, mu[3]), vtilde(z, mu[3])], factor)
    )
]

_mode3_const_sub = Eq(
    -d_u_v_tildes[2].rhs.coeff(utilde(z, mu[3])),
    -d_u_v_tildes[2].rhs.coeff(utilde(z, mu[3]))
    .expand().collect([b[0], b[1], b[2]], simplify)
    .subs(gamma[1]**2, x + lamda[1]**2)
    .expand().collect(x, factor)
    .subs(x, gamma[1]**2 - lamda[1]**2)
)
d_u_v_tildes = [_.subs([_mode3_const_sub.args]) for _ in d_u_v_tildes]


for _ in d_u_v_tildes:
    _

Eq(Derivative(utilde(z, mu[1]), z), -4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*vtilde(z, mu[2])*vtilde(z, mu[3])**2/d[5] + 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*utilde(z, mu[1])**2*vtilde(z, mu[1])/d[5] + utilde(z, mu[1])*b[1]/2)

Eq(Derivative(utilde(z, mu[2]), z), -4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*vtilde(z, mu[1])*vtilde(z, mu[3])**2/d[5] + 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*utilde(z, mu[2])**2*vtilde(z, mu[2])/d[5] + utilde(z, mu[2])*b[1]/2)

Eq(Derivative(utilde(z, mu[3]), z), -8*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*vtilde(z, mu[1])*vtilde(z, mu[2])*vtilde(z, mu[3])/d[5] + 2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*utilde(z, mu[3])**2*vtilde(z, mu[3])/d[5] - (b[1] + b[2]*lamda[1] + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(2*(gamma[1]**2 - lamda[1]**2)))*utilde(z, mu[3]))

Eq(Derivative(vtilde(z, mu[1]), z), 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*utilde(z, mu[2])*utilde(z, mu[3])**2/d[5] - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*utilde(z, mu[1])*vtilde(z, mu[1])**2/d[5] - vtilde(z, mu[1])*b[1]/2)

Eq(Derivative(vtilde(z, mu[2]), z), 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*utilde(z, mu[1])*utilde(z, mu[3])**2/d[5] - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*utilde(z, mu[2])*vtilde(z, mu[2])**2/d[5] - vtilde(z, mu[2])*b[1]/2)

Eq(Derivative(vtilde(z, mu[3]), z), 8*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*utilde(z, mu[1])*utilde(z, mu[2])*utilde(z, mu[3])/d[5] - 2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*utilde(z, mu[3])*vtilde(z, mu[3])**2/d[5] + (b[1] + b[2]*lamda[1] + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(2*(gamma[1]**2 - lamda[1]**2)))*vtilde(z, mu[3]))

We perform one more linear phase shift to even up propagation constants:

In [42]:
# Define U V modes

prop_const_shift = -(d_u_v_tildes[0].rhs.coeff(utilde(z, mu[1])) - d_u_v_tildes[2].rhs.coeff(utilde(z, mu[3])))/2

new_prop_const_1 = d_u_v_tildes[0].rhs.coeff(utilde(z, mu[1])) + prop_const_shift
new_prop_const_2 = d_u_v_tildes[1].rhs.coeff(utilde(z, mu[2])) + prop_const_shift
new_prop_const_3 = d_u_v_tildes[2].rhs.coeff(utilde(z, mu[3])) - prop_const_shift

assert new_prop_const_1 == new_prop_const_2
assert new_prop_const_1 == new_prop_const_3

U_V_from_tilde = [
    Eq(utilde(z, mu[1]), U(z, mu[1])*exp(-prop_const_shift*z)),
    Eq(utilde(z, mu[2]), U(z, mu[2])*exp(-prop_const_shift*z)),
    Eq(utilde(z, mu[3]), U(z, mu[3])*exp(prop_const_shift*z)),
    Eq(vtilde(z, mu[1]), V(z, mu[1])*exp(prop_const_shift*z)),
    Eq(vtilde(z, mu[2]), V(z, mu[2])*exp(prop_const_shift*z)),
    Eq(vtilde(z, mu[3]), V(z, mu[3])*exp(-prop_const_shift*z))
]
U_V_from_tilde_subs = [_.args for _ in U_V_from_tilde]

# Implement transfrom to U V

_U_V_diffs = [
    diff(U(z, mu[1]),z), diff(U(z, mu[2]),z), diff(U(z, mu[3]),z),
    diff(V(z, mu[1]),z), diff(V(z, mu[2]),z), diff(V(z, mu[3]),z)
]
_U_V_s = [U(z, mu[1]), U(z, mu[2]), U(z, mu[3]), V(z, mu[1]), V(z, mu[2]), V(z, mu[3])]


d_U_V_s = [
    Eq(
        _U_V_diffs[_j], 
        solve(d_u_v_tildes[_j].subs(U_V_from_tilde_subs).doit(), _U_V_diffs[_j])[0]
        .expand().collect([U(z, mu[(_j % 3) + 1])*V(z,mu[(_j % 3) + 1]), _U_V_s[_j]], factor)
    )
    for _j in range(6)
]

b_U_V_sub_id = Eq(
    -d_U_V_s[0].rhs.coeff(_U_V_s[0],1),
    -d_U_V_s[0].rhs.coeff(_U_V_s[0],1)
    .expand().simplify()
    .subs(gamma[1]**2, x+lamda[1]**2)
    .expand().collect(x, factor)
    .subs(x, gamma[1]**2 - lamda[1]**2)
)
d_U_V_s = [_.subs([b_U_V_sub_id.args]) for _ in d_U_V_s]


for _ in U_V_from_tilde:
    _

for _j in range(1, 3):
    assert d_U_V_s[_j].rhs.coeff(_U_V_s[_j],1) == d_U_V_s[0].rhs.coeff(_U_V_s[0],1)
for _j in range(3):
    assert d_U_V_s[_j + 3].rhs.coeff(_U_V_s[_j + 3], 1) == -d_U_V_s[0].rhs.coeff(_U_V_s[0],1)

for _ in d_U_V_s:
    _

Eq(utilde(z, mu[1]), U(z, mu[1])*exp(z*(3*b[1]/4 + b[2]*lamda[1]/2 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))))

Eq(utilde(z, mu[2]), U(z, mu[2])*exp(z*(3*b[1]/4 + b[2]*lamda[1]/2 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))))

Eq(utilde(z, mu[3]), U(z, mu[3])*exp(z*(-3*b[1]/4 - b[2]*lamda[1]/2 - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))))

Eq(vtilde(z, mu[1]), V(z, mu[1])*exp(z*(-3*b[1]/4 - b[2]*lamda[1]/2 - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))))

Eq(vtilde(z, mu[2]), V(z, mu[2])*exp(z*(-3*b[1]/4 - b[2]*lamda[1]/2 - (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))))

...

Eq(Derivative(U(z, mu[2]), z), -((b[1] + 2*b[2]*lamda[1])/4 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))*U(z, mu[2]) - 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*V(z, mu[1])*V(z, mu[3])**2/d[5] + 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*U(z, mu[2])**2*V(z, mu[2])/d[5])

Eq(Derivative(U(z, mu[3]), z), -((b[1] + 2*b[2]*lamda[1])/4 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))*U(z, mu[3]) - 8*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*V(z, mu[1])*V(z, mu[2])*V(z, mu[3])/d[5] + 2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*U(z, mu[3])**2*V(z, mu[3])/d[5])

Eq(Derivative(V(z, mu[1]), z), ((b[1] + 2*b[2]*lamda[1])/4 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))*V(z, mu[1]) + 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*U(z, mu[2])*U(z, mu[3])**2/d[5] - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*U(z, mu[1])*V(z, mu[1])**2/d[5])

Eq(Derivative(V(z, mu[2]), z), ((b[1] + 2*b[2]*lamda[1])/4 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))*V(z, mu[2]) + 4*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*U(z, mu[1])*U(z, mu[3])**2/d[5] - 4*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*U(z, mu[2])*V(z, mu[2])**2/d[5])

Eq(Derivative(V(z, mu[3]), z), ((b[1] + 2*b[2]*lamda[1])/4 + (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*lamda[1]/(4*(gamma[1]**2 - lamda[1]**2)))*V(z, mu[3]) + 8*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])*U(z, mu[1])*U(z, mu[2])*U(z, mu[3])/d[5] - 2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)*U(z, mu[3])*V(z, mu[3])**2/d[5])

We now take inspiration from our FWM studies and define similar constants:

In [43]:
gamma_j_prod_b_poly = Eq(
    Product(gamma[j] -  lamda[1], (j,1,2)), 
    (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4
)
C_gam_prod = Eq(Product(sqrt(gamma[j] - lamda[1]), (j,1,2)), C)
C_gam_prod_sqrd = Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)
C_gam_prod_sign = Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m * C)
chi_d5_C = Eq(chi, 16*(-1)**m*C/d[5])
C_d5_chi = Eq(C, solve(chi_d5_C,C)[0])

b_C_one_over_chi = Eq(
    (
        d5_dj.rhs
        .subs([_.args for _ in d_j_coeffs])
        .subs([_.args for _ in c_j_coeffs])
        .subs(b[0], solve(C_gam_prod_sign, b[0])[0])
        /(16*(-1)**m*C)
    )
    .expand(),
    (d5_dj.lhs/(16*(-1)**m*C)).subs([C_d5_chi.args])
)

d5_C_b_lam = Eq(
    d5_dj.lhs,
    d5_dj.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([(b[0], solve(C_gam_prod_sign, b[0])[0])])
    .expand()
    .collect(C, factor)
)

gamma_j_prod_b_poly
C_gam_prod
C_gam_prod_sqrd
C_gam_prod_sign
chi_d5_C
C_d5_chi
b_C_one_over_chi
d5_C_b_lam

Eq(Product(gamma[j] - lamda[1], (j, 1, 2)), (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)

Eq(Product(sqrt(gamma[j] - lamda[1]), (j, 1, 2)), C)

Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)

Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m*C)

Eq(chi, 16*(-1)**m*C/d[5])

Eq(C, chi*d[5]/(16*(-1)**m))

Eq(b[1]/4 + b[2]*lamda[1]/2 - lamda[1]/(2*(-1)**m*C), 1/chi)

Eq(d[5], 4*(-1)**m*C*(b[1] + 2*b[2]*lamda[1]) - 8*lamda[1])

And we then see that we have a simplified single parameter degenerate FWM system:

In [44]:
to_chi_subs = [
    gamma_lamda_bj_gam1.args,
    (sqrt(-gamma[1] - lamda[1]), sqrt(-gamma[1] - lamda[1]).subs(gamma[1], - gamma[2])),
    C_gam_prod_sign.args,
    C_gam_prod.doit().args,
    b_C_one_over_chi.args,
    C_d5_chi.args,
    ((-1)**(-m), (-1)**m)
]

d_U_V_chi_s = [_.subs(to_chi_subs) for _ in d_U_V_s]

for _ in d_U_V_chi_s:
    _

Eq(Derivative(U(z, mu[1]), z), -(-1)**m*chi*V(z, mu[2])*V(z, mu[3])**2/4 + chi*U(z, mu[1])**2*V(z, mu[1])/2 - U(z, mu[1])/chi)

Eq(Derivative(U(z, mu[2]), z), -(-1)**m*chi*V(z, mu[1])*V(z, mu[3])**2/4 + chi*U(z, mu[2])**2*V(z, mu[2])/2 - U(z, mu[2])/chi)

Eq(Derivative(U(z, mu[3]), z), -(-1)**m*chi*V(z, mu[1])*V(z, mu[2])*V(z, mu[3])/2 + chi*U(z, mu[3])**2*V(z, mu[3])/4 - U(z, mu[3])/chi)

Eq(Derivative(V(z, mu[1]), z), (-1)**m*chi*U(z, mu[2])*U(z, mu[3])**2/4 - chi*U(z, mu[1])*V(z, mu[1])**2/2 + V(z, mu[1])/chi)

Eq(Derivative(V(z, mu[2]), z), (-1)**m*chi*U(z, mu[1])*U(z, mu[3])**2/4 - chi*U(z, mu[2])*V(z, mu[2])**2/2 + V(z, mu[2])/chi)

Eq(Derivative(V(z, mu[3]), z), (-1)**m*chi*U(z, mu[1])*U(z, mu[2])*U(z, mu[3])/2 - chi*U(z, mu[3])*V(z, mu[3])**2/4 + V(z, mu[3])/chi)

We then calculate the Hamiltonian for this system:

In [45]:
# Constant from transforming the old Hamiltonian

H_UV = Eq(
    Htilde_uv.lhs.subs(U_V_from_tilde_subs).simplify(),
    sum(_.subs(U_V_from_tilde_subs).simplify() for _ in Htilde_uv.rhs.args)
)

# Canonical Hamiltonian for U, V by inspecting the diff eqns
H_UV_chi = Eq(x, 
   -(-1)**m*chi/4*(U(z, mu[1])*U(z, mu[2])*U(z, mu[3])**2 + V(z, mu[1])*V(z, mu[2])*V(z, mu[3])**2)
   + chi/8*(2*(U(z, mu[1])*V(z, mu[1]))**2 + 2*(U(z, mu[2])*V(z, mu[2]))**2 + (U(z, mu[3])*V(z, mu[3]))**2)
   - 1/chi*Sum(U(z, mu[j])*V(z, mu[j]), (j,1,3))
)

# U, V as rhobar for simplifying

U_V_rhobar_gambar = [
    Eq(
        _.lhs.subs(u_v_bar_to_tilde_subs).subs(U_V_from_tilde_subs).simplify(),
        _.rhs
    )
    for _ in uv_bar_rho_gambar
]
U_V_rhobar_gambar_subs = [_.args for _ in U_V_rhobar_gambar]

# Get the constant x value of Hamiltonian

x_value = Eq(
    H_UV_chi.lhs,
    (H_UV_chi.rhs + H_UV.lhs - H_UV.rhs)
    .subs([
        C_gam_prod.doit().args,
        chi_d5_C.args,
        (C, solve(C_gam_prod_sign, C)[0]),
    ])
)
x_value = Eq(
    x_value.lhs,
    sum(
        _.factor()
        for _ in
        x_value.rhs
        .doit()
        .subs(U_V_rhobar_gambar_subs)
        .subs(gammabar[3], - gammabar[1] - gammabar[2])
        .subs([_.args for _ in gamma_bar_js_gamma_b])
        .subs(gamma[2], -gamma[1])
        .args
    )
    .expand()
    .collect(
        [rhobar(z), U(z, mu[1])*U(z, mu[2])*U(z, mu[3])**2, V(z, mu[1])*V(z, mu[2])*V(z, mu[3])**2], 
        simplify
    )
    .subs([
        ((-1)**(2*m),1),
        d5_dj
        .subs([_.args for _ in d_j_coeffs])
        .subs([_.args for _ in c_j_coeffs])
        .args
    ])
    .collect(rhobar(z), factor)
)
_rho_coeff = Eq(
    x_value.rhs.coeff(rhobar(z)),
    x_value.rhs.coeff(rhobar(z))
    .subs([gamma_lamda_bj_gam1.args])
    .simplify()
)
x_value = x_value.subs([_rho_coeff.args])

x_value = Eq(
    x_value.lhs, 
    x_value.rhs
    .subs([gamma_lamda_bj_gam1.args])
    .factor()
    .subs([
        (gamma[2], - gamma[1]),
        (b[0], solve(C_gam_prod_sign, b[0])[0]),
        (b[1], solve(b_C_one_over_chi, b[1])[0]),
        C_d5_chi.args
    ])
    .simplify()
    .subs([gamma_lamda_bj_gam1.args])
    .subs([
        (gamma[2], - gamma[1]),
        (b[0], solve(C_gam_prod_sign, b[0])[0]),
        (b[1], solve(b_C_one_over_chi, b[1])[0]),
        C_d5_chi.args
    ])
    .simplify()
    .subs([chi_d5_C.args])
    .collect(b[2], factor)
    .subs([((-1)**(2*m), 1), ((-1)**(-3*m), (-1)**m)])
    .expand()
    .collect(C, factor)
    .subs([C_d5_chi.args, ((-1)**(2*m), 1)])
)

H_UV_chi = H_UV_chi.subs([x_value.args])



H_UV_chi

Eq(-b[2]*d[5]/8 - 2/chi + 64*(d[5] + 16*lamda[1])*lamda[1]/(chi**3*d[5]**2), -(-1)**m*chi*(U(z, mu[1])*U(z, mu[2])*U(z, mu[3])**2 + V(z, mu[1])*V(z, mu[2])*V(z, mu[3])**2)/4 + chi*(2*U(z, mu[1])**2*V(z, mu[1])**2 + 2*U(z, mu[2])**2*V(z, mu[2])**2 + U(z, mu[3])**2*V(z, mu[3])**2)/8 - Sum(U(z, mu[j])*V(z, mu[j]), (j, 1, 3))/chi)

In [46]:
dU_H_chis = [Eq(diff(U(z, mu[_j + 1]),z), diff(H_UV_chi.rhs.doit(), V(z, mu[_j + 1]))) for _j in range(3)]
dV_H_chis = [Eq(diff(V(z, mu[_j + 1]),z), -diff(H_UV_chi.rhs.doit(), U(z, mu[_j + 1]))) for _j in range(3)]
d_U_V_H_chis = dU_H_chis + dV_H_chis

for _eq1, _eq2 in zip(d_U_V_H_chis, d_U_V_chi_s):
    assert (_eq1.rhs - _eq2.rhs).simplify() == 0, 'hamiltonian not giving correct diff eqns' 
    
assert diff(H_UV_chi.rhs.doit(),z).subs([_.args for _ in d_U_V_H_chis]).simplify() == 0, 'ham not conserved'

_inter_mod_pows = [
    U(z, mu[1])*V(z, mu[1]) - U(z, mu[2])*V(z, mu[2]),
    2*U(z, mu[1])*V(z, mu[1]) - U(z, mu[3])*V(z, mu[3])
]
for _ in _inter_mod_pows:
    assert diff(_,z).subs([_.args for _ in d_U_V_H_chis]).simplify() == 0, 'intermodal power not conserved'

# for _ in d_U_V_H_chis:
#     _

### Final proof from - transforming degenerate FWM to the coherent coupler

Starting from the $U, V$ system parameterised by $\chi$ and a yet unfixed new parameter $\theta$, we define the transformations and obtain the system for $\hat{u},\hat{v}$. We use $\theta$ because we cannot yet be certain the choice of gauge will deliver the correct propagation constant in the target system.

In [47]:
_theta_shift_subs = [
    (U(z, mu[1]), U(z, mu[1])*exp(theta*z)),
    (U(z, mu[2]), U(z, mu[2])*exp(-theta*z)),
    (V(z, mu[1]), V(z, mu[1])*exp(-theta*z)),
    (V(z, mu[2]), V(z, mu[2])*exp(theta*z))
]
u_v_hat_from_U_V = [
    _
    .replace(ubar, U)
    .replace(vbar, V)
    .subs(_theta_shift_subs) 
    for _ in u_v_hat_to_bar_eqs
]
U_V_hat_from_u_v = [
    Eq(U(z,mu[1]), solve(u_v_hat_from_U_V[0], U(z,mu[1]))[0]),
    Eq(U(z,mu[2]), solve(u_v_hat_from_U_V[1], U(z,mu[2]))[0]),
    Eq(V(z,mu[1]), solve(u_v_hat_from_U_V[2], V(z,mu[1]))[0]),
    Eq(V(z,mu[2]), solve(u_v_hat_from_U_V[3], V(z,mu[2]))[0])
]
U_V_hat_from_u_v_subs = [_.args for _ in U_V_hat_from_u_v]

UV_int_mod_pow = [_.replace(ubar, U).replace(vbar, V) for _ in uv_bar_sig_dj_ids]

UV3_as_uv12_hat = Eq(
    U(z, mu[3])*V(z, mu[3]), 
    solve(UV_int_mod_pow[0].subs([_.args for _ in U_V_hat_from_u_v]), U(z, mu[3])*V(z, mu[3]))[0]
    .factor()
)

d_U_V_subs = [_.args for _ in d_U_V_H_chis]
d_u_v_hat_from_U_V = [
    Eq(
        diff(_.lhs, z), 
        sum(
            __.factor()
            for __ in 
            diff(_.rhs,z).doit()
            .subs(d_U_V_subs)
            .subs(U_V_hat_from_u_v_subs)
            .expand().collect([(-1)**m, U(z, mu[3])*V(z, mu[3])], factor)
            .subs([UV3_as_uv12_hat.args])
            .args
        )
        
        
    ) 
    for _ in u_v_hat_from_U_V
]

uvhat_12_int_gam = Eq(
    uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), 
    solve(
        UV_int_mod_pow[1].subs(U_V_hat_from_u_v_subs).subs([UV3_as_uv12_hat.args]).simplify(), 
        uhat(z, mu[1])*vhat(z, mu[1])
    )[0]
     - uhat(z, mu[2])*vhat(z, mu[2])
)


for _ in u_v_hat_from_U_V:
    _
print()
for _ in UV_int_mod_pow:
    _ 
UV3_as_uv12_hat

for _ in d_u_v_hat_from_U_V:
    print()
    _
print()    
uvhat_12_int_gam

Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*U(z, mu[1])*exp(z*theta)/V(z, mu[3]))

Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*U(z, mu[2])*exp(-z*theta)/V(z, mu[3]))

Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*V(z, mu[1])*exp(-z*theta)/U(z, mu[3]))

Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*V(z, mu[2])*exp(z*theta)/U(z, mu[3]))

Eq(U(z, mu[1])*V(z, mu[1]) - U(z, mu[2])*V(z, mu[2]), d[5]/(8*(-gamma[2] + lamda[1])) - d[5]/(8*(-gamma[1] + lamda[1])))

...

Eq(Derivative(uhat(z, mu[1]), z), -(-1)**m*chi*(gamma[1] - gamma[2])*(uhat(z, mu[1])**2*uhat(z, mu[2]) + vhat(z, mu[2])*gamma[1] - vhat(z, mu[2])*lamda[1])*d[5]/(16*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) + chi*(gamma[1] - gamma[2])*(uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])*uhat(z, mu[1])*d[5]/(16*(gamma[1] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) + (chi*theta - 2)*uhat(z, mu[1])/chi)

Eq(Derivative(uhat(z, mu[2]), z), -(-1)**m*chi*(gamma[1] - gamma[2])*(uhat(z, mu[1])*uhat(z, mu[2])**2 + vhat(z, mu[1])*gamma[2] - vhat(z, mu[1])*lamda[1])*d[5]/(16*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) + chi*(gamma[1] - gamma[2])*(uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])*uhat(z, mu[2])*d[5]/(16*(gamma[2] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) - (chi*theta + 2)*uhat(z, mu[2])/chi)

Eq(Derivative(vhat(z, mu[1]), z), (-1)**m*chi*(gamma[1] - gamma[2])*(uhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*lamda[1] + vhat(z, mu[1])**2*vhat(z, mu[2]))*d[5]/(16*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) - chi*(gamma[1] - gamma[2])*(uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])*vhat(z, mu[1])*d[5]/(16*(gamma[1] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) - (chi*theta - 2)*vhat(z, mu[1])/chi)

Eq(Derivative(vhat(z, mu[2]), z), (-1)**m*chi*(gamma[1] - gamma[2])*(uhat(z, mu[1])*gamma[2] - uhat(z, mu[1])*lamda[1] + vhat(z, mu[1])*vhat(z, mu[2])**2)*d[5]/(16*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) - chi*(gamma[1] - gamma[2])*(uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])*vhat(z, mu[2])*d[5]/(16*(gamma[2] - lamda[1])*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])) + (chi*theta + 2)*vhat(z, mu[2])/chi)

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

We then express the $U,V$ Hamiltonian in $\hat{u}, \hat{v}$ modes:

In [48]:
_wm_term = H_UV_chi.rhs.coeff(chi*(-1)**m, 1)*chi*(-1)**m
H_UV_chi_wm_to_lhs = Eq(
    (-_wm_term)
    .expand().collect(chi, factor), 
    (H_UV_chi.rhs - H_UV_chi.lhs - _wm_term)
    .doit().expand().collect(chi, factor)
)
H_UV_chi_wm_to_lhs = Eq(
    (
        H_UV_chi_wm_to_lhs.lhs
        .doit().subs(U_V_hat_from_u_v_subs)
        /(U(z, mu[3])*V(z, mu[3]))**2
    )
    .expand().collect(U(z, mu[3])*V(z, mu[3]), factor)
    ,
    (
        H_UV_chi_wm_to_lhs.rhs
        .doit().subs(U_V_hat_from_u_v_subs)
        /(U(z, mu[3])*V(z, mu[3]))**2
    )
    .expand().collect(U(z, mu[3])*V(z, mu[3]), factor)
    .subs([UV3_as_uv12_hat.args])
)
_chi_coeff_swrt = H_UV_chi_wm_to_lhs.lhs.coeff(uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]), 1)
H_UV_chi_wm_to_lhs = Eq(
    H_UV_chi_wm_to_lhs.lhs/_chi_coeff_swrt,
    sum((_/_chi_coeff_swrt).factor() for _ in H_UV_chi_wm_to_lhs.rhs.args)
)


H_UV_chi_wm_to_lhs

Eq(uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]), (uhat(z, mu[1])**2*vhat(z, mu[1])**2*gamma[2]**2 - 2*uhat(z, mu[1])**2*vhat(z, mu[1])**2*gamma[2]*lamda[1] + uhat(z, mu[1])**2*vhat(z, mu[1])**2*lamda[1]**2 + uhat(z, mu[2])**2*vhat(z, mu[2])**2*gamma[1]**2 - 2*uhat(z, mu[2])**2*vhat(z, mu[2])**2*gamma[1]*lamda[1] + uhat(z, mu[2])**2*vhat(z, mu[2])**2*lamda[1]**2 + 2*gamma[1]**2*gamma[2]**2 - 4*gamma[1]**2*gamma[2]*lamda[1] + 2*gamma[1]**2*lamda[1]**2 - 4*gamma[1]*gamma[2]**2*lamda[1] + 8*gamma[1]*gamma[2]*lamda[1]**2 - 4*gamma[1]*lamda[1]**3 + 2*gamma[2]**2*lamda[1]**2 - 4*gamma[2]*lamda[1]**3 + 2*lamda[1]**4)/(2*(-1)**m*(gamma[1] - lamda[1])**(3/2)*(gamma[2] - lamda[1])**(3/2)) - 16*(-uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] + uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*vhat(z, mu[2])*lamda[1])*(uhat(z, mu[1])*vhat(z, mu[1])*gamma[2] - uhat(z, mu[1])*vhat(z, mu[1])*lamda[1] + uhat(z, mu[2])*vhat(z, mu[2])*gamma[1] - uh

We then use the Hamiltonian to simplify the $\hat{u}, \hat{v}$, in particular we want to ensure the familiar coupling to conjugate modes in the wavemixing terms:

In [49]:
# Takes about 1 - 2 minutes to run this block
_t0 = time()

uhat12_a0_chi = Eq(
    H_UV_chi_wm_to_lhs.lhs - vhat(z, mu[1])*vhat(z, mu[2]),
    H_UV_chi_wm_to_lhs.rhs - vhat(z, mu[1])*vhat(z, mu[2])
)
vhat12_a0_chi = Eq(
    H_UV_chi_wm_to_lhs.lhs - uhat(z, mu[1])*uhat(z, mu[2]),
    H_UV_chi_wm_to_lhs.rhs - uhat(z, mu[1])*uhat(z, mu[2])
)

uvhat2_chi = Eq(uhat(z, mu[2])*vhat(z, mu[2]), solve(uvhat_12_int_gam, uhat(z, mu[2])*vhat(z, mu[2]))[0])
uvhat1_chi = Eq(uhat(z, mu[1])*vhat(z, mu[1]), solve(uvhat_12_int_gam, uhat(z, mu[1])*vhat(z, mu[1]))[0])

_w_sub_eq_1_chi = Eq(w(z), -uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])
_w_sub_eq_2_chi = Eq(w(z), -uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])
_uhat_sub_1_chi = (uhat(z, mu[1]), solve(_w_sub_eq_1_chi, uhat(z, mu[1]))[0])
_vhat_sub_1_chi = (vhat(z, mu[1]), solve(_w_sub_eq_1_chi, vhat(z, mu[1]))[0])
_uhat_sub_2_chi = (uhat(z, mu[2]), solve(_w_sub_eq_2_chi, uhat(z, mu[2]))[0])
_vhat_sub_2_chi = (vhat(z, mu[2]), solve(_w_sub_eq_2_chi, vhat(z, mu[2]))[0])

du_dv_hat_simpd_chi = [
    Eq(
        d_u_v_hat_from_U_V[0].lhs,
        d_u_v_hat_from_U_V[0].rhs
        .subs([uvhat2_chi.args]).factor()
        .subs([_vhat_sub_1_chi])
        .expand().collect(w(z), factor)
        .subs([uhat12_a0_chi.args])
        .simplify()
        .subs([uvhat2_chi.args])
        .subs([_vhat_sub_1_chi])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_1_chi.args])
        .expand().collect([vhat(z, mu[2]), vhat(z, mu[1])], factor)
    ),
    Eq(
        d_u_v_hat_from_U_V[1].lhs,
        d_u_v_hat_from_U_V[1].rhs
        .subs([uvhat1_chi.args]).factor()
        .subs([_vhat_sub_2_chi])
        .expand().collect(w(z), factor)
        .subs([uhat12_a0_chi.args])
        .simplify()
        .subs([uvhat1_chi.args])
        .subs([_vhat_sub_2_chi])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_2_chi.args])
        .expand().collect([vhat(z, mu[2]), vhat(z, mu[1])], factor)
    ),
    Eq(
        d_u_v_hat_from_U_V[2].lhs,
        d_u_v_hat_from_U_V[2].rhs
        .subs([uvhat2_chi.args]).factor()
        .subs([_uhat_sub_1_chi])
        .expand().collect(w(z), factor)
        .subs([vhat12_a0_chi.args])
        .simplify()
        .subs([uvhat2_chi.args])
        .subs([_uhat_sub_1_chi])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_1_chi.args])
        .expand().collect([uhat(z, mu[2]), uhat(z, mu[1])], factor)
    ),
    Eq(
        d_u_v_hat_from_U_V[3].lhs,
        d_u_v_hat_from_U_V[3].rhs
        .subs([uvhat1_chi.args]).factor()
        .subs([_uhat_sub_2_chi])
        .expand().collect(w(z), factor)
        .subs([vhat12_a0_chi.args])
        .simplify()
        .subs([uvhat1_chi.args])
        .subs([_uhat_sub_2_chi])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_2_chi.args])
        .expand().collect([uhat(z, mu[2]), uhat(z, mu[1])], factor)
    )
]

gam_sqrd_as_C = Eq(
    -(C_gam_prod.doit().subs(gamma[2],-gamma[1]).lhs**2).expand() + lamda[1]**2, 
    -C_gam_prod.doit().subs(gamma[2],-gamma[1]).rhs**2  + lamda[1]**2
)
gam_cbd_as_C = Eq(gam_sqrd_as_C.lhs*gamma[1], gam_sqrd_as_C.rhs*gamma[1])

for _j in range(len(du_dv_hat_simpd_chi)):
    _eq = Eq(
        du_dv_hat_simpd_chi[_j].lhs,
        sum(
            _.factor() 
            for _ in
            du_dv_hat_simpd_chi[_j].rhs
            .subs([
                C_gam_prod.doit().args,
                chi_d5_C.args,
                ((-1)**(2*m),1),
                (gamma[2], -gamma[1]),
                d5_C_b_lam.args,
                gam_sqrd_as_C.args,
                gam_cbd_as_C.args,
                (C, solve(C_gam_prod_sign, C)[0]),
                ((-1)**(2*m),1)
            ])
            .args
        )
    )
    du_dv_hat_simpd_chi[_j] = Eq(
        _eq.lhs, 
        sum(_.subs(lam_step_down).factor() for _ in _eq.rhs.args)
    )
    
# Simplify prop constants  
_b_prop_const_1 = Eq(
    du_dv_hat_simpd_chi[0].rhs
    .coeff(uhat(z, mu[1]),1),
    du_dv_hat_simpd_chi[0].rhs
    .coeff(uhat(z, mu[1]),1)
    .expand().collect([gamma[1], b[2]*gamma[1]], factor)
)
_b_prop_const_2 = Eq(
    du_dv_hat_simpd_chi[1].rhs
    .coeff(uhat(z, mu[2]),1),
    du_dv_hat_simpd_chi[1].rhs
    .coeff(uhat(z, mu[2]),1)
    .expand().collect([gamma[1], b[2]*gamma[1]], factor)
)
_b_prop_const_1 = Eq(
    fraction(_b_prop_const_1.lhs)[0].factor()/fraction(_b_prop_const_1.lhs)[1].simplify(), 
    _b_prop_const_1.rhs
)
_b_prop_const_2 = Eq(
    fraction(_b_prop_const_2.lhs)[0].factor()/fraction(_b_prop_const_2.lhs)[1].simplify(), 
    _b_prop_const_2.rhs
)
du_dv_hat_simpd_chi = [
    _.subs([
        _b_prop_const_1.args, 
        _b_prop_const_2.args
    ]) 
    for _ in du_dv_hat_simpd_chi
]


for _ in du_dv_hat_simpd_chi:
    _
    
print(f'time taken: {time() - _t0}')

Eq(Derivative(uhat(z, mu[1]), z), (b[1]/2 + b[2]*gamma[1] + theta - 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))*uhat(z, mu[1]) - uhat(z, mu[1])**2*vhat(z, mu[1])*b[2] + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), ((b[1] - 2*theta)/2 - b[2]*gamma[1] + 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))*uhat(z, mu[2]) - uhat(z, mu[2])**2*vhat(z, mu[2])*b[2] + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), -(b[1]/2 + b[2]*gamma[1] + theta - 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))*vhat(z, mu[1]) + uhat(z, mu[1])*vhat(z, mu[1])**2*b[2] - uhat(z, mu[2]))

Eq(Derivative(vhat(z, mu[2]), z), -((b[1] - 2*theta)/2 - b[2]*gamma[1] + 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))*vhat(z, mu[2]) - uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*b[2])

time taken: 104.56123185157776


We now fix $\theta$ and obtain our target system, thus proving the theorem that the degenerate FWM system projects to a coherent coupler system:

In [50]:
theta_b_gam_lam = Eq(theta, -b[2]*gamma[1] + 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))
du_dv_hat_simpd_theta = [_.subs([theta_b_gam_lam.args]).expand() for _ in du_dv_hat_simpd_chi]
du_dv_hat_simpd_theta_subs = [_.args for _ in du_dv_hat_simpd_theta]

for _ in du_dv_hat_eqs:
    assert _.subs(du_dv_hat_simpd_theta_subs)

theta_b_gam_lam
print()
for _ in du_dv_hat_simpd_theta:
    _

Eq(theta, -b[2]*gamma[1] + 2*gamma[1]/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))

Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])**2*vhat(z, mu[1])*b[2] + uhat(z, mu[1])*b[1]/2 + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[2])**2*vhat(z, mu[2])*b[2] + uhat(z, mu[2])*b[1]/2 + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), uhat(z, mu[1])*vhat(z, mu[1])**2*b[2] - uhat(z, mu[2]) - vhat(z, mu[1])*b[1]/2)

Eq(Derivative(vhat(z, mu[2]), z), -uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*b[2] - vhat(z, mu[2])*b[1]/2)

We can simplify the Hamiltonian to the following expected form:

In [51]:
# Check the Hamiltonian simplifies to teh expected form
gamma1_sqrd_chi = Eq(gamma[1]**2, 
   solve(
        gamma_j_prod_b_poly.doit().subs([
            (gamma[2], - gamma[1]),
            (C_gam_prod_sqrd.rhs, C_gam_prod_sqrd.lhs),
            C_d5_chi.args,
            ((-1)**(-2*m),1)
        ]).expand(),
       gamma[1]**2
   )[0]
)
d5_b_chi_lam = Eq(d[5], solve(b_C_one_over_chi.subs([C_d5_chi.args]),d[5])[0])

rhohat_uv12 = Eq(rhohat(z), - (uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))/2)

_uvhat12_rhohat_subs = solve(
    [uvhat_12_int_gam, rhohat_uv12], 
    [uhat(z, mu[1])*vhat(z, mu[1]), uhat(z, mu[2])*vhat(z, mu[2])]
)


H_UV_chi_wm_rhohat = Eq(
    H_UV_chi_wm_to_lhs.lhs,
    H_UV_chi_wm_to_lhs.rhs
    .subs(_uvhat12_rhohat_subs)
    .subs(gamma[2], -gamma[1])
    .expand().collect(rhohat(z), factor)
)
H_UV_chi_wm_rhohat = Eq(
    H_UV_chi_wm_rhohat.lhs,
    sum(
        _.factor() 
        for _ in 
        H_UV_chi_wm_rhohat.rhs
        .subs([
            ((-1)**m, solve(C_gam_prod_sign, (-1)**m)[0]),
            (
                sqrt(-gamma[1] - lamda[1]), 
                solve(C_gam_prod.doit().subs(gamma[2], - gamma[1]), sqrt(-gamma[1] - lamda[1]))[0]
            ),
            C_gam_prod_sign.subs([C_d5_chi.args]).args,
            (
                gamma[1] - lamda[1],
                solve(gamma_j_prod_b_poly.doit(), gamma[1] - lamda[1])[0]
                .subs(gamma[2], - gamma[1])
                .subs([C_gam_prod_sign.subs([C_d5_chi.args]).args])
            ),
            gamma1_sqrd_chi.args,
            d5_b_chi_lam.args,
            (chi, solve(b_C_one_over_chi,chi)[0]),
            (C, solve(C_gam_prod_sign, C)[0])
        ])
        .args
    )
)

H_uvhat_from_chi = Eq(
    0,
    H_UV_chi_wm_rhohat.lhs -
    (
        H_UV_chi_wm_rhohat.rhs
        .subs([rhohat_uv12.args])
        + b[2]*(uvhat_12_int_gam.lhs**2 - uvhat_12_int_gam.rhs**2)/4
    )
    .expand()
)
_const_part_H_uvhat = H_uvhat_from_chi.rhs.subs([(uhat(z, mu[1]),0), (vhat(z, mu[2]),0)])
H_uvhat_from_chi = Eq(
    (H_uvhat_from_chi.lhs - _const_part_H_uvhat).expand().collect(b[0], factor),
    (H_uvhat_from_chi.rhs - _const_part_H_uvhat).expand().collect([b[1], b[2]], factor)
)

assert H_uvhat_from_chi.subs([Hhat_uv_b.doit().args]).simplify()
    
uvhat_12_int_gam
rhohat_uv12

H_UV_chi_wm_rhohat
H_uvhat_from_chi

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

Eq(rhohat(z), -uhat(z, mu[1])*vhat(z, mu[1])/2 - uhat(z, mu[2])*vhat(z, mu[2])/2)

Eq(uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]), rhohat(z)**2*b[2] + rhohat(z)*b[1] + b[0])

Eq(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0], (uhat(z, mu[1])*vhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2]))*b[1]/2 - (uhat(z, mu[1])**2*vhat(z, mu[1])**2 + uhat(z, mu[2])**2*vhat(z, mu[2])**2)*b[2]/2 + uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]))

## Projecting a generic 3-mode degenerate fwm model to a 2-mode coherent coupler model

In this section we prove the theorem in which a generic 3-mode degenerate four-wave mixing model can be projected into a 2-mode coherent coupler model. We choose to start from a degenerate fwm model in which there is no quartic term. We have previously proven that every fwm model can be brought to that form so we do not lose any generality.

We begin by defining the canonical Hamiltonian for a 3 mode degenerate four-wave mixing model with arbitrary parameters $a_j$. We have simplified matters by picking simple parameterless SPM/XPM coefficients at the start but that is not an important detail as we have previously shown how to transform fwm models to this form.

In [52]:
a0deg = Eq(
    a[0] + Sum(a[j]*u(z, mu[j])*(v(z, mu[j])),(j,1,3)) 
    + 2*Sum((u(z, mu[j])*v(z, mu[j]))**2/3, (j,1,3)),
    u(z,mu[1])*u(z, mu[2])*u(z, mu[3])**2 + v(z,mu[1])*v(z, mu[2])*v(z, mu[3])**2
)
Hdeg = Eq(
    H(u(z,mu[1]),u(z,mu[2]),u(z,mu[3]),v(z,mu[1]),v(z,mu[2]),v(z,mu[3])),
     - Sum(a[j]*u(z, mu[j])*(v(z, mu[j])),(j,1,3)) 
    - 2*Sum((u(z, mu[j])*v(z, mu[j]))**2/3, (j,1,3))
    + u(z,mu[1])*u(z, mu[2])*u(z, mu[3])**2 + v(z,mu[1])*v(z, mu[2])*v(z, mu[3])**2
)
a0deg

Hdeg

Eq(a[0] + 2*Sum(u(z, mu[j])**2*v(z, mu[j])**2/3, (j, 1, 3)) + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 3)), u(z, mu[1])*u(z, mu[2])*u(z, mu[3])**2 + v(z, mu[1])*v(z, mu[2])*v(z, mu[3])**2)

Eq(H(u(z, mu[1]), u(z, mu[2]), u(z, mu[3]), v(z, mu[1]), v(z, mu[2]), v(z, mu[3])), u(z, mu[1])*u(z, mu[2])*u(z, mu[3])**2 + v(z, mu[1])*v(z, mu[2])*v(z, mu[3])**2 - 2*Sum(u(z, mu[j])**2*v(z, mu[j])**2/3, (j, 1, 3)) - Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 3)))

The equations of motion for the degenerate fwm model are then:

In [53]:
du_degs = [
    Eq(diff(u(z, mu[_j+1]),z), diff(Hdeg.doit().rhs, v(z, mu[_j+1])))
    for _j in range(3)
]
dv_degs = [
    Eq(diff(v(z, mu[_j+1]),z), -diff(Hdeg.doit().rhs, u(z, mu[_j+1])))
    for _j in range(3)
]
du_dv_degs = du_degs + dv_degs
du_dv_deg_subs = [_.args for _ in du_dv_degs]

for _ in du_dv_degs:
    _

Eq(Derivative(u(z, mu[1]), z), -4*u(z, mu[1])**2*v(z, mu[1])/3 - u(z, mu[1])*a[1] + v(z, mu[2])*v(z, mu[3])**2)

Eq(Derivative(u(z, mu[2]), z), -4*u(z, mu[2])**2*v(z, mu[2])/3 - u(z, mu[2])*a[2] + v(z, mu[1])*v(z, mu[3])**2)

Eq(Derivative(u(z, mu[3]), z), -4*u(z, mu[3])**2*v(z, mu[3])/3 - u(z, mu[3])*a[3] + 2*v(z, mu[1])*v(z, mu[2])*v(z, mu[3]))

Eq(Derivative(v(z, mu[1]), z), 4*u(z, mu[1])*v(z, mu[1])**2/3 - u(z, mu[2])*u(z, mu[3])**2 + v(z, mu[1])*a[1])

Eq(Derivative(v(z, mu[2]), z), -u(z, mu[1])*u(z, mu[3])**2 + 4*u(z, mu[2])*v(z, mu[2])**2/3 + v(z, mu[2])*a[2])

Eq(Derivative(v(z, mu[3]), z), -2*u(z, mu[1])*u(z, mu[2])*u(z, mu[3]) + 4*u(z, mu[3])*v(z, mu[3])**2/3 + v(z, mu[3])*a[3])

We then derive the model power evolution differential equations and intermodal power conservation laws, and define the negative mean mode power function $\rho(z)$. We confirm that there is no quartic term in the $\rho(z)$ differential equation.

In [54]:
# Define diff eqns for modal powers 
duv_degs = [
    Eq(
        Derivative(u(z, mu[_j+1])*v(z, mu[_j+1]),z),
        Derivative(u(z, mu[_j+1])*v(z, mu[_j+1]),z)
        .doit()
        .subs(du_dv_deg_subs)
        .expand().factor()
    )
    for _j in range(3)
]

# Define the negative mean rho for the fwm model
rho_aj = Eq(rho(z), -Sum(u(z, mu[j])*v(z, mu[j]),(j,1,3))/3)

# Derive intermodal power conservation laws
duv_pow_rules = [
    Eq(duv_degs[0].lhs - duv_degs[1].lhs, duv_degs[0].rhs - duv_degs[1].rhs),
    Eq(2*duv_degs[0].lhs - duv_degs[2].lhs, (2*duv_degs[0].rhs - duv_degs[2].rhs).simplify()),
    Eq(2*duv_degs[1].lhs - duv_degs[2].lhs, (2*duv_degs[1].rhs - duv_degs[2].rhs).simplify())
]


duv_rhos = [
    Eq(
        Derivative(
            -Rational(3,4)*rho_aj.doit().rhs
            .subs([
                (u(z,mu[2])*v(z,mu[2]), u(z,mu[1])*v(z,mu[1])), 
                (u(z,mu[3])*v(z,mu[3]), 2*u(z,mu[1])*v(z,mu[1]))
            ])
            .factor()
            ,
            z
        ),
        -diff(Rational(3,4)*rho_aj.lhs, z)
    ),
    Eq(
        Derivative(
            -Rational(3,4)*rho_aj.doit().rhs
            .subs([
                (u(z,mu[1])*v(z,mu[1]), u(z,mu[2])*v(z,mu[2])), 
                (u(z,mu[3])*v(z,mu[3]), 2*u(z,mu[2])*v(z,mu[2]))
            ]).factor()
            .factor()
            ,
            z
        ),
        -diff(Rational(3,4)*rho_aj.lhs, z)
    ),
    Eq(
        Derivative(
            -Rational(3,2)*rho_aj.doit().rhs
            .subs([
                (u(z,mu[1])*v(z,mu[1]), u(z,mu[3])*v(z,mu[3])/2), 
                (u(z,mu[2])*v(z,mu[2]), u(z,mu[3])*v(z,mu[3])/2)
            ])
            .factor()
            ,
            z
        ),
        -diff(Rational(3,2)*rho_aj.lhs, z)
    )
]
drho_mean = Eq(sum(_.rhs for _ in duv_rhos)/3, sum(_.lhs for _ in duv_rhos)/3)

assert drho_mean.subs([rho_aj.doit().args]).simplify()

# Express mdoal powers in terms of rho
uv_gam_rhos = [
    Eq(u(z, mu[1])*v(z, mu[1]), gamma[1] - Rational(3,4)*rho(z)),
    Eq(u(z, mu[2])*v(z, mu[2]), gamma[2] - Rational(3,4)*rho(z)),
    Eq(u(z, mu[3])*v(z, mu[3]), gamma[3] - Rational(3,2)*rho(z))
]
rho_mean = Eq(sum(_.rhs for _ in uv_gam_rhos)/3, sum(_.lhs for _ in uv_gam_rhos)/3)
gamj_sum = Eq(
    Sum(gamma[j], (j,1,3)), 
    3*solve(rho_mean.subs([rho_aj.args]).doit(), Sum(gamma[j], (j,1,3)).doit()/3)[0]
)
uv_gam_rho_subs = [_.args for _ in uv_gam_rhos]

# Check no quartic term (we chose to fix this condition it is not neccessarily required)
drho_no_quartic = Eq(
    Rational(16,9)*duv_degs[0].lhs.subs(uv_gam_rho_subs).doit()**2,
    (
        Rational(16,9)*(duv_degs[0].rhs**2 - a0deg.rhs**2 + a0deg.lhs.doit()**2)
    )
    .expand()
    .subs(uv_gam_rho_subs)
    .expand().collect(rho(z), factor)
)
assert drho_no_quartic.rhs.coeff(rho(z),4) == 0, 'error: rho^4 term found in diff eqn'

for _ in duv_degs:
    _
print()

for _ in duv_pow_rules:
    _
print()

for _ in duv_rhos:
    _
print()
    
for _ in uv_gam_rhos:
    _
print()

gamj_sum

Eq(Derivative(u(z, mu[1])*v(z, mu[1]), z), -u(z, mu[1])*u(z, mu[2])*u(z, mu[3])**2 + v(z, mu[1])*v(z, mu[2])*v(z, mu[3])**2)

Eq(Derivative(u(z, mu[2])*v(z, mu[2]), z), -u(z, mu[1])*u(z, mu[2])*u(z, mu[3])**2 + v(z, mu[1])*v(z, mu[2])*v(z, mu[3])**2)

Eq(Derivative(u(z, mu[3])*v(z, mu[3]), z), -2*(u(z, mu[1])*u(z, mu[2])*u(z, mu[3])**2 - v(z, mu[1])*v(z, mu[2])*v(z, mu[3])**2))

Eq(Derivative(u(z, mu[1])*v(z, mu[1]), z) - Derivative(u(z, mu[2])*v(z, mu[2]), z), 0)

Eq(2*Derivative(u(z, mu[1])*v(z, mu[1]), z) - Derivative(u(z, mu[3])*v(z, mu[3]), z), 0)

...

Eq(Derivative(u(z, mu[3])*v(z, mu[3]), z), -3*Derivative(rho(z), z)/2)

Eq(u(z, mu[1])*v(z, mu[1]), -3*rho(z)/4 + gamma[1])

Eq(u(z, mu[2])*v(z, mu[2]), -3*rho(z)/4 + gamma[2])

Eq(u(z, mu[3])*v(z, mu[3]), -3*rho(z)/2 + gamma[3])

Eq(Sum(gamma[j], (j, 1, 3)), 0)

Now we define the transform to hat coordinates and in doing so reduce the modes from 3 to 2. We initially leave constants $\epsilon_j$ arbitrary but we'll later see a natural choice is evident. We note the intermodal power conservation laws for the two new modes in hat coordinates.

In [55]:
# Define transforms to hat coordinates 3 modes -> 2 modes
u_v_hat_degs = [
    Eq(uhat(z, mu[1]), epsilon[1]*u(z, mu[1])/v(z, mu[3])),
    Eq(uhat(z, mu[2]), epsilon[2]*u(z, mu[2])/v(z, mu[3])),
    Eq(vhat(z, mu[1]), epsilon[1]*v(z, mu[1])/u(z, mu[3])),
    Eq(vhat(z, mu[2]), epsilon[2]*v(z, mu[2])/u(z, mu[3]))
]
u_v_hat_degs_subs = [_.args for _ in u_v_hat_degs]

# Derive expresssion for uv3 in terms of uvhat1 uvhat2
uv3_uv1hat_uv2hat = Eq(
    u_v_hat_degs[0].lhs*u_v_hat_degs[2].lhs*epsilon[2]**2 - 
    u_v_hat_degs[1].lhs*u_v_hat_degs[3].lhs*epsilon[1]**2,
    (
    u_v_hat_degs[0].rhs*u_v_hat_degs[2].rhs*epsilon[2]**2 - 
    u_v_hat_degs[1].rhs*u_v_hat_degs[3].rhs*epsilon[1]**2
    )
    .subs([uv_gam_rho_subs[0], uv_gam_rho_subs[1]])
    .simplify()
)
uv3_uv1hat_uv2hat = Eq(u(z, mu[3])*v(z, mu[3]), solve(uv3_uv1hat_uv2hat, u(z, mu[3])*v(z, mu[3]))[0])

# Derive intermodal power conservation for uvhat1 - uvhat2
_uvhat12_term = (
    - (2*gamma[1] - gamma[3])*uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2 
    + (2*gamma[2] - gamma[3])*uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2
)
uvhat12_const_law = Eq(
    _uvhat12_term,
    _uvhat12_term
    .subs(u_v_hat_degs_subs)
    .subs(uv_gam_rho_subs)
    .simplify()
)

# Derive utility functions for converting between uvhat1 and uvhat2
uvhat1 = Eq(
    uhat(z, mu[1])*vhat(z, mu[1]), 
    solve(uvhat12_const_law, uhat(z, mu[1])*vhat(z, mu[1]))[0]
)
uvhat2 = Eq(
    uhat(z, mu[2])*vhat(z, mu[2]), 
    solve(uvhat12_const_law, uhat(z, mu[2])*vhat(z, mu[2]))[0]
)


for _ in u_v_hat_degs:
    _
    
uv3_uv1hat_uv2hat
uvhat12_const_law

Eq(uhat(z, mu[1]), u(z, mu[1])*epsilon[1]/v(z, mu[3]))

Eq(uhat(z, mu[2]), u(z, mu[2])*epsilon[2]/v(z, mu[3]))

Eq(vhat(z, mu[1]), v(z, mu[1])*epsilon[1]/u(z, mu[3]))

Eq(vhat(z, mu[2]), v(z, mu[2])*epsilon[2]/u(z, mu[3]))

Eq(u(z, mu[3])*v(z, mu[3]), (gamma[1] - gamma[2])*epsilon[1]**2*epsilon[2]**2/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2))

Eq((-2*gamma[1] + gamma[3])*uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2 + (2*gamma[2] - gamma[3])*uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2, (-gamma[1] + gamma[2])*epsilon[1]**2*epsilon[2]**2)

We can then derive differential equations for these modes using the differential equations of the FWM system. These initially appear complicated with unwanted terms in denominators. We recognise that there are wave mixing terms that are not conjugated like we would conventionally expect them to be. 

In [56]:
u_v_as_hats = [
    (u(z, mu[1]), solve(u_v_hat_degs[0], u(z, mu[1]))[0]),
    (u(z, mu[2]), solve(u_v_hat_degs[1], u(z, mu[2]))[0]),
    (v(z, mu[1]), solve(u_v_hat_degs[2], v(z, mu[1]))[0]),
    (v(z, mu[2]), solve(u_v_hat_degs[3], v(z, mu[2]))[0])
]

du_dv_hat_from_degs = [
    Eq(
        diff(_.lhs, z),
        diff(_.rhs, z)
        .subs(du_dv_deg_subs)
        .subs(u_v_as_hats)
        .expand()
        .subs([uv3_uv1hat_uv2hat.args])
    )
    for _ in u_v_hat_degs
]
du_dv_hat_from_degs_subs = [_.args for _ in du_dv_hat_from_degs]

for _ in du_dv_hat_from_degs:
    _

Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])*a[1] - uhat(z, mu[1])*a[3] + 2*(gamma[1] - gamma[2])*uhat(z, mu[1])**2*uhat(z, mu[2])*epsilon[1]*epsilon[2]/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2) - 4*(gamma[1] - gamma[2])*uhat(z, mu[1])**2*vhat(z, mu[1])*epsilon[2]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)) - 4*(gamma[1] - gamma[2])*uhat(z, mu[1])*epsilon[1]**2*epsilon[2]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)) + (gamma[1] - gamma[2])*vhat(z, mu[2])*epsilon[1]**3*epsilon[2]/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[2])*a[2] - uhat(z, mu[2])*a[3] + 2*(gamma[1] - gamma[2])*uhat(z, mu[1])*uhat(z, mu[2])**2*epsilon[1]*epsilon[2]/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2) - 4*(gamma[1] - gamma[2])*uhat(z, mu[2])**2*vhat(z, mu[2])*epsilon[1]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)) - 4*(gamma[1] - gamma[2])*uhat(z, mu[2])*epsilon[1]**2*epsilon[2]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)) + (gamma[1] - gamma[2])*vhat(z, mu[1])*epsilon[1]*epsilon[2]**3/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2))

Eq(Derivative(vhat(z, mu[1]), z), vhat(z, mu[1])*a[1] + vhat(z, mu[1])*a[3] + 4*(gamma[1] - gamma[2])*uhat(z, mu[1])*vhat(z, mu[1])**2*epsilon[2]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)) - (gamma[1] - gamma[2])*uhat(z, mu[2])*epsilon[1]**3*epsilon[2]/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2) - 2*(gamma[1] - gamma[2])*vhat(z, mu[1])**2*vhat(z, mu[2])*epsilon[1]*epsilon[2]/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2) + 4*(gamma[1] - gamma[2])*vhat(z, mu[1])*epsilon[1]**2*epsilon[2]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)))

Eq(Derivative(vhat(z, mu[2]), z), vhat(z, mu[2])*a[2] + vhat(z, mu[2])*a[3] - (gamma[1] - gamma[2])*uhat(z, mu[1])*epsilon[1]*epsilon[2]**3/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2) + 4*(gamma[1] - gamma[2])*uhat(z, mu[2])*vhat(z, mu[2])**2*epsilon[1]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)) - 2*(gamma[1] - gamma[2])*vhat(z, mu[1])*vhat(z, mu[2])**2*epsilon[1]*epsilon[2]/(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2) + 4*(gamma[1] - gamma[2])*vhat(z, mu[2])*epsilon[1]**2*epsilon[2]**2/(3*(uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2 - uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2)))

We also transform the conserved FWM hamiltonian into a conservation law in hat coordinates. We can then use this to swap unconjugated wave mixing terms into conjugated ones.

In [57]:
a0hat = a0deg.doit().subs(u_v_as_hats)
a0hat = Eq(
    (epsilon[1]*epsilon[2]*a0hat.lhs/(u(z, mu[3])*v(z, mu[3]))**2)
    .expand()
    .subs([uv3_uv1hat_uv2hat.args])
    .expand()
    .collect([
        (uhat(z, mu[1])*vhat(z, mu[1]))**2,
        (uhat(z, mu[2])*vhat(z, mu[2]))**2,
        uhat(z, mu[1])*vhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[2]),
        uhat(z, mu[1])*vhat(z, mu[1]),
        uhat(z, mu[2])*vhat(z, mu[2])
    ], factor),
    (epsilon[1]*epsilon[2]*a0hat.rhs/(u(z, mu[3])*v(z, mu[3]))**2)
    .expand()
)

a0hat

Eq(2*epsilon[1]*epsilon[2]/3 + uhat(z, mu[1])*vhat(z, mu[1])*a[3]*epsilon[2]/((gamma[1] - gamma[2])*epsilon[1]) - uhat(z, mu[2])*vhat(z, mu[2])*a[3]*epsilon[1]/((gamma[1] - gamma[2])*epsilon[2]) - (2*a[0] + a[1]*gamma[1] - a[1]*gamma[2] - a[2]*gamma[1] + a[2]*gamma[2])*uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])/((gamma[1] - gamma[2])**2*epsilon[1]*epsilon[2]) + (3*a[0] + 3*a[1]*gamma[1] - 3*a[1]*gamma[2] + 2*gamma[1]**2 - 4*gamma[1]*gamma[2] + 2*gamma[2]**2)*uhat(z, mu[1])**2*vhat(z, mu[1])**2*epsilon[2]/(3*(gamma[1] - gamma[2])**2*epsilon[1]**3) + (3*a[0] - 3*a[2]*gamma[1] + 3*a[2]*gamma[2] + 2*gamma[1]**2 - 4*gamma[1]*gamma[2] + 2*gamma[2]**2)*uhat(z, mu[2])**2*vhat(z, mu[2])**2*epsilon[1]/(3*(gamma[1] - gamma[2])**2*epsilon[2]**3), uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]))

After using the transformed hamiltonian constant to swap unconjugated for conjugated wave mixing terms we reassuringly recover the expected form of a coherent coupler system with recognisable phase velocity mismatch, SPM, and linear wave mixing terms.

In [58]:
uhat12_a0 = Eq(a0hat.rhs - vhat(z, mu[1])*vhat(z, mu[2]), a0hat.lhs - vhat(z, mu[1])*vhat(z, mu[2]))
vhat12_a0 = Eq(a0hat.rhs - uhat(z, mu[1])*uhat(z, mu[2]), a0hat.lhs - uhat(z, mu[1])*uhat(z, mu[2]))

_w_sub_eq_1 = Eq(w(z), -2*uhat(z, mu[1])*vhat(z, mu[1]) + epsilon[1]**2)
_w_sub_eq_2 = Eq(w(z), -2*uhat(z, mu[2])*vhat(z, mu[2]) + epsilon[2]**2)
_uhat_sub_1 = (uhat(z, mu[1]), solve(_w_sub_eq_1, uhat(z, mu[1]))[0])
_vhat_sub_1 = (vhat(z, mu[1]), solve(_w_sub_eq_1, vhat(z, mu[1]))[0])
_uhat_sub_2 = (uhat(z, mu[2]), solve(_w_sub_eq_2, uhat(z, mu[2]))[0])
_vhat_sub_2 = (vhat(z, mu[2]), solve(_w_sub_eq_2, vhat(z, mu[2]))[0])

du_dv_hat_simpd = [
    Eq(
        du_dv_hat_from_degs[0].lhs,
        du_dv_hat_from_degs[0].rhs
        .subs([uvhat2.args]).factor()
        .subs([_vhat_sub_1])
        .expand().collect(w(z), factor)
        .subs([uhat12_a0.args])
        .simplify()
        .subs([uvhat2.args])
        .subs([_vhat_sub_1])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_1.args])
        .expand().collect([vhat(z, mu[2]), vhat(z, mu[1])], factor)
    ),
    Eq(
        du_dv_hat_from_degs[1].lhs,
        du_dv_hat_from_degs[1].rhs
        .subs([uvhat1.args]).factor()
        .subs([_vhat_sub_2])
        .expand().collect(w(z), factor)
        .subs([uhat12_a0.args])
        .simplify()
        .subs([uvhat1.args])
        .subs([_vhat_sub_2])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_2.args])
        .expand().collect([vhat(z, mu[2]), vhat(z, mu[1])], factor)
    ),
    Eq(
        du_dv_hat_from_degs[2].lhs,
        du_dv_hat_from_degs[2].rhs
        .subs([uvhat2.args]).factor()
        .subs([_uhat_sub_1])
        .expand().collect(w(z), factor)
        .subs([vhat12_a0.args])
        .simplify()
        .subs([uvhat2.args])
        .subs([_uhat_sub_1])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_1.args])
        .expand().collect([uhat(z, mu[2]), uhat(z, mu[1])], factor)
    ),
    Eq(
        du_dv_hat_from_degs[3].lhs,
        du_dv_hat_from_degs[3].rhs
        .subs([uvhat1.args]).factor()
        .subs([_uhat_sub_2])
        .expand().collect(w(z), factor)
        .subs([vhat12_a0.args])
        .simplify()
        .subs([uvhat1.args])
        .subs([_uhat_sub_2])
        .expand().collect(w(z), factor)
        .subs([_w_sub_eq_2.args])
        .expand().collect([uhat(z, mu[2]), uhat(z, mu[1])], factor)
    )
]


for _ in du_dv_hat_simpd:
    _

Eq(Derivative(uhat(z, mu[1]), z), -(2*gamma[1] - gamma[3])*vhat(z, mu[2])*epsilon[1]/epsilon[2] - (6*a[0] + 6*a[1]*gamma[1] - 3*a[1]*gamma[3] - 6*a[2]*gamma[1] + 6*a[2]*gamma[2] - 6*a[3]*gamma[1] + 3*a[3]*gamma[3] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*uhat(z, mu[1])/(3*(2*gamma[1] - gamma[3])) + 2*(6*a[0] + 6*a[1]*gamma[1] - 3*a[1]*gamma[3] + 6*a[2]*gamma[2] - 3*a[2]*gamma[3] + 4*gamma[1]**2 - 4*gamma[1]*gamma[3] + 4*gamma[2]**2 - 4*gamma[2]*gamma[3] + 2*gamma[3]**2)*uhat(z, mu[1])**2*vhat(z, mu[1])/(3*(2*gamma[1] - gamma[3])*epsilon[1]**2))

Eq(Derivative(uhat(z, mu[2]), z), -(2*gamma[2] - gamma[3])*vhat(z, mu[1])*epsilon[2]/epsilon[1] - (6*a[0] + 6*a[1]*gamma[1] - 6*a[1]*gamma[2] + 6*a[2]*gamma[2] - 3*a[2]*gamma[3] - 6*a[3]*gamma[2] + 3*a[3]*gamma[3] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*uhat(z, mu[2])/(3*(2*gamma[2] - gamma[3])) + 2*(6*a[0] + 6*a[1]*gamma[1] - 3*a[1]*gamma[3] + 6*a[2]*gamma[2] - 3*a[2]*gamma[3] + 4*gamma[1]**2 - 4*gamma[1]*gamma[3] + 4*gamma[2]**2 - 4*gamma[2]*gamma[3] + 2*gamma[3]**2)*uhat(z, mu[2])**2*vhat(z, mu[2])/(3*(2*gamma[2] - gamma[3])*epsilon[2]**2))

Eq(Derivative(vhat(z, mu[1]), z), (2*gamma[1] - gamma[3])*uhat(z, mu[2])*epsilon[1]/epsilon[2] + (6*a[0] + 6*a[1]*gamma[1] - 3*a[1]*gamma[3] - 6*a[2]*gamma[1] + 6*a[2]*gamma[2] - 6*a[3]*gamma[1] + 3*a[3]*gamma[3] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*vhat(z, mu[1])/(3*(2*gamma[1] - gamma[3])) - 2*(6*a[0] + 6*a[1]*gamma[1] - 3*a[1]*gamma[3] + 6*a[2]*gamma[2] - 3*a[2]*gamma[3] + 4*gamma[1]**2 - 4*gamma[1]*gamma[3] + 4*gamma[2]**2 - 4*gamma[2]*gamma[3] + 2*gamma[3]**2)*uhat(z, mu[1])*vhat(z, mu[1])**2/(3*(2*gamma[1] - gamma[3])*epsilon[1]**2))

Eq(Derivative(vhat(z, mu[2]), z), (2*gamma[2] - gamma[3])*uhat(z, mu[1])*epsilon[2]/epsilon[1] + (6*a[0] + 6*a[1]*gamma[1] - 6*a[1]*gamma[2] + 6*a[2]*gamma[2] - 3*a[2]*gamma[3] - 6*a[3]*gamma[2] + 3*a[3]*gamma[3] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*vhat(z, mu[2])/(3*(2*gamma[2] - gamma[3])) - 2*(6*a[0] + 6*a[1]*gamma[1] - 3*a[1]*gamma[3] + 6*a[2]*gamma[2] - 3*a[2]*gamma[3] + 4*gamma[1]**2 - 4*gamma[1]*gamma[3] + 4*gamma[2]**2 - 4*gamma[2]*gamma[3] + 2*gamma[3]**2)*uhat(z, mu[2])*vhat(z, mu[2])**2/(3*(2*gamma[2] - gamma[3])*epsilon[2]**2))

In [59]:
duv_hat_from_degs = [
    Eq(
        Derivative(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]),z), 
        Derivative(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]),z)
        .doit()
        .subs(du_dv_hat_from_degs_subs)
        .simplify()
        .factor()
        .subs([uvhat2.args])
        .factor()
    )
    for _j in range(2)
]

duv12_const = Eq(
    sum(Derivative(_,z) for _ in uvhat12_const_law.lhs.args),
    diff(uvhat12_const_law.lhs,z)
    .doit()
    .subs(du_dv_hat_from_degs_subs)
    .simplify()
    .factor()
    .subs([
        uvhat2.args
    ])
    .simplify()
    .factor()
)


for _ in duv_hat_from_degs:
    _

duv12_const

Eq(Derivative(uhat(z, mu[1])*vhat(z, mu[1]), z), (uhat(z, mu[1])*uhat(z, mu[2]) - vhat(z, mu[1])*vhat(z, mu[2]))*(2*gamma[1] - gamma[3])*epsilon[1]/epsilon[2])

Eq(Derivative(uhat(z, mu[2])*vhat(z, mu[2]), z), (uhat(z, mu[1])*uhat(z, mu[2]) - vhat(z, mu[1])*vhat(z, mu[2]))*(2*gamma[2] - gamma[3])*epsilon[2]/epsilon[1])

Eq(Derivative((-2*gamma[1] + gamma[3])*uhat(z, mu[2])*vhat(z, mu[2])*epsilon[1]**2, z) + Derivative((2*gamma[2] - gamma[3])*uhat(z, mu[1])*vhat(z, mu[1])*epsilon[2]**2, z), 0)

At this point we note that one particular choice for the $\epsilon_j$ constants seems natural. The following choice makes the wave mixing terms equal in the differential equations which enables us to obtain them from a single wave mixing term in a canonical Hamiltonian. Simultaneously, it gives an unweighted natural intermodal power conservation law.

In [60]:
epsilon_values = [
    (epsilon[1], I*sqrt(2*gamma[2] - gamma[3])),
    (epsilon[2], I*sqrt(2*gamma[1] - gamma[3]))
]
uvhat12_gam12_cons = Eq(
    (-uvhat12_const_law.lhs/epsilon[1]**2/epsilon[2]**2).subs(epsilon_values)
    .factor(),
    (-uvhat12_const_law.rhs/epsilon[1]**2/epsilon[2]**2).subs(epsilon_values)
)


for _ in epsilon_values:
    Eq(*_)
    
uvhat12_gam12_cons

Eq(epsilon[1], I*sqrt(2*gamma[2] - gamma[3]))

Eq(epsilon[2], I*sqrt(2*gamma[1] - gamma[3]))

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

In [61]:
gammahats_as_gammas = [
    Eq(gammahat[1] - gammahat[2], gamma[1] - gamma[2]),
    Eq(gammahat[1] + gammahat[2], 0)
]
gammahats_sols = solve(gammahats_as_gammas, [gammahat[1], gammahat[2]])
gammahats = [Eq(_k, gammahats_sols[_k]) for _k in gammahats_sols]
for _ in gammahats_as_gammas:
    _
print() 
for _ in gammahats:
    _

Eq(gammahat[1] - gammahat[2], gamma[1] - gamma[2])

Eq(gammahat[1] + gammahat[2], 0)

Eq(gammahat[1], gamma[1]/2 - gamma[2]/2)

Eq(gammahat[2], -gamma[1]/2 + gamma[2]/2)

The differential equations then become:

In [62]:
du_dv_hat_simpd_b = [
    _.subs(epsilon_values).subs(gamma[3], -gamma[1]-gamma[2]) 
    for _ in du_dv_hat_simpd
]
du_dv_hat_simpd_b = [Eq(_.lhs, sum(__.factor() for __ in _.rhs.args)) for _ in du_dv_hat_simpd_b]

for _ in du_dv_hat_simpd_b:
    _

Eq(Derivative(uhat(z, mu[1]), z), -sqrt(gamma[1] + 3*gamma[2])*sqrt(3*gamma[1] + gamma[2])*vhat(z, mu[2]) - (6*a[0] + 9*a[1]*gamma[1] + 3*a[1]*gamma[2] - 6*a[2]*gamma[1] + 6*a[2]*gamma[2] - 9*a[3]*gamma[1] - 3*a[3]*gamma[2] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*uhat(z, mu[1])/(3*(3*gamma[1] + gamma[2])) - 2*(6*a[0] + 9*a[1]*gamma[1] + 3*a[1]*gamma[2] + 3*a[2]*gamma[1] + 9*a[2]*gamma[2] + 10*gamma[1]**2 + 12*gamma[1]*gamma[2] + 10*gamma[2]**2)*uhat(z, mu[1])**2*vhat(z, mu[1])/(3*(gamma[1] + 3*gamma[2])*(3*gamma[1] + gamma[2])))

Eq(Derivative(uhat(z, mu[2]), z), -sqrt(gamma[1] + 3*gamma[2])*sqrt(3*gamma[1] + gamma[2])*vhat(z, mu[1]) - (6*a[0] + 6*a[1]*gamma[1] - 6*a[1]*gamma[2] + 3*a[2]*gamma[1] + 9*a[2]*gamma[2] - 3*a[3]*gamma[1] - 9*a[3]*gamma[2] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*uhat(z, mu[2])/(3*(gamma[1] + 3*gamma[2])) - 2*(6*a[0] + 9*a[1]*gamma[1] + 3*a[1]*gamma[2] + 3*a[2]*gamma[1] + 9*a[2]*gamma[2] + 10*gamma[1]**2 + 12*gamma[1]*gamma[2] + 10*gamma[2]**2)*uhat(z, mu[2])**2*vhat(z, mu[2])/(3*(gamma[1] + 3*gamma[2])*(3*gamma[1] + gamma[2])))

Eq(Derivative(vhat(z, mu[1]), z), sqrt(gamma[1] + 3*gamma[2])*sqrt(3*gamma[1] + gamma[2])*uhat(z, mu[2]) + (6*a[0] + 9*a[1]*gamma[1] + 3*a[1]*gamma[2] - 6*a[2]*gamma[1] + 6*a[2]*gamma[2] - 9*a[3]*gamma[1] - 3*a[3]*gamma[2] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*vhat(z, mu[1])/(3*(3*gamma[1] + gamma[2])) + 2*(6*a[0] + 9*a[1]*gamma[1] + 3*a[1]*gamma[2] + 3*a[2]*gamma[1] + 9*a[2]*gamma[2] + 10*gamma[1]**2 + 12*gamma[1]*gamma[2] + 10*gamma[2]**2)*uhat(z, mu[1])*vhat(z, mu[1])**2/(3*(gamma[1] + 3*gamma[2])*(3*gamma[1] + gamma[2])))

Eq(Derivative(vhat(z, mu[2]), z), sqrt(gamma[1] + 3*gamma[2])*sqrt(3*gamma[1] + gamma[2])*uhat(z, mu[1]) + (6*a[0] + 6*a[1]*gamma[1] - 6*a[1]*gamma[2] + 3*a[2]*gamma[1] + 9*a[2]*gamma[2] - 3*a[3]*gamma[1] - 9*a[3]*gamma[2] + 4*gamma[1]**2 - 8*gamma[1]*gamma[2] + 4*gamma[2]**2)*vhat(z, mu[2])/(3*(gamma[1] + 3*gamma[2])) + 2*(6*a[0] + 9*a[1]*gamma[1] + 3*a[1]*gamma[2] + 3*a[2]*gamma[1] + 9*a[2]*gamma[2] + 10*gamma[1]**2 + 12*gamma[1]*gamma[2] + 10*gamma[2]**2)*uhat(z, mu[2])*vhat(z, mu[2])**2/(3*(gamma[1] + 3*gamma[2])*(3*gamma[1] + gamma[2])))

The transformed constant that comes from the original FWM Hamiltonian is not the canonical Hamiltonian for this system but it is a conserved quantity.

In [63]:
# This is the constant derived by substituting mode transforms into the original hamiltonian
a0hat_b = Eq(
    sum(_*sqrt(2*gamma[1] - gamma[3])*sqrt(2*gamma[2] - gamma[3]) for _ in a0hat.lhs.subs(epsilon_values).args),
    a0hat.rhs.subs(epsilon_values)*sqrt(2*gamma[1] - gamma[3])*sqrt(2*gamma[2] - gamma[3])
)
a0hat_b = Eq(0, a0hat_b.lhs - a0hat_b.rhs)
a0hat_const_term = a0hat_b.rhs.subs([
    (uhat(z, mu[1]),0),(uhat(z, mu[2]),0),(vhat(z, mu[1]),0),(vhat(z, mu[2]),0)
]).factor()
a0hat_b = Eq(
    (a0hat_b.lhs - a0hat_const_term)
    .subs(gamma[3], -gamma[1]-gamma[2]), 
    (a0hat_b.rhs - a0hat_const_term)
    .expand()
    .collect([
        (uhat(z, mu[1])*vhat(z, mu[1]))**2,
        (uhat(z, mu[2])*vhat(z, mu[2]))**2,
        uhat(z, mu[1])*vhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[2]),
        uhat(z, mu[1])*vhat(z, mu[1]),
        uhat(z, mu[2])*vhat(z, mu[2])
    ], factor)
    .subs(gamma[3], -gamma[1]-gamma[2])
    .subs([
        ((2*(gamma[1] - gamma[2])**2).expand(), 2*(gamma[1] - gamma[2])**2),
        ((3*a[1]*(gamma[1] - gamma[2])).expand(), (3*a[1]*(gamma[1] - gamma[2]))),
        ((3*a[2]*(gamma[1] - gamma[2])).expand(), (3*a[2]*(gamma[1] - gamma[2])))
    ])
)

gamma_upsilon_eqs = [
    Eq(gamma[1] - gamma[2], upsilon[m]),
    Eq(gamma[1] + gamma[2], upsilon[p])
]

_gamma_subs = solve(gamma_upsilon_eqs, [gamma[1], gamma[2]])

a0hat_c = Eq(
    a0hat_b.lhs.subs(_gamma_subs).simplify(),
    sum(_.subs(_gamma_subs).simplify() for _ in a0hat_b.rhs.args)
)


a0hat_b
print()
for _ in gamma_upsilon_eqs:
    _
print()
a0hat_c

Eq(2*(gamma[1] + 3*gamma[2])*(3*gamma[1] + gamma[2])/3, -(uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]))*sqrt(gamma[1] + 3*gamma[2])*sqrt(3*gamma[1] + gamma[2]) - (gamma[1] + 3*gamma[2])*uhat(z, mu[2])*vhat(z, mu[2])*a[3]/(gamma[1] - gamma[2]) + (3*gamma[1] + gamma[2])*uhat(z, mu[1])*vhat(z, mu[1])*a[3]/(gamma[1] - gamma[2]) - (gamma[1] + 3*gamma[2])*(2*(gamma[1] - gamma[2])**2 - 3*(gamma[1] - gamma[2])*a[2] + 3*a[0])*uhat(z, mu[2])**2*vhat(z, mu[2])**2/(3*(gamma[1] - gamma[2])**2*(3*gamma[1] + gamma[2])) + (2*a[0] + a[1]*gamma[1] - a[1]*gamma[2] - a[2]*gamma[1] + a[2]*gamma[2])*uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])/(gamma[1] - gamma[2])**2 - (3*gamma[1] + gamma[2])*(2*(gamma[1] - gamma[2])**2 + 3*(gamma[1] - gamma[2])*a[1] + 3*a[0])*uhat(z, mu[1])**2*vhat(z, mu[1])**2/(3*(gamma[1] - gamma[2])**2*(gamma[1] + 3*gamma[2])))

Eq(gamma[1] - gamma[2], upsilon[m])

Eq(gamma[1] + gamma[2], upsilon[p])

Eq(-2*upsilon[m]**2/3 + 8*upsilon[p]**2/3, (-uhat(z, mu[1])*uhat(z, mu[2]) - vhat(z, mu[1])*vhat(z, mu[2]))*sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p]) + (upsilon[m] - 2*upsilon[p])*uhat(z, mu[2])*vhat(z, mu[2])*a[3]/upsilon[m] + (upsilon[m] - 2*upsilon[p])*(3*a[0] - 3*a[2]*upsilon[m] + 2*upsilon[m]**2)*uhat(z, mu[2])**2*vhat(z, mu[2])**2/(3*(upsilon[m] + 2*upsilon[p])*upsilon[m]**2) + (upsilon[m] + 2*upsilon[p])*uhat(z, mu[1])*vhat(z, mu[1])*a[3]/upsilon[m] + (2*a[0] + a[1]*upsilon[m] - a[2]*upsilon[m])*uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])/upsilon[m]**2 + (upsilon[m] + 2*upsilon[p])*(3*a[0] + 3*a[1]*upsilon[m] + 2*upsilon[m]**2)*uhat(z, mu[1])**2*vhat(z, mu[1])**2/(3*(upsilon[m] - 2*upsilon[p])*upsilon[m]**2))

In [64]:
_ao_hat_cons_check = (
    diff(a0hat_b.rhs,z)
    .subs([_.args for _ in du_dv_hat_simpd_b])
    .subs(gamma[3], -gamma[1]-gamma[2])
    .simplify()
    .subs([uhat12_a0.args, vhat12_a0.args, uvhat2.args])
    .subs(epsilon_values)
    .subs(gamma[3], -gamma[1]-gamma[2])
    .simplify()
)

assert _ao_hat_cons_check == 0, 'transformed hamiltonian not conserveed'

However, we are able to guess the canonical Hamiltonian from the differential equations of the derived coherent coupler and we can then show how to recast the transformed FWM hamiltonian constant into this form.

In [65]:
du_dv_hat_simpd_c = [
    Eq(
        _.lhs.subs(_gamma_subs).simplify(),
        sum(
            __
            .subs([
                ((4*(gamma[1] - gamma[2])**2).expand(), 4*(gamma[1] - gamma[2])**2),
                ((3*a[1]*(gamma[1] - gamma[2])).expand(), (3*a[1]*(gamma[1] - gamma[2]))),
                ((3*a[2]*(gamma[1] - gamma[2])).expand(), (3*a[2]*(gamma[1] - gamma[2])))
            ])
            .subs(_gamma_subs)
            .simplify() 
            for __ in _.rhs.args
        )
    )
    for _ in du_dv_hat_simpd_b
]

for _ in du_dv_hat_simpd_c:
    _

Eq(Derivative(uhat(z, mu[1]), z), -sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p])*vhat(z, mu[2]) + 2*(6*a[0] + 3*a[1]*upsilon[m] + 6*a[1]*upsilon[p] - 3*a[2]*upsilon[m] + 6*a[2]*upsilon[p] + 2*upsilon[m]**2 + 8*upsilon[p]**2)*uhat(z, mu[1])**2*vhat(z, mu[1])/(3*(upsilon[m]**2 - 4*upsilon[p]**2)) + (-6*a[0] - 3*a[1]*upsilon[m] - 6*a[1]*upsilon[p] + 6*a[2]*upsilon[m] + 3*a[3]*upsilon[m] + 6*a[3]*upsilon[p] - 4*upsilon[m]**2)*uhat(z, mu[1])/(3*(upsilon[m] + 2*upsilon[p])))

Eq(Derivative(uhat(z, mu[2]), z), -sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p])*vhat(z, mu[1]) + 2*(6*a[0] + 3*a[1]*upsilon[m] + 6*a[1]*upsilon[p] - 3*a[2]*upsilon[m] + 6*a[2]*upsilon[p] + 2*upsilon[m]**2 + 8*upsilon[p]**2)*uhat(z, mu[2])**2*vhat(z, mu[2])/(3*(upsilon[m]**2 - 4*upsilon[p]**2)) + (6*a[0] + 6*a[1]*upsilon[m] - 3*a[2]*upsilon[m] + 6*a[2]*upsilon[p] + 3*a[3]*upsilon[m] - 6*a[3]*upsilon[p] + 4*upsilon[m]**2)*uhat(z, mu[2])/(3*(upsilon[m] - 2*upsilon[p])))

Eq(Derivative(vhat(z, mu[1]), z), sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p])*uhat(z, mu[2]) + 2*(-6*a[0] - 3*a[1]*upsilon[m] - 6*a[1]*upsilon[p] + 3*a[2]*upsilon[m] - 6*a[2]*upsilon[p] - 2*upsilon[m]**2 - 8*upsilon[p]**2)*uhat(z, mu[1])*vhat(z, mu[1])**2/(3*(upsilon[m]**2 - 4*upsilon[p]**2)) + (6*a[0] + 3*a[1]*upsilon[m] + 6*a[1]*upsilon[p] - 6*a[2]*upsilon[m] - 3*a[3]*upsilon[m] - 6*a[3]*upsilon[p] + 4*upsilon[m]**2)*vhat(z, mu[1])/(3*(upsilon[m] + 2*upsilon[p])))

Eq(Derivative(vhat(z, mu[2]), z), sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p])*uhat(z, mu[1]) + 2*(-6*a[0] - 3*a[1]*upsilon[m] - 6*a[1]*upsilon[p] + 3*a[2]*upsilon[m] - 6*a[2]*upsilon[p] - 2*upsilon[m]**2 - 8*upsilon[p]**2)*uhat(z, mu[2])*vhat(z, mu[2])**2/(3*(upsilon[m]**2 - 4*upsilon[p]**2)) + (-6*a[0] - 6*a[1]*upsilon[m] + 3*a[2]*upsilon[m] - 6*a[2]*upsilon[p] - 3*a[3]*upsilon[m] + 6*a[3]*upsilon[p] - 4*upsilon[m]**2)*vhat(z, mu[2])/(3*(upsilon[m] - 2*upsilon[p])))

In [66]:
# Actual canonical hamiltonian of the transformed diff eqns
ham_transd = Eq(
    x,
    -sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p])
    *(uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]))
    + (
        -6*a[0] - 3*a[1]*upsilon[m] - 6*a[1]*upsilon[p] + 6*a[2]*upsilon[m] + 
        3*a[3]*upsilon[m] + 6*a[3]*upsilon[p] - 4*upsilon[m]**2
    )/(3*(upsilon[m] + 2*upsilon[p]))*uhat(z, mu[1])*vhat(z, mu[1]) +
    (
        6*a[0] + 3*a[1]*upsilon[m] + 6*a[1]*upsilon[p] - 3*a[2]*upsilon[m] + 
        6*a[2]*upsilon[p] + 2*upsilon[m]**2 + 8*upsilon[p]**2
    )/(3*(upsilon[m]**2 - 4*upsilon[p]**2))
    *(uhat(z, mu[1])**2*vhat(z, mu[1])**2 + uhat(z, mu[2])**2*vhat(z, mu[2])**2) +
    (
        6*a[0] + 6*a[1]*upsilon[m] - 3*a[2]*upsilon[m] + 6*a[2]*upsilon[p] + 
        3*a[3]*upsilon[m] - 6*a[3]*upsilon[p] + 4*upsilon[m]**2
    )/(3*(upsilon[m] - 2*upsilon[p]))*uhat(z, mu[2])*vhat(z, mu[2])
)

# Find x

rhohat_uvhat12 = Eq(rhohat(z), uhat(z, mu[1])*vhat(z, mu[1])/2 + uhat(z, mu[2])*vhat(z, mu[2])/2)
uvhat12_rho_sols = solve(
    [uvhat12_gam12_cons.subs(_gamma_subs), rhohat_uvhat12],
    [uhat(z, mu[1])*vhat(z, mu[1]), uhat(z, mu[2])*vhat(z, mu[2])]
)
_x_val = Eq(
    x,
    solve(Eq(
        ham_transd.lhs - a0hat_c.lhs,
        (ham_transd.rhs - a0hat_c.rhs)
        .subs(uvhat12_rho_sols)
        .expand().collect(rhohat(z), simplify)
    ),x)[0]
    .expand().collect([a[0], a[1], a[2], a[3]], factor)
)
ham_transd = ham_transd.subs([_x_val.args])

assert (diff(ham_transd.rhs, vhat(z, mu[1])) - du_dv_hat_simpd_c[0].rhs).simplify() == 0
assert (diff(ham_transd.rhs, vhat(z, mu[2])) - du_dv_hat_simpd_c[1].rhs).simplify() == 0
assert (diff(ham_transd.rhs, uhat(z, mu[1])) + du_dv_hat_simpd_c[2].rhs).simplify() == 0
assert (diff(ham_transd.rhs, uhat(z, mu[2])) + du_dv_hat_simpd_c[3].rhs).simplify() == 0

ham_transd

Eq(-2*a[3]*upsilon[p] + a[2]*upsilon[m]**2/(upsilon[m] + 2*upsilon[p]) - a[1]*upsilon[m]**2/(upsilon[m] - 2*upsilon[p]) - (upsilon[m]**2 + 4*upsilon[p]**2)*a[0]/((upsilon[m] - 2*upsilon[p])*(upsilon[m] + 2*upsilon[p])) - 2*(3*upsilon[m]**4 - 8*upsilon[m]**2*upsilon[p]**2 + 16*upsilon[p]**4)/(3*(upsilon[m] - 2*upsilon[p])*(upsilon[m] + 2*upsilon[p])), -(uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]))*sqrt(-upsilon[m] + 2*upsilon[p])*sqrt(upsilon[m] + 2*upsilon[p]) + (uhat(z, mu[1])**2*vhat(z, mu[1])**2 + uhat(z, mu[2])**2*vhat(z, mu[2])**2)*(6*a[0] + 3*a[1]*upsilon[m] + 6*a[1]*upsilon[p] - 3*a[2]*upsilon[m] + 6*a[2]*upsilon[p] + 2*upsilon[m]**2 + 8*upsilon[p]**2)/(3*upsilon[m]**2 - 12*upsilon[p]**2) + (-6*a[0] - 3*a[1]*upsilon[m] - 6*a[1]*upsilon[p] + 6*a[2]*upsilon[m] + 3*a[3]*upsilon[m] + 6*a[3]*upsilon[p] - 4*upsilon[m]**2)*uhat(z, mu[1])*vhat(z, mu[1])/(3*upsilon[m] + 6*upsilon[p]) + (6*a[0] + 6*a[1]*upsilon[m] - 3*a[2]*upsilon[m] + 6*a[2]*upsilon[p] + 3*a[3]*upsilon[

This completes the proof that a degenerate 3-mode FWM model can be projected into a canonical Hamiltonian system for a 2-mode coherent coupler.

## Switching the conjugation on mode 2 to match Jensen

In this section we show how the abstract coherent coupler model we have been considering is related to the physical model considered by Jensen.

In [67]:
_vu2_switch = [(v(z, mu[2]),x), (u(z, mu[2]), v(z, mu[2])), (x, u(z, mu[2]))]
a_0_eq_swtch = a_0_eq.doit().subs(_vu2_switch)
H_swtch = Eq(
    a_0_eq_swtch.rhs - a_0_eq_swtch.rhs + a[0], 
    (a_0_eq_swtch.rhs - a_0_eq_swtch.lhs + a[0])
    .subs(ajk_syms)
)

du_dv_swtch = [
    Eq(diff(u(z, mu[1]),z), diff(H_swtch.rhs, v(z, mu[1])).simplify().collect([u(z, mu[1]), u(z, mu[2])])),
    Eq(diff(u(z, mu[2]),z), diff(H_swtch.rhs, v(z, mu[2])).simplify().collect([u(z, mu[1]), u(z, mu[2])])),
    Eq(diff(v(z, mu[1]),z), -diff(H_swtch.rhs, u(z, mu[1])).simplify().collect([v(z, mu[1]), v(z, mu[2])])),
    Eq(diff(v(z, mu[2]),z), -diff(H_swtch.rhs, u(z, mu[2])).simplify().collect([v(z, mu[1]), v(z, mu[2])]))   
]
du_dv_swtch_subs = [_.args for _ in du_dv_swtch]

for _ in du_dv_swtch:
    _
    
H_swtch

Eq(Derivative(u(z, mu[1]), z), (-u(z, mu[2])*v(z, mu[2])*a[1, 2] - a[1])*u(z, mu[1]) - u(z, mu[1])**2*v(z, mu[1])*a[1, 1] + u(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), (-u(z, mu[2])*v(z, mu[1])*a[1, 2] + 1)*u(z, mu[1]) - u(z, mu[2])**2*v(z, mu[2])*a[2, 2] - u(z, mu[2])*a[2])

Eq(Derivative(v(z, mu[1]), z), -(-u(z, mu[2])*v(z, mu[2])*a[1, 2] - a[1])*v(z, mu[1]) + u(z, mu[1])*v(z, mu[1])**2*a[1, 1] - v(z, mu[2]))

Eq(Derivative(v(z, mu[2]), z), -(-u(z, mu[1])*v(z, mu[2])*a[1, 2] + 1)*v(z, mu[1]) + u(z, mu[2])*v(z, mu[2])**2*a[2, 2] + v(z, mu[2])*a[2])

Eq(a[0], -(u(z, mu[1])*v(z, mu[1])*a[1, 1] + u(z, mu[2])*v(z, mu[2])*a[1, 2])*u(z, mu[1])*v(z, mu[1])/2 - (u(z, mu[1])*v(z, mu[1])*a[1, 2] + u(z, mu[2])*v(z, mu[2])*a[2, 2])*u(z, mu[2])*v(z, mu[2])/2 - u(z, mu[1])*v(z, mu[1])*a[1] + u(z, mu[1])*v(z, mu[2]) + u(z, mu[2])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2])*a[2])

In [68]:
Horig = Eq(a[0], solve(a_0_eq.doit(), a[0])[0])

Horig
H_swtch.expand()

Eq(a[0], -u(z, mu[1])**2*v(z, mu[1])**2*a[1, 1]/2 - u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[1, 2]/2 - u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[2, 1]/2 + u(z, mu[1])*u(z, mu[2]) - u(z, mu[1])*v(z, mu[1])*a[1] - u(z, mu[2])**2*v(z, mu[2])**2*a[2, 2]/2 - u(z, mu[2])*v(z, mu[2])*a[2] + v(z, mu[1])*v(z, mu[2]))

Eq(a[0], -u(z, mu[1])**2*v(z, mu[1])**2*a[1, 1]/2 - u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*v(z, mu[1])*a[1] + u(z, mu[1])*v(z, mu[2]) - u(z, mu[2])**2*v(z, mu[2])**2*a[2, 2]/2 + u(z, mu[2])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2])*a[2])

In [69]:
(Horig.rhs - H_swtch.rhs.subs(_vu2_switch)).subs(ajk_syms).expand()

0

In [70]:
duv_pow_diff_swtch = Eq(
    Derivative(u(z,mu[1])*v(z, mu[1]) + u(z,mu[2])*v(z, mu[2]), z),
    diff(u(z,mu[1])*v(z, mu[1]) + u(z,mu[2])*v(z, mu[2]), z)
    .subs(du_dv_swtch_subs).simplify()
)

duv_pow_diff_swtch

Eq(Derivative(u(z, mu[1])*v(z, mu[1]) + u(z, mu[2])*v(z, mu[2]), z), 0)

In [71]:
for _ in du_dv_swtch:
    _.subs(_vu2_switch).expand()

Eq(Derivative(u(z, mu[1]), z), -u(z, mu[1])**2*v(z, mu[1])*a[1, 1] - u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*a[1] + v(z, mu[2]))

Eq(Derivative(v(z, mu[2]), z), -u(z, mu[1])*v(z, mu[1])*v(z, mu[2])*a[1, 2] + u(z, mu[1]) - u(z, mu[2])*v(z, mu[2])**2*a[2, 2] - v(z, mu[2])*a[2])

Eq(Derivative(v(z, mu[1]), z), u(z, mu[1])*v(z, mu[1])**2*a[1, 1] + u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*a[1, 2] - u(z, mu[2]) + v(z, mu[1])*a[1])

Eq(Derivative(u(z, mu[2]), z), u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*a[1, 2] + u(z, mu[2])**2*v(z, mu[2])*a[2, 2] + u(z, mu[2])*a[2] - v(z, mu[1]))

In [72]:
_vu2_switchb = [(v(z, mu[2]),x), (u(z, mu[2]), v(z, mu[2])), (x, u(z, mu[2]))]

duj.subs(j,1).doit().expand()
du_dv_swtch[0].subs(_vu2_switchb).expand()

duj.subs(j,2).doit().expand()
du_dv_swtch[3].subs(_vu2_switchb).expand()
# dvj

Eq(Derivative(u(z, mu[1]), z), -u(z, mu[1])**2*v(z, mu[1])*a[1, 1] - u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*a[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[1]), z), -u(z, mu[1])**2*v(z, mu[1])*a[1, 1] - u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*a[1, 2] - u(z, mu[1])*a[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), -u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*a[2, 1] - u(z, mu[2])**2*v(z, mu[2])*a[2, 2] - u(z, mu[2])*a[2] + v(z, mu[1]))

Eq(Derivative(u(z, mu[2]), z), u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*a[1, 2] + u(z, mu[2])**2*v(z, mu[2])*a[2, 2] + u(z, mu[2])*a[2] - v(z, mu[1]))

Jensen consider the following model where $\mathrm{Ac}$ denotes $A^*$ the complex conjugate:

In [73]:
jensen = [
    Eq(
        -I*Derivative(A(z, mu[1]), z),
        Omega[1]*A(z, mu[1]) + Omega[2]*A(z, mu[2]) +
        (Omega[3]*A(z, mu[1])*Ac(z, mu[1]) + 2*Omega[4]*A(z, mu[2])*Ac(z, mu[2]))*A(z, mu[1])
    ),
    Eq(
        -I*Derivative(A(z, mu[2]), z),
        Omega[1]*A(z, mu[2]) + Omega[2]*A(z, mu[1]) +
        (Omega[3]*A(z, mu[2])*Ac(z, mu[2]) + 2*Omega[4]*A(z, mu[1])*Ac(z, mu[1]))*A(z, mu[2])
    )
]
jensen[0] = Eq(I*jensen[0].lhs, I*jensen[0].rhs)
jensen[1] = Eq(I*jensen[1].lhs, I*jensen[1].rhs)

_conj = [(A, u), (Ac, A), (u, Ac), (I,-I)]
jensen.append(jensen[0].subs(_conj))
jensen.append(jensen[1].subs(_conj))

jensen_subs = [_.args for _ in jensen]

jens_p0 = Eq(p[0], A(z, mu[1])*Ac(z, mu[1]) + A(z, mu[2])*Ac(z, mu[2]))
jens_p1 = Eq(
    p[1], 
    2*(A(z, mu[1])*Ac(z, mu[2]) + A(z, mu[2])*Ac(z, mu[1])) -
    2*(Omega[3] - 2*Omega[4])/Omega[2]*A(z, mu[1])*Ac(z, mu[1])*A(z, mu[2])*Ac(z, mu[2])
)

assert diff(jens_p0.rhs, z).doit().subs(jensen_subs).simplify() == 0
assert diff(jens_p1.rhs, z).doit().subs(jensen_subs).simplify().collect(Omega[4], factor) == 0

for _ in jensen:
    _
    
jens_p0
jens_p1

Eq(Derivative(A(z, mu[1]), z), I*((A(z, mu[1])*Ac(z, mu[1])*Omega[3] + 2*A(z, mu[2])*Ac(z, mu[2])*Omega[4])*A(z, mu[1]) + A(z, mu[1])*Omega[1] + A(z, mu[2])*Omega[2]))

Eq(Derivative(A(z, mu[2]), z), I*((2*A(z, mu[1])*Ac(z, mu[1])*Omega[4] + A(z, mu[2])*Ac(z, mu[2])*Omega[3])*A(z, mu[2]) + A(z, mu[1])*Omega[2] + A(z, mu[2])*Omega[1]))

Eq(Derivative(Ac(z, mu[1]), z), -I*((A(z, mu[1])*Ac(z, mu[1])*Omega[3] + 2*A(z, mu[2])*Ac(z, mu[2])*Omega[4])*Ac(z, mu[1]) + Ac(z, mu[1])*Omega[1] + Ac(z, mu[2])*Omega[2]))

Eq(Derivative(Ac(z, mu[2]), z), -I*((2*A(z, mu[1])*Ac(z, mu[1])*Omega[4] + A(z, mu[2])*Ac(z, mu[2])*Omega[3])*Ac(z, mu[2]) + Ac(z, mu[1])*Omega[2] + Ac(z, mu[2])*Omega[1]))

Eq(p[0], A(z, mu[1])*Ac(z, mu[1]) + A(z, mu[2])*Ac(z, mu[2]))

Eq(p[1], -(2*Omega[3] - 4*Omega[4])*A(z, mu[1])*A(z, mu[2])*Ac(z, mu[1])*Ac(z, mu[2])/Omega[2] + 2*A(z, mu[1])*Ac(z, mu[2]) + 2*A(z, mu[2])*Ac(z, mu[1]))

If we normalise the length scale so the wave mixing term is equal to 1 then we can obtain Jensen's system as folows:

In [74]:
my_subs = [
    (A(z, mu[1]), u(z, mu[1])*exp(I*pi/4)), 
    (A(z, mu[2]), v(z, mu[2])*exp(-I*pi/4)),
    (Ac(z, mu[1]), -v(z, mu[1])*exp(-I*pi/4)),
    (Ac(z, mu[2]), u(z, mu[2])*exp(I*pi/4)),
    (Omega[2], 1)
]

_eq_ = jensen[2].subs(my_subs).expand().doit()
_eq_ = Eq(-_eq_.lhs, -_eq_.rhs)


jensen_to_uv = [
    Eq(
        diff(u(z, mu[1]),z), 
        solve(jensen[0].subs(my_subs).expand().doit(), diff(u(z, mu[1]),z))[0]
        .expand()
    ),
    Eq(
        diff(u(z, mu[2]),z), 
        solve(jensen[3].subs(my_subs).expand().doit(), diff(u(z, mu[2]),z))[0]
        .expand()
    ),
    Eq(
        diff(v(z, mu[1]),z), 
        solve(jensen[2].subs(my_subs).expand().doit(), diff(v(z, mu[1]),z))[0]
        .expand()
    ),
    Eq(
        diff(v(z, mu[2]),z), 
        solve(jensen[1].subs(my_subs).expand().doit(), diff(v(z, mu[2]),z))[0]
        .expand()
    )
]

for _ in my_subs:
    Eq(*_)
print()
for _ in jensen_to_uv:
    _

Eq(A(z, mu[1]), u(z, mu[1])*exp(I*pi/4))

Eq(A(z, mu[2]), v(z, mu[2])*exp(-I*pi/4))

Eq(Ac(z, mu[1]), -v(z, mu[1])*exp(-I*pi/4))

Eq(Ac(z, mu[2]), u(z, mu[2])*exp(I*pi/4))

Eq(Omega[2], 1)

Eq(Derivative(u(z, mu[1]), z), -I*u(z, mu[1])**2*v(z, mu[1])*Omega[3] + 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*Omega[4] + I*u(z, mu[1])*Omega[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*Omega[4] - I*u(z, mu[2])**2*v(z, mu[2])*Omega[3] - I*u(z, mu[2])*Omega[1] + v(z, mu[1]))

Eq(Derivative(v(z, mu[1]), z), I*u(z, mu[1])*v(z, mu[1])**2*Omega[3] - 2*I*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[2]) - I*v(z, mu[1])*Omega[1])

Eq(Derivative(v(z, mu[2]), z), -2*I*u(z, mu[1])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[1]) + I*u(z, mu[2])*v(z, mu[2])**2*Omega[3] + I*v(z, mu[2])*Omega[1])

In [75]:
my_subs_b = [
    (a[1], -I*Omega[1]),
    (a[2], I*Omega[1]),
    (a[1,1], I*Omega[3]),
    (a[2,2], I*Omega[3]),
    (a[1,2], -2*I*Omega[4]),
    (a[2,1], -2*I*Omega[4])
]

uv_to_jensen = [
    duj.subs(j,1).doit().expand().subs(my_subs_b),
    duj.subs(j,2).doit().expand().subs(my_subs_b),
    dvj.subs(j,1).doit().expand().subs(my_subs_b),
    dvj.subs(j,2).doit().expand().subs(my_subs_b)
]

for _j in range(4):
    assert (jensen_to_uv[_j].rhs - uv_to_jensen[_j].rhs).factor().simplify() == 0, 'not equivalent to jensen'

for _ in my_subs_b:
    Eq(*_)
print()
for _ in uv_to_jensen:
    _

Eq(a[1], -I*Omega[1])

Eq(a[2], I*Omega[1])

Eq(a[1, 1], I*Omega[3])

Eq(a[2, 2], I*Omega[3])

Eq(a[1, 2], -2*I*Omega[4])

Eq(a[2, 1], -2*I*Omega[4])

Eq(Derivative(u(z, mu[1]), z), -I*u(z, mu[1])**2*v(z, mu[1])*Omega[3] + 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[2])*Omega[4] + I*u(z, mu[1])*Omega[1] + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), 2*I*u(z, mu[1])*u(z, mu[2])*v(z, mu[1])*Omega[4] - I*u(z, mu[2])**2*v(z, mu[2])*Omega[3] - I*u(z, mu[2])*Omega[1] + v(z, mu[1]))

Eq(Derivative(v(z, mu[1]), z), I*u(z, mu[1])*v(z, mu[1])**2*Omega[3] - 2*I*u(z, mu[2])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[2]) - I*v(z, mu[1])*Omega[1])

Eq(Derivative(v(z, mu[2]), z), -2*I*u(z, mu[1])*v(z, mu[1])*v(z, mu[2])*Omega[4] - u(z, mu[1]) + I*u(z, mu[2])*v(z, mu[2])**2*Omega[3] + I*v(z, mu[2])*Omega[1])

More generally we would do:

In [76]:
my_subs_c = [
    (a[1], -I*Omega[1]/Omega[2]),
    (a[2], I*Omega[1]/Omega[2]),
    (a[1,1], I*Omega[3]/Omega[2]),
    (a[2,2], I*Omega[3]/Omega[2]),
    (a[1,2], -I*2*Omega[4]/Omega[2]),
    (a[2,1], -I*2*Omega[4]/Omega[2]),
    (u(z, mu[1]), A(z/Omega[2], mu[1])*exp(-I*pi/4)), 
    (v(z, mu[2]), A(z/Omega[2], mu[2])*exp(I*pi/4)),
    (v(z, mu[1]), -Ac(z/Omega[2], mu[1])*exp(I*pi/4)),
    (u(z, mu[2]), Ac(z/Omega[2], mu[2])*exp(-I*pi/4))
]

uv_to_jensen_A = [
    duj.subs(j,1).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit(),
    duj.subs(j,2).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit(),
    dvj.subs(j,1).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit(),
    dvj.subs(j,2).doit().expand().subs(my_subs_c).subs(z, x*Omega[2]).doit()
]
uv_to_jensen_A = [
    Eq(diff(A(x, mu[1]),x), solve(uv_to_jensen_A[0], diff(A(x, mu[1]),x))[0].expand()),
    Eq(diff(A(x, mu[2]),x), solve(uv_to_jensen_A[3], diff(A(x, mu[2]),x))[0].expand()),
    Eq(diff(Ac(x, mu[1]),x), solve(uv_to_jensen_A[2], diff(Ac(x, mu[1]),x))[0].expand()),
    Eq(diff(Ac(x, mu[2]),x), solve(uv_to_jensen_A[1], diff(Ac(x, mu[2]),x))[0].expand())
]

for _j in range(4):
    assert (jensen[_j].rhs - uv_to_jensen_A[_j].rhs.subs(x,z)).factor().simplify() == 0, 'not equivalent to Jensen'

for _ in my_subs_c:
    Eq(*_)
print()
for _ in uv_to_jensen_A:
    _

Eq(a[1], -I*Omega[1]/Omega[2])

Eq(a[2], I*Omega[1]/Omega[2])

Eq(a[1, 1], I*Omega[3]/Omega[2])

Eq(a[2, 2], I*Omega[3]/Omega[2])

Eq(a[1, 2], -2*I*Omega[4]/Omega[2])

...

Eq(u(z, mu[2]), Ac(z/Omega[2], mu[2])*exp(-I*pi/4))

Eq(Derivative(A(x, mu[1]), x), I*A(x, mu[1])**2*Ac(x, mu[1])*Omega[3] + 2*I*A(x, mu[1])*A(x, mu[2])*Ac(x, mu[2])*Omega[4] + I*A(x, mu[1])*Omega[1] + I*A(x, mu[2])*Omega[2])

Eq(Derivative(A(x, mu[2]), x), 2*I*A(x, mu[1])*A(x, mu[2])*Ac(x, mu[1])*Omega[4] + I*A(x, mu[1])*Omega[2] + I*A(x, mu[2])**2*Ac(x, mu[2])*Omega[3] + I*A(x, mu[2])*Omega[1])

Eq(Derivative(Ac(x, mu[1]), x), -I*A(x, mu[1])*Ac(x, mu[1])**2*Omega[3] - 2*I*A(x, mu[2])*Ac(x, mu[1])*Ac(x, mu[2])*Omega[4] - I*Ac(x, mu[1])*Omega[1] - I*Ac(x, mu[2])*Omega[2])

Eq(Derivative(Ac(x, mu[2]), x), -2*I*A(x, mu[1])*Ac(x, mu[1])*Ac(x, mu[2])*Omega[4] - I*A(x, mu[2])*Ac(x, mu[2])**2*Omega[3] - I*Ac(x, mu[1])*Omega[2] - I*Ac(x, mu[2])*Omega[1])

## Linear diagonalisation of wave mixing in the coherent coupler

In this section we show that the Jensen coherent coupler can be transformed into a degeneration of a FWM system, namely that of nonlinear polarisation in which the four modes form 2 pairs.

In [77]:
MM = Matrix([[Omega[1], Omega[2]], [Omega[2], Omega[1]]])
PP, DD = MM.diagonalize()
PPi = PP**(-1)
PP = 1/sqrt(2)*PP
PPi = sqrt(2)*PPi

AA = Matrix([A(z, mu[1]), A(z, mu[2])])
AAc = Matrix([Ac(z, mu[1]), Ac(z, mu[2])])
BB = Matrix([B(z, mu[1]), B(z, mu[2])])
BBc = Matrix([Bc(z, mu[1]), Bc(z, mu[2])])
# B_as_A = Eq(BB, PP*DD*PPi*AA)
# Bc_as_Ac = Eq(BBc, PP*DD*PPi*AAc)
# A_as_B = Eq(AA, PP*DD**(-1)*PPi*BB)
# Ac_as_Bc = Eq(AAc, PP*DD**(-1)*PPi*BBc)
B_as_A = Eq(BB, PPi*AA)
Bc_as_Ac = Eq(BBc, PPi*AAc)
A_as_B = Eq(AA, PP*BB)
Ac_as_Bc = Eq(AAc, PP*BBc)

A_as_B_subs = [
    (A_as_B.lhs[0], A_as_B.rhs[0]), 
    (A_as_B.lhs[1], A_as_B.rhs[1])
]
Ac_as_Bc_subs = [
    (Ac_as_Bc.lhs[0], Ac_as_Bc.rhs[0]), 
    (Ac_as_Bc.lhs[1], Ac_as_Bc.rhs[1])
]
all_A_as_B_subs = A_as_B_subs + Ac_as_Bc_subs
B_as_A_subs = [
    (B_as_A.lhs[0], B_as_A.rhs[0]), 
    (B_as_A.lhs[1], B_as_A.rhs[1])
]
Bc_as_Ac_subs = [
    (Bc_as_Ac.lhs[0], Bc_as_Ac.rhs[0]), 
    (Bc_as_Ac.lhs[1], Bc_as_Ac.rhs[1])
]

assert B_as_A.subs(A_as_B_subs).simplify()
assert Bc_as_Ac.subs(Ac_as_Bc_subs).simplify()
assert A_as_B.subs(B_as_A_subs).simplify()
assert Ac_as_Bc.subs(Bc_as_Ac_subs).simplify()

MM
PP
PPi
DD
B_as_A
Bc_as_Ac

Matrix([
[Omega[1], Omega[2]],
[Omega[2], Omega[1]]])

Matrix([
[-sqrt(2)/2, sqrt(2)/2],
[ sqrt(2)/2, sqrt(2)/2]])

Matrix([
[-sqrt(2)/2, sqrt(2)/2],
[ sqrt(2)/2, sqrt(2)/2]])

Matrix([
[Omega[1] - Omega[2],                   0],
[                  0, Omega[1] + Omega[2]]])

Eq(Matrix([
[B(z, mu[1])],
[B(z, mu[2])]]), Matrix([
[-sqrt(2)*A(z, mu[1])/2 + sqrt(2)*A(z, mu[2])/2],
[ sqrt(2)*A(z, mu[1])/2 + sqrt(2)*A(z, mu[2])/2]]))

Eq(Matrix([
[Bc(z, mu[1])],
[Bc(z, mu[2])]]), Matrix([
[-sqrt(2)*Ac(z, mu[1])/2 + sqrt(2)*Ac(z, mu[2])/2],
[ sqrt(2)*Ac(z, mu[1])/2 + sqrt(2)*Ac(z, mu[2])/2]]))

In [78]:
B_jensen = [_.subs(x,z).subs(all_A_as_B_subs).doit() for _ in uv_to_jensen_A]
B_diffs = [diff(B(z, mu[1]),z), diff(B(z, mu[2]),z), diff(Bc(z, mu[1]),z), diff(Bc(z, mu[2]),z)]
B_jensen_sols = solve(B_jensen, B_diffs)
dB_jensen = [Eq(_, B_jensen_sols[_]) for _ in B_diffs]

_collects =[
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), B(z, mu[1])],
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), B(z, mu[2])],
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), Bc(z, mu[1])],
    [B(z, mu[1])*Bc(z, mu[1]), B(z, mu[2])*Bc(z, mu[2]), Bc(z, mu[2])]
]

Omega_beta_subs = [
    (Omega[1] - Omega[2], psi[0]),
    (Omega[1] + Omega[2], psi[1]),
    (Omega[3] - 2*Omega[4], 2*psi[2]),
    (Omega[3] + 2*Omega[4], 2*psi[3]),
    (Omega[3], psi[4])
]

dB_jensen = [
    Eq(
        _eq.lhs, 
        _eq.rhs
        .expand().collect(_c, factor)
        .subs(Omega_beta_subs)
        .expand().collect([psi[0], psi[1], psi[2]], factor)
    ) 
    for _c, _eq in zip(_collects, dB_jensen)
]

for _ in Omega_beta_subs:
    Eq(*_)
print()
for _ in dB_jensen:
    _

Eq(Omega[1] - Omega[2], psi[0])

Eq(Omega[1] + Omega[2], psi[1])

Eq(Omega[3] - 2*Omega[4], 2*psi[2])

Eq(Omega[3] + 2*Omega[4], 2*psi[3])

Eq(Omega[3], psi[4])

Eq(Derivative(B(z, mu[1]), z), I*(B(z, mu[1])*Bc(z, mu[1])*psi[3] + B(z, mu[2])*Bc(z, mu[2])*psi[4])*B(z, mu[1]) + I*B(z, mu[1])*psi[0] + I*B(z, mu[2])**2*Bc(z, mu[1])*psi[2])

Eq(Derivative(B(z, mu[2]), z), I*(B(z, mu[1])*Bc(z, mu[1])*psi[4] + B(z, mu[2])*Bc(z, mu[2])*psi[3])*B(z, mu[2]) + I*B(z, mu[1])**2*Bc(z, mu[2])*psi[2] + I*B(z, mu[2])*psi[1])

Eq(Derivative(Bc(z, mu[1]), z), -I*(B(z, mu[1])*Bc(z, mu[1])*psi[3] + B(z, mu[2])*Bc(z, mu[2])*psi[4])*Bc(z, mu[1]) - I*B(z, mu[1])*Bc(z, mu[2])**2*psi[2] - I*Bc(z, mu[1])*psi[0])

Eq(Derivative(Bc(z, mu[2]), z), -I*(B(z, mu[1])*Bc(z, mu[1])*psi[4] + B(z, mu[2])*Bc(z, mu[2])*psi[3])*Bc(z, mu[2]) - I*B(z, mu[2])*Bc(z, mu[1])**2*psi[2] - I*Bc(z, mu[2])*psi[1])

In [79]:
jens_p0_B = jens_p0.subs(all_A_as_B_subs).simplify()
jens_p1_B = Eq(
    jens_p1.lhs, 
    jens_p1.rhs.subs(all_A_as_B_subs).simplify().subs(Omega_beta_subs).expand().collect(psi[2], simplify)
)

jens_p0_B
jens_p1_B

Eq(p[0], B(z, mu[1])*Bc(z, mu[1]) + B(z, mu[2])*Bc(z, mu[2]))

Eq(p[1], (-B(z, mu[1])**2*Bc(z, mu[1])**2 + B(z, mu[1])**2*Bc(z, mu[2])**2 + B(z, mu[2])**2*Bc(z, mu[1])**2 - B(z, mu[2])**2*Bc(z, mu[2])**2)*psi[2]/Omega[2] - 2*B(z, mu[1])*Bc(z, mu[1]) + 2*B(z, mu[2])*Bc(z, mu[2]))